
# Crypto Trading Strategy – The Overfitters

## Syfte

Syftet med denna notebook är att undersöka
om historiska marknadsmönster kan användas
för att identifiera potentiella köp- och
säljtillfällen för Ethereum.

Vi analyserar:

1. Långsiktiga pristrender med SMA 200.
2. Momentum genom RSI 14.
3. Bull Market Support Band (BMSB).
4. Historiska köp- och säljsignaler.
5. Kapitalutveckling genom backtesting.
6. Jämförelse mot Buy & Hold.
7. Möjligheten att kombinera teknisk analys
   med maskininlärning.

## Forskningsfråga

Kan en kombination av tekniska indikatorer
identifiera historiska köp- och säljtillfällen
som ger bättre riskjusterad avkastning än
en passiv Buy & Hold-strategi?

## Metod

Vi börjar med en enkel regelbaserad modell.

Vi utvärderar strategin på historiska data
med hänsyn till transaktionskostnader
och tidsordning.

Därefter undersöker vi om maskininlärning
kan bidra med ytterligare prognosförmåga.

Vi skiljer tydligt mellan historiska resultat
och prognoser som ännu inte har validerats.



## Steg 1 – Datainläsning och datakvalitet

### Syfte

Vi börjar med att läsa in historiska prisdata
för Ethereum (ETH) och Bitcoin (BTC).

Syftet är att säkerställa att datamaterialet
är komplett och användbart för analys av
historiska köp- och säljsignaler.

### Metod

Vi använder historiska dagspriser från 2015
till senaste tillgängliga observationsdatum.

Ethereum och Bitcoin behandlas som två
separata tidsserier.

Vi undersöker:

1. Antal observationer.
2. Första och sista datum.
3. Saknade och ogiltiga priser.
4. Dubbletter.
5. Saknade kalenderdagar.
6. Historisk prisutveckling.

### Nästa steg

Efter datakontrollen beräknar vi tekniska
indikatorer för respektive kryptovaluta.


In [1]:

%pip install -U nbformat plotly ipykernel


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:

# STEG 1 – Datainläsning och kontroll av ETH/BTC

from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display

# 1. Hitta data-mappen
DATA_DIR = Path("data")

if not DATA_DIR.exists():
    DATA_DIR = Path("Ethereum_Bitcoin_ML/data")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        "Hittar inte data-mappen. Kontrollera arbetskatalogen."
    )

# 2. Läs in historiska priser
df_prices = pd.read_csv(
    DATA_DIR / "crypto_historical_prices.csv"
)

df_prices["time"] = pd.to_datetime(
    df_prices["time"],
    utc=True,
    errors="coerce"
)

# 3. Kontrollera originaldata
print("ORIGINALDATA")
print("Antal rader:", len(df_prices))
print("Kolumner:", df_prices.columns.tolist())

print("\nDubbletter i datum:")
print(df_prices["time"].duplicated().sum())

print("\nSaknade priser:")
display(
    df_prices[["ETH_Price", "BTC_Price"]]
    .isna()
    .sum()
)

# 4. Separera ETH och BTC
def prepare_asset(data, asset):

    price_col = f"{asset}_Price"

    df = data[
        ["time", price_col]
    ].copy()

    df[price_col] = pd.to_numeric(
        df[price_col],
        errors="coerce"
    )

    df = df.dropna(subset=["time", price_col])

    df = df[df[price_col] > 0]

    df = df.sort_values("time")

    df = df.drop_duplicates(
        subset=["time"],
        keep="last"
    )

    return df.reset_index(drop=True)

df_eth = prepare_asset(df_prices, "ETH")
df_btc = prepare_asset(df_prices, "BTC")

# 5. Kontrollera saknade kalenderdagar
def missing_days(df):

    expected = pd.date_range(
        start=df["time"].min(),
        end=df["time"].max(),
        freq="D",
        tz="UTC"
    )

    return len(
        expected.difference(
            pd.DatetimeIndex(df["time"])
        )
    )

# 6. Skapa resultattabell
overview = pd.DataFrame([
    {
        "Valuta": "Ethereum",
        "Observationer": len(df_eth),
        "Första datum": df_eth["time"].min().date(),
        "Sista datum": df_eth["time"].max().date(),
        "Senaste pris USD": df_eth["ETH_Price"].iloc[-1],
        "Saknade dagar": missing_days(df_eth)
    },
    {
        "Valuta": "Bitcoin",
        "Observationer": len(df_btc),
        "Första datum": df_btc["time"].min().date(),
        "Sista datum": df_btc["time"].max().date(),
        "Senaste pris USD": df_btc["BTC_Price"].iloc[-1],
        "Saknade dagar": missing_days(df_btc)
    }
])

print("\nDATAÖVERSIKT – ETH OCH BTC")
display(overview.round(2))

print("\nSENASTE ETH-PRISER")
display(df_eth.tail(5).round(2))

print("\nSENASTE BTC-PRISER")
display(df_btc.tail(5).round(2))



ORIGINALDATA
Antal rader: 4323
Kolumner: ['time', 'BTC_Price', 'ETH_Price']

Dubbletter i datum:
0

Saknade priser:


ETH_Price    219
BTC_Price      0
dtype: int64


DATAÖVERSIKT – ETH OCH BTC


,Valuta,Observationer,Första datum,Sista datum,Senaste pris USD,Saknade dagar
0,Ethereum,4104,2015-08-08,2026-11-01,2269.99,0
1,Bitcoin,4323,2015-01-01,2026-11-01,78135.08,0



SENASTE ETH-PRISER


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\2776504770.py:116: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(df_eth.tail(5).round(2))


,time,ETH_Price
4099,2026-10-28 00:00:00+00:00,2356.99
4100,2026-10-29 00:00:00+00:00,2256.03
4101,2026-10-30 00:00:00+00:00,2240.23
4102,2026-10-31 00:00:00+00:00,2252.07
4103,2026-11-01 00:00:00+00:00,2269.99



SENASTE BTC-PRISER


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\2776504770.py:119: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(df_btc.tail(5).round(2))


,time,BTC_Price
4318,2026-10-28 00:00:00+00:00,81237.44
4319,2026-10-29 00:00:00+00:00,78193.43
4320,2026-10-30 00:00:00+00:00,77109.59
4321,2026-10-31 00:00:00+00:00,77401.02
4322,2026-11-01 00:00:00+00:00,78135.08


In [3]:

# STEG 1.1 – Historisk prisutveckling ETH och BTC

import plotly.graph_objects as go

def plot_historical_price(df, asset):

    price_col = f"{asset}_Price"

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df[price_col],
        mode="lines",
        name=f"{asset}-pris",
        line=dict(
            color="#5DADE2" if asset == "ETH" else "#F39C12",
            width=2
        )
    ))

    fig.update_layout(
        title=f"{asset} – Historisk prisutveckling",
        xaxis_title="Datum",
        yaxis_title="Pris (USD)",
        template="plotly_dark",
        hovermode="x unified",
        height=550
    )

    fig.update_xaxes(
        rangeslider_visible=True
    )

    fig.show()


# DIAGRAM 1 – Ethereum
plot_historical_price(df_eth, "ETH")

# DIAGRAM 2 – Bitcoin
plot_historical_price(df_btc, "BTC")



## Steg 2 – Tekniska indikatorer för ETH och BTC

### Syfte

Vi undersöker om tekniska indikatorer kan
användas för att identifiera historiska
marknadstrender och möjliga köp- och säljtillfällen.

Ethereum och Bitcoin analyseras separat.

### Indikatorer

**SMA 200**

Ett glidande medelvärde av de senaste
200 dagarnas priser.

Används för att identifiera långsiktig trend.

**RSI 14**

Relative Strength Index mäter momentum
baserat på historiska prisförändringar.

Traditionella tolkningsnivåer:
- RSI under 30: potentiellt översåld marknad.
- RSI över 70: potentiellt överköpt marknad.

**Bull Market Support Band (BMSB)**

Består av:
- 20 veckors Simple Moving Average (SMA).
- 21 veckors Exponential Moving Average (EMA).

Bandet används för att undersöka
långsiktiga marknadstrender.

### Metod

Alla indikatorer beräknas med historiska
priser fram till respektive observationsdatum.

Inga framtida priser används vid beräkningen.

Veckoindikatorer baseras endast på
avslutade veckor när de kopplas till
dagliga observationer.


In [4]:

# STEG 2.1 – Beräkna tekniska indikatorer

import pandas as pd
import numpy as np
from IPython.display import display

def calculate_indicators(df, asset):

    data = df.copy()

    price_col = f"{asset}_Price"
    prices = data[price_col]

    data = data.sort_values("time").reset_index(drop=True)

    # 1. SMA 200 – 200 dagars glidande medelvärde
    data["SMA_200"] = (
        data[price_col]
        .rolling(window=200, min_periods=200)
        .mean()
    )

    # 2. RSI 14 – Relative Strength Index
    delta = data[price_col].diff()

    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)

    avg_gain = gain.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()

    avg_loss = loss.ewm(
        alpha=1/14,
        adjust=False,
        min_periods=14
    ).mean()

    rs = avg_gain / avg_loss.replace(0, np.nan)

    data["RSI_14"] = 100 - (100 / (1 + rs))

    # 3. Bull Market Support Band
    # Veckor avslutas på söndagar (UTC).
    weekly = (
        data.set_index("time")[[price_col]]
        .resample("W-SUN")
        .last()
    )

    # Endast avslutade kalenderveckor får användas.
    # Sista veckans värde behandlas inte som färdigt
    # om söndagens observation saknas.
    last_observation = data["time"].max()

    weekly = weekly[
        weekly.index <= last_observation
    ].copy()

    weekly["SMA_20W"] = (
        weekly[price_col]
        .rolling(20, min_periods=20)
        .mean()
    )

    weekly["EMA_21W"] = (
        weekly[price_col]
        .ewm(
            span=21,
            adjust=False,
            min_periods=21
        )
        .mean()
    )

    # För dagliga signaler får samma veckas
    # söndagsstängning inte användas i förtid.
    # Föregående avslutade vecka blir tillgänglig
    # från följande måndag.
    weekly_available = weekly[
        ["SMA_20W", "EMA_21W"]
    ].copy()

    weekly_available.index = (
        weekly_available.index
        + pd.Timedelta(days=1)
    )

    data = pd.merge_asof(
        data.sort_values("time"),
        weekly_available.sort_index(),
        left_on="time",
        right_index=True,
        direction="backward"
    )

    data["BMSB_Upper"] = data[
        ["SMA_20W", "EMA_21W"]
    ].max(axis=1)

    data["BMSB_Lower"] = data[
        ["SMA_20W", "EMA_21W"]
    ].min(axis=1)

    # Saknas någon av indikatorerna
    # saknas också ett fullständigt BMSB.
    invalid_band = data[
        ["SMA_20W", "EMA_21W"]
    ].isna().any(axis=1)

    data.loc[
        invalid_band,
        ["BMSB_Upper", "BMSB_Lower"]
    ] = np.nan

    return data


# Beräkna separat
df_eth_ind = calculate_indicators(df_eth, "ETH")
df_btc_ind = calculate_indicators(df_btc, "BTC")

# Kontrollera resultaten
for asset, data in [
    ("ETH", df_eth_ind),
    ("BTC", df_btc_ind)
]:

    print(f"\n===== {asset} =====")

    cols = [
        "SMA_200",
        "RSI_14",
        "SMA_20W",
        "EMA_21W",
        "BMSB_Upper",
        "BMSB_Lower"
    ]

    print("Observationer:", len(data))
    print("\nSaknade indikatorvärden:")
    display(data[cols].isna().sum())

    print("\nSenaste observationerna:")
    display(
        data[
            ["time", f"{asset}_Price"] + cols
        ].tail(10).round(2)
    )



===== ETH =====
Observationer: 4104

Saknade indikatorvärden:


SMA_200       199
RSI_14         14
SMA_20W       135
EMA_21W       142
BMSB_Upper    142
BMSB_Lower    142
dtype: int64


Senaste observationerna:


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\3908139636.py:150: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,ETH_Price,SMA_200,RSI_14,SMA_20W,EMA_21W,BMSB_Upper,BMSB_Lower
4094,2026-10-23 00:00:00+00:00,2378.26,2155.88,38.14,2131.19,2318.52,2318.52,2131.19
4095,2026-10-24 00:00:00+00:00,2351.99,2156.40,36.13,2131.19,2318.52,2318.52,2131.19
4096,2026-10-25 00:00:00+00:00,2325.80,2157.09,34.19,2131.19,2318.52,2318.52,2131.19
4097,2026-10-26 00:00:00+00:00,2316.60,2157.72,33.51,2163.38,2319.18,2319.18,2163.38
4098,2026-10-27 00:00:00+00:00,2347.56,2158.24,37.98,2163.38,2319.18,2319.18,2163.38
4099,2026-10-28 00:00:00+00:00,2356.99,2158.59,39.32,2163.38,2319.18,2319.18,2163.38
4100,2026-10-29 00:00:00+00:00,2256.03,2158.92,31.48,2163.38,2319.18,2319.18,2163.38
4101,2026-10-30 00:00:00+00:00,2240.23,2158.25,30.46,2163.38,2319.18,2319.18,2163.38
4102,2026-10-31 00:00:00+00:00,2252.07,2157.90,32.24,2163.38,2319.18,2319.18,2163.38
4103,2026-11-01 00:00:00+00:00,2269.99,2157.46,34.95,2163.38,2319.18,2319.18,2163.38



===== BTC =====
Observationer: 4323

Saknade indikatorvärden:


SMA_200       199
RSI_14         14
SMA_20W       137
EMA_21W       144
BMSB_Upper    144
BMSB_Lower    144
dtype: int64


Senaste observationerna:


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\3908139636.py:150: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,BTC_Price,SMA_200,RSI_14,SMA_20W,EMA_21W,BMSB_Upper,BMSB_Lower
4313,2026-10-23 00:00:00+00:00,80924.18,72849.49,44.76,71509.00,76192.48,76192.48,71509.00
4314,2026-10-24 00:00:00+00:00,80699.31,72892.70,43.99,71509.00,76192.48,76192.48,71509.00
4315,2026-10-25 00:00:00+00:00,80038.91,72937.47,41.73,71509.00,76192.48,76192.48,71509.00
4316,2026-10-26 00:00:00+00:00,79949.93,72978.27,41.42,72353.03,76542.16,76542.16,72353.03
4317,2026-10-27 00:00:00+00:00,80800.54,73017.55,45.57,72353.03,76542.16,76542.16,72353.03
4318,2026-10-28 00:00:00+00:00,81237.44,73058.14,47.63,72353.03,76542.16,76542.16,72353.03
4319,2026-10-29 00:00:00+00:00,78193.43,73095.68,37.11,72353.03,76542.16,76542.16,72353.03
4320,2026-10-30 00:00:00+00:00,77109.59,73108.06,34.22,72353.03,76542.16,76542.16,72353.03
4321,2026-10-31 00:00:00+00:00,77401.02,73124.20,35.67,72353.03,76542.16,76542.16,72353.03
4322,2026-11-01 00:00:00+00:00,78135.08,73141.07,39.31,72353.03,76542.16,76542.16,72353.03


In [5]:

# STEG 2.2 – Pris och SMA 200

import plotly.graph_objects as go

def plot_sma200(data, asset):

    price_col = f"{asset}_Price"

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=data["time"],
        y=data[price_col],
        name=f"{asset}-pris",
        mode="lines",
        line=dict(color="#5DADE2", width=1.5)
    ))

    fig.add_trace(go.Scatter(
        x=data["time"],
        y=data["SMA_200"],
        name="SMA 200",
        mode="lines",
        line=dict(color="#F39C12", width=2.5)
    ))

    fig.update_layout(
        title=f"{asset} – Pris och SMA 200",
        xaxis_title="Datum",
        yaxis_title="Pris (USD)",
        template="plotly_dark",
        height=600,
        hovermode="x unified"
    )

    fig.update_xaxes(rangeslider_visible=True)

    fig.show()


plot_sma200(df_eth_ind, "ETH")
plot_sma200(df_btc_ind, "BTC")


In [6]:

# STEG 2.3 – RSI 14 för ETH och BTC

import plotly.graph_objects as go

def plot_rsi(data, asset):

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=data["time"],
        y=data["RSI_14"],
        name="RSI 14",
        mode="lines",
        line=dict(color="#9B59B6", width=1.5)
    ))

    fig.add_hline(
        y=70,
        line_dash="dash",
        line_color="#FF4B4B",
        annotation_text="Överköpt – 70"
    )

    fig.add_hline(
        y=30,
        line_dash="dash",
        line_color="#2ECC71",
        annotation_text="Översålt – 30"
    )

    fig.update_layout(
        title=f"{asset} – RSI 14",
        xaxis_title="Datum",
        yaxis_title="RSI",
        yaxis_range=[0, 100],
        template="plotly_dark",
        height=500,
        hovermode="x unified"
    )

    fig.update_xaxes(rangeslider_visible=True)
    fig.show()


plot_rsi(df_eth_ind, "ETH")
plot_rsi(df_btc_ind, "BTC")


In [7]:

# STEG 2.4 – Bull Market Support Band

import plotly.graph_objects as go

def plot_bmsb(data, asset):

    price_col = f"{asset}_Price"

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=data["time"],
        y=data["BMSB_Upper"],
        name="Övre BMSB",
        mode="lines",
        line=dict(color="#2ECC71", width=1)
    ))

    fig.add_trace(go.Scatter(
        x=data["time"],
        y=data["BMSB_Lower"],
        name="Nedre BMSB",
        mode="lines",
        fill="tonexty",
        fillcolor="rgba(46,204,113,0.18)",
        line=dict(color="#2ECC71", width=1)
    ))

    fig.add_trace(go.Scatter(
        x=data["time"],
        y=data["SMA_20W"],
        name="20W SMA",
        line=dict(color="#F39C12", width=2)
    ))

    fig.add_trace(go.Scatter(
        x=data["time"],
        y=data["EMA_21W"],
        name="21W EMA",
        line=dict(color="#9B59B6", width=2)
    ))

    fig.add_trace(go.Scatter(
        x=data["time"],
        y=data[price_col],
        name=f"{asset}-pris",
        line=dict(color="#5DADE2", width=1.5)
    ))

    fig.update_layout(
        title=f"{asset} – Bull Market Support Band",
        xaxis_title="Datum",
        yaxis_title="Pris (USD, logaritmisk skala)",
        yaxis_type="log",
        template="plotly_dark",
        height=650,
        hovermode="x unified"
    )

    fig.update_xaxes(rangeslider_visible=True)

    fig.show()


plot_bmsb(df_eth_ind, "ETH")
plot_bmsb(df_btc_ind, "BTC")



## Steg 3 – Kombinerade köp- och säljsignaler

### Syfte

Vi undersöker om tre tekniska indikatorer
tillsammans kan identifiera möjliga köp-
och säljtillfällen för Ethereum och Bitcoin.

### Metod

Vi kombinerar:

1. SMA 200 – långsiktig pristrend.
2. RSI 14 – marknadens momentum.
3. BMSB – långsiktigt trendband.

### Signalregler

**KÖP**
- Priset ligger över SMA 200.
- Priset ligger över övre BMSB.
- RSI ligger mellan 50 och 70.

**SÄLJ**
- Priset ligger under SMA 200.
- Priset ligger under nedre BMSB.
- RSI är under 45.

**AVVAKTA**
- Varken köp- eller säljvillkoren uppfylls.

**INGEN DATA**
- Någon indikator saknar ett giltigt värde.

Reglerna är förutbestämda och explorativa.
Vi optimerar dem inte mot historisk avkastning
i detta steg.

En signal är inte samma sak som en genomförd
transaktion. Vid framtida backtesting måste
vi även ta hänsyn till positioner, kostnader
och att handeln sker efter signalen.


In [8]:

# STEG 3.1 – Köp- och säljsignaler för ETH och BTC

import pandas as pd
import numpy as np
from IPython.display import display

def create_signals(data, asset):

    df = data.copy().sort_values(
        "time"
    ).reset_index(drop=True)

    price_col = f"{asset}_Price"

    required = [
        price_col,
        "SMA_200",
        "RSI_14",
        "BMSB_Upper",
        "BMSB_Lower"
    ]

    # Kontrollera att indikatorerna finns
    missing_cols = [
        c for c in required if c not in df.columns
    ]

    if missing_cols:
        raise ValueError(
            f"Kolumner saknas för {asset}: {missing_cols}"
        )

    valid = df[required].notna().all(axis=1)

    # Köpregler
    buy = (
        valid
        & (df[price_col] > df["SMA_200"])
        & (df[price_col] > df["BMSB_Upper"])
        & df["RSI_14"].between(50, 70)
    )

    # Säljregler
    sell = (
        valid
        & (df[price_col] < df["SMA_200"])
        & (df[price_col] < df["BMSB_Lower"])
        & (df["RSI_14"] < 45)
    )

    df["Signal"] = np.select(
        [buy, sell, valid],
        ["KÖP", "SÄLJ", "AVVAKTA"],
        default="INGEN DATA"
    )

    # Markera enbart när signalen ändras
    previous_signal = df["Signal"].shift(1)

    df["Ny_Signal"] = np.where(
        df["Signal"] != previous_signal,
        df["Signal"],
        ""
    )

    df.loc[
        ~df["Ny_Signal"].isin(["KÖP", "SÄLJ"]),
        "Ny_Signal"
    ] = ""

    return df


# Separata DataFrames
df_eth_signals = create_signals(df_eth_ind, "ETH")
df_btc_signals = create_signals(df_btc_ind, "BTC")

# Resultattabeller
for asset, df in [
    ("ETH", df_eth_signals),
    ("BTC", df_btc_signals)
]:

    print(f"\n===== {asset} =====")

    summary = (
        df["Signal"]
        .value_counts()
        .rename_axis("Signal")
        .reset_index(name="Antal dagar")
    )

    summary["Andel (%)"] = (
        summary["Antal dagar"] / len(df) * 100
    ).round(2)

    display(summary)

    print("\nSenaste 10 observationerna:")

    display(
        df[
            [
                "time",
                f"{asset}_Price",
                "SMA_200",
                "RSI_14",
                "BMSB_Upper",
                "BMSB_Lower",
                "Signal"
            ]
        ].tail(10).round(2)
    )




===== ETH =====


,Signal,Antal dagar,Andel (%)
0,AVVAKTA,1911,46.56
1,KÖP,1094,26.66
2,SÄLJ,900,21.93
3,INGEN DATA,199,4.85



Senaste 10 observationerna:


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\1667644194.py:112: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,ETH_Price,SMA_200,RSI_14,BMSB_Upper,BMSB_Lower,Signal
4094,2026-10-23 00:00:00+00:00,2378.26,2155.88,38.14,2318.52,2131.19,AVVAKTA
4095,2026-10-24 00:00:00+00:00,2351.99,2156.40,36.13,2318.52,2131.19,AVVAKTA
4096,2026-10-25 00:00:00+00:00,2325.80,2157.09,34.19,2318.52,2131.19,AVVAKTA
4097,2026-10-26 00:00:00+00:00,2316.60,2157.72,33.51,2319.18,2163.38,AVVAKTA
4098,2026-10-27 00:00:00+00:00,2347.56,2158.24,37.98,2319.18,2163.38,AVVAKTA
4099,2026-10-28 00:00:00+00:00,2356.99,2158.59,39.32,2319.18,2163.38,AVVAKTA
4100,2026-10-29 00:00:00+00:00,2256.03,2158.92,31.48,2319.18,2163.38,AVVAKTA
4101,2026-10-30 00:00:00+00:00,2240.23,2158.25,30.46,2319.18,2163.38,AVVAKTA
4102,2026-10-31 00:00:00+00:00,2252.07,2157.90,32.24,2319.18,2163.38,AVVAKTA
4103,2026-11-01 00:00:00+00:00,2269.99,2157.46,34.95,2319.18,2163.38,AVVAKTA



===== BTC =====


,Signal,Antal dagar,Andel (%)
0,AVVAKTA,2067,47.81
1,KÖP,1287,29.77
2,SÄLJ,770,17.81
3,INGEN DATA,199,4.60



Senaste 10 observationerna:


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\1667644194.py:112: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,BTC_Price,SMA_200,RSI_14,BMSB_Upper,BMSB_Lower,Signal
4313,2026-10-23 00:00:00+00:00,80924.18,72849.49,44.76,76192.48,71509.00,AVVAKTA
4314,2026-10-24 00:00:00+00:00,80699.31,72892.70,43.99,76192.48,71509.00,AVVAKTA
4315,2026-10-25 00:00:00+00:00,80038.91,72937.47,41.73,76192.48,71509.00,AVVAKTA
4316,2026-10-26 00:00:00+00:00,79949.93,72978.27,41.42,76542.16,72353.03,AVVAKTA
4317,2026-10-27 00:00:00+00:00,80800.54,73017.55,45.57,76542.16,72353.03,AVVAKTA
4318,2026-10-28 00:00:00+00:00,81237.44,73058.14,47.63,76542.16,72353.03,AVVAKTA
4319,2026-10-29 00:00:00+00:00,78193.43,73095.68,37.11,76542.16,72353.03,AVVAKTA
4320,2026-10-30 00:00:00+00:00,77109.59,73108.06,34.22,76542.16,72353.03,AVVAKTA
4321,2026-10-31 00:00:00+00:00,77401.02,73124.20,35.67,76542.16,72353.03,AVVAKTA
4322,2026-11-01 00:00:00+00:00,78135.08,73141.07,39.31,76542.16,72353.03,AVVAKTA


In [9]:

# STEG 3.2 – Visualisera köp- och säljsignaler

import plotly.graph_objects as go

def plot_trading_signals(df, asset):

    price_col = f"{asset}_Price"

    fig = go.Figure()

    # Historiskt pris
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df[price_col],
        mode="lines",
        name=f"{asset}-pris",
        line=dict(color="#5DADE2", width=1.5)
    ))

    # SMA 200
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["SMA_200"],
        name="SMA 200",
        line=dict(color="#F39C12", width=2)
    ))

    # BMSB
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["BMSB_Upper"],
        name="Övre BMSB",
        line=dict(color="#2ECC71", width=1)
    ))

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["BMSB_Lower"],
        name="Nedre BMSB",
        fill="tonexty",
        fillcolor="rgba(46,204,113,0.12)",
        line=dict(color="#2ECC71", width=1)
    ))

    buys = df[df["Ny_Signal"] == "KÖP"]
    sells = df[df["Ny_Signal"] == "SÄLJ"]

    fig.add_trace(go.Scatter(
        x=buys["time"],
        y=buys[price_col],
        mode="markers",
        name="Köpsignal",
        marker=dict(
            symbol="triangle-up",
            color="#00E676",
            size=11
        )
    ))

    fig.add_trace(go.Scatter(
        x=sells["time"],
        y=sells[price_col],
        mode="markers",
        name="Säljsignal",
        marker=dict(
            symbol="triangle-down",
            color="#FF5252",
            size=11
        )
    ))

    fig.update_layout(
        title=f"{asset} – Historiska köp- och säljsignaler",
        xaxis_title="Datum",
        yaxis_title="Pris (USD, logaritmisk skala)",
        yaxis_type="log",
        template="plotly_dark",
        height=650,
        hovermode="x unified"
    )

    fig.update_xaxes(rangeslider_visible=True)

    fig.show()


plot_trading_signals(df_eth_signals, "ETH")
plot_trading_signals(df_btc_signals, "BTC")


In [10]:

# STEG 3.3 – Signalfördelning

import plotly.express as px

signal_rows = []

for asset, df in [
    ("ETH", df_eth_signals),
    ("BTC", df_btc_signals)
]:
    for signal, count in df["Signal"].value_counts().items():
        signal_rows.append({
            "Kryptovaluta": asset,
            "Signal": signal,
            "Antal dagar": count
        })

df_signal_summary = pd.DataFrame(signal_rows)

fig = px.bar(
    df_signal_summary,
    x="Signal",
    y="Antal dagar",
    color="Kryptovaluta",
    barmode="group",
    text_auto=True,
    title="ETH och BTC – Fördelning av handelssignaler",
    template="plotly_dark",
    category_orders={
        "Signal": [
            "KÖP",
            "SÄLJ",
            "AVVAKTA",
            "INGEN DATA"
        ]
    }
)

fig.update_layout(height=550)
fig.show()



## Steg 4 – Backtesting av handelsstrategier

### Syfte

Vi undersöker om våra tekniska handelssignaler
historiskt hade kunnat ge bättre riskjusterad
avkastning än Buy & Hold.

### Metod

Vi simulerar en handelsstrategi med:

- Startkapital: 10 000 USD.
- Köp vid signalen KÖP.
- Försäljning vid signalen SÄLJ.
- AVVAKTA innebär att befintlig position behålls.
- Handel vid nästa dags stängningspris.
- Transaktionskostnad: 0,1 % per affär.
- Ingen hävstång eller blankning.

Vi jämför med Buy & Hold under samma period.

### Utvärdering

Vi beräknar:

1. Slutkapital.
2. Total avkastning.
3. Maximal drawdown.
4. Sharpekvot.
5. Antal köp och försäljningar.
6. Andel dagar investerad.

Ethereum och Bitcoin utvärderas separat.

Resultaten är historiska simuleringar
och innehåller inte slippage eller skatter.


In [11]:

# STEG 4.1 – Backtesting ETH och BTC

import numpy as np
import pandas as pd
from IPython.display import display

INITIAL_CAPITAL = 10000
FEE = 0.001  # 0,1 % per transaktion

def backtest_strategy(data, asset):

    df = data.copy().sort_values(
        "time"
    ).reset_index(drop=True)

    price_col = f"{asset}_Price"

    # Samma startperiod för strategi och Buy & Hold
    df = df[
        df["Signal"] != "INGEN DATA"
    ].copy().reset_index(drop=True)

    # Önskad position efter signalen
    # 1 = äger kryptovalutan, 0 = kontanter
    desired = []
    position = 0

    for signal in df["Signal"]:

        if signal == "KÖP":
            position = 1

        elif signal == "SÄLJ":
            position = 0

        # AVVAKTA: behåll tidigare position
        desired.append(position)

    df["Desired_Position"] = desired

    # Signalen idag verkställs vid nästa stängning
    df["Position"] = (
        df["Desired_Position"]
        .shift(1)
        .fillna(0)
        .astype(int)
    )

    # Dagens stängning-till-stängning-avkastning
    df["Market_Return"] = (
        df[price_col].pct_change().fillna(0)
    )

    # Position som faktiskt bars under dagen
    exposure = df["Position"].shift(1).fillna(0)

    # Genomförda positionstransaktioner
    df["Trades"] = (
        df["Position"].diff()
        .fillna(df["Position"])
        .abs()
    )

    # Strategins dagliga nettoavkastning
    gross_return = exposure * df["Market_Return"]

    df["Strategy_Return"] = (
        (1 + gross_return) *
        (1 - FEE * df["Trades"])
        - 1
    )

    # Kapitalutveckling
    df["Strategy_Value"] = (
        INITIAL_CAPITAL *
        (1 + df["Strategy_Return"]).cumprod()
    )

    df["BuyHold_Value"] = (
        INITIAL_CAPITAL *
        (1 + df["Market_Return"]).cumprod()
    )

    # Drawdown
    for name in ["Strategy", "BuyHold"]:

        value = df[f"{name}_Value"]

        df[f"{name}_Drawdown"] = (
            value / value.cummax() - 1
        ) * 100

    def sharpe(returns):
        std = returns.std()

        if std == 0 or np.isnan(std):
            return np.nan

        # Kryptomarknaden handlas 365 dagar/år
        return (
            returns.mean() / std * np.sqrt(365)
        )

    summary = []

    for name, label, ret_col in [
        (
            "Strategy",
            "Teknisk strategi",
            "Strategy_Return"
        ),
        (
            "BuyHold",
            "Buy & Hold",
            "Market_Return"
        )
    ]:

        final_value = df[f"{name}_Value"].iloc[-1]

        summary.append({
            "Valuta": asset,
            "Strategi": label,
            "Startkapital USD": INITIAL_CAPITAL,
            "Slutkapital USD": final_value,
            "Total avkastning (%)": (
                final_value / INITIAL_CAPITAL - 1
            ) * 100,
            "Max Drawdown (%)": (
                df[f"{name}_Drawdown"].min()
            ),
            "Sharpe": sharpe(df[ret_col]),
            "Antal köp": (
                ((df["Position"].diff().fillna(
                    df["Position"]
                )) == 1).sum()
                if name == "Strategy" else 1
            ),
            "Antal sälj": (
                ((df["Position"].diff().fillna(
                    df["Position"]
                )) == -1).sum()
                if name == "Strategy" else 0
            ),
            "Investerad tid (%)": (
                df["Position"].mean() * 100
                if name == "Strategy" else 100
            )
        })

    return df, pd.DataFrame(summary)


# Kör separat för Ethereum och Bitcoin
df_eth_bt, result_eth = backtest_strategy(
    df_eth_signals, "ETH"
)

df_btc_bt, result_btc = backtest_strategy(
    df_btc_signals, "BTC"
)

df_backtest_results = pd.concat(
    [result_eth, result_btc],
    ignore_index=True
)

print("STEG 4 – RESULTAT BACKTESTING")

display(df_backtest_results.round(2))

print("\nTestperioder:")

for asset, df in [
    ("ETH", df_eth_bt),
    ("BTC", df_btc_bt)
]:
    print(
        asset,
        df["time"].min(),
        "till",
        df["time"].max()
    )


STEG 4 – RESULTAT BACKTESTING


,Valuta,Strategi,Startkapital USD,Slutkapital USD,Total avkastning (%),Max Drawdown (%),Sharpe,Antal köp,Antal sälj,Investerad tid (%)
0,ETH,Teknisk strategi,10000,11233426.25,112234.26,-78.54,1.24,16,15,56.65
1,ETH,Buy & Hold,10000,4046371.16,40363.71,-94.00,1.07,1,0,100.00
2,BTC,Teknisk strategi,10000,2977205.88,29672.06,-66.09,1.23,17,16,60.14
3,BTC,Buy & Hold,10000,2838365.14,28283.65,-83.78,1.09,1,0,100.00



Testperioder:
ETH 2016-02-23 00:00:00+00:00 till 2026-11-01 00:00:00+00:00
BTC 2015-07-19 00:00:00+00:00 till 2026-11-01 00:00:00+00:00


In [12]:

# STEG 4.2 – Kapitalutveckling ETH och BTC

import plotly.graph_objects as go

def plot_backtest(df, asset):

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["Strategy_Value"],
        name="Teknisk handelsstrategi",
        line=dict(color="#2ECC71", width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["BuyHold_Value"],
        name="Buy & Hold",
        line=dict(color="#5DADE2", width=2)
    ))

    fig.add_hline(
        y=INITIAL_CAPITAL,
        line_dash="dash",
        line_color="gray"
    )

    fig.update_layout(
        title=f"{asset} – Kapitalutveckling från 10 000 USD",
        xaxis_title="Datum",
        yaxis_title="Portföljvärde (USD)",
        template="plotly_dark",
        height=600,
        hovermode="x unified"
    )

    fig.update_xaxes(rangeslider_visible=True)

    fig.show()


plot_backtest(df_eth_bt, "ETH")
plot_backtest(df_btc_bt, "BTC")


In [13]:

# STEG 4.3 – Historisk drawdown

import plotly.graph_objects as go

def plot_drawdown(df, asset):

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["Strategy_Drawdown"],
        name="Teknisk strategi",
        line=dict(color="#2ECC71", width=2)
    ))

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["BuyHold_Drawdown"],
        name="Buy & Hold",
        line=dict(color="#FF5252", width=2)
    ))

    fig.update_layout(
        title=f"{asset} – Historisk drawdown",
        xaxis_title="Datum",
        yaxis_title="Drawdown (%)",
        template="plotly_dark",
        height=550,
        hovermode="x unified"
    )

    fig.update_xaxes(rangeslider_visible=True)

    fig.show()


plot_drawdown(df_eth_bt, "ETH")
plot_drawdown(df_btc_bt, "BTC")



## Steg 5 – Historisk mönsteranalys för ETH och BTC

### Forskningsfråga

Vilka kombinationer av tekniska indikatorer
har historiskt följts av gynnsam eller
ogynnsam framtida prisutveckling?

### Syfte

Vi undersöker samband mellan marknadens
historiska tillstånd och efterföljande
avkastning för Ethereum och Bitcoin.

### Variabler

- RSI 14: momentum
- SMA 200: långsiktig trend
- Bull Market Support Band: trendposition
- Volatilitet 30 dagar: historisk prisvariation

### Prognoshorisonter

- 7 dagar
- 30 dagar
- 90 dagar
- 180 dagar
- 365 dagar

### Metod

Vi klassificerar historiska observationer
utifrån indikatorernas värden.

Därefter beräknar vi framtida avkastning
för varje marknadstillstånd.

Vi jämför medianavkastning, andel positiva
utfall och risken för större förluster.

Vi analyserar ETH och BTC separat.

### Metodologiska begränsningar

Detta är en explorativ analys, inte en
färdig prognosmodell.

Historiska samband behöver senare
utvärderas på tidsmässigt separerade data.

Framtida avkastningsperioder överlappar,
vilket innebär att observationerna inte
är statistiskt oberoende.

Vi använder därför inte resultaten för
att optimera handelsregler på testperioden.


In [14]:

# STEG 5.1 – Mönster och framtida avkastning

import numpy as np
import pandas as pd
from IPython.display import display

HORIZONS = [7, 30, 90, 180, 365]

def build_pattern_data(data, asset):

    df = data.copy()
    df["time"] = pd.to_datetime(df["time"], utc=True)

    df = (
        df.sort_values("time")
        .drop_duplicates("time", keep="last")
        .set_index("time")
    )

    price_col = f"{asset}_Price"

    # Säkerställ en daglig tidsaxel
    full_dates = pd.date_range(
        df.index.min(), df.index.max(),
        freq="D", tz="UTC"
    )

    df = df.reindex(full_dates)
    df.index.name = "time"

    price = pd.to_numeric(
        df[price_col], errors="coerce"
    )

    # 30 dagars historisk volatilitet
    returns = price.pct_change(fill_method=None)

    df["Volatility_30D"] = (
        returns.rolling(30, min_periods=30)
        .std() * np.sqrt(365) * 100
    )

    # RSI-kategorier
    df["RSI_Zone"] = pd.cut(
        df["RSI_14"],
        bins=[-np.inf, 30, 50, 70, np.inf],
        labels=[
            "RSI under 30",
            "RSI 30–50",
            "RSI 50–70",
            "RSI över 70"
        ]
    )

    # Långsiktig trend
    df["Trend_200"] = np.select(
        [
            price > df["SMA_200"],
            price < df["SMA_200"]
        ],
        ["Över SMA 200", "Under SMA 200"],
        default="INGEN DATA"
    )

    # Position relativt BMSB
    valid_band = df[
        ["BMSB_Upper", "BMSB_Lower"]
    ].notna().all(axis=1)

    df["BMSB_Zone"] = np.select(
        [
            valid_band & (price > df["BMSB_Upper"]),
            valid_band & (price < df["BMSB_Lower"]),
            valid_band
        ],
        [
            "Över BMSB",
            "Under BMSB",
            "Inom BMSB"
        ],
        default="INGEN DATA"
    )

    # Volatilitetskategorier
    # 80 % är en fast explorativ gräns.
    df["Volatility_Zone"] = np.where(
        df["Volatility_30D"].isna(),
        "INGEN DATA",
        np.where(
            df["Volatility_30D"] >= 80,
            "Hög volatilitet",
            "Lägre volatilitet"
        )
    )

    # Framtida avkastning:
    # Signal vid dagens stängning.
    # Köp nästa dag, sälj efter h dagar.
    entry = price.shift(-1)

    df["Entry_Price"] = entry

    for h in HORIZONS:
        exit_price = price.shift(-(h + 1))

        df[f"Return_{h}D"] = (
            exit_price / entry - 1
        ) * 100

        df[f"Positive_{h}D"] = (
            df[f"Return_{h}D"] > 0
        ).where(df[f"Return_{h}D"].notna())

    # Endast datum med komplett indikatorhistorik
    required = [
        price_col, "RSI_14", "SMA_200",
        "BMSB_Upper", "BMSB_Lower",
        "Volatility_30D"
    ]

    df["Valid_Features"] = df[required].notna().all(axis=1)

    df = df.reset_index()

    return df


df_eth_patterns = build_pattern_data(df_eth_ind, "ETH")
df_btc_patterns = build_pattern_data(df_btc_ind, "BTC")

for asset, df in [
    ("ETH", df_eth_patterns),
    ("BTC", df_btc_patterns)
]:
    print(f"\n===== {asset} =====")
    print("Totala datum:", len(df))
    print("Kompletta indikatorer:", df["Valid_Features"].sum())

    print("\nGiltiga framtida utfall:")
    for h in HORIZONS:
        valid = (
            df["Valid_Features"]
            & df[f"Return_{h}D"].notna()
        )
        print(f"{h} dagar: {valid.sum()}")

    print("\nSenaste observationerna:")
    display(
        df[
            [
                "time", f"{asset}_Price",
                "RSI_Zone", "Trend_200",
                "BMSB_Zone", "Volatility_30D"
            ]
        ].tail(5)
    )



===== ETH =====
Totala datum: 4104
Kompletta indikatorer: 3905

Giltiga framtida utfall:
7 dagar: 3897
30 dagar: 3874
90 dagar: 3814
180 dagar: 3724
365 dagar: 3539

Senaste observationerna:


,time,ETH_Price,RSI_Zone,Trend_200,BMSB_Zone,Volatility_30D
4099,2026-10-28 00:00:00+00:00,2356.988509,RSI 30–50,Över SMA 200,Över BMSB,33.454349
4100,2026-10-29 00:00:00+00:00,2256.031994,RSI 30–50,Över SMA 200,Inom BMSB,36.060129
4101,2026-10-30 00:00:00+00:00,2240.229980,RSI 30–50,Över SMA 200,Inom BMSB,35.937829
4102,2026-10-31 00:00:00+00:00,2252.073358,RSI 30–50,Över SMA 200,Inom BMSB,35.886473
4103,2026-11-01 00:00:00+00:00,2269.988406,RSI 30–50,Över SMA 200,Inom BMSB,36.103785



===== BTC =====
Totala datum: 4323
Kompletta indikatorer: 4124

Giltiga framtida utfall:
7 dagar: 4116
30 dagar: 4093
90 dagar: 4033
180 dagar: 3943
365 dagar: 3758

Senaste observationerna:


,time,BTC_Price,RSI_Zone,Trend_200,BMSB_Zone,Volatility_30D
4318,2026-10-28 00:00:00+00:00,81237.437826,RSI 30–50,Över SMA 200,Över BMSB,25.298409
4319,2026-10-29 00:00:00+00:00,78193.426404,RSI 30–50,Över SMA 200,Över BMSB,28.310535
4320,2026-10-30 00:00:00+00:00,77109.590747,RSI 30–50,Över SMA 200,Över BMSB,28.599709
4321,2026-10-31 00:00:00+00:00,77401.017287,RSI 30–50,Över SMA 200,Över BMSB,28.046429
4322,2026-11-01 00:00:00+00:00,78135.080289,RSI 30–50,Över SMA 200,Över BMSB,28.378122


In [15]:

# STEG 5.2 – Sammanfattning av marknadsmönster

def summarize_patterns(df, asset):

    results = []

    valid_df = df[df["Valid_Features"]].copy()

    for h in HORIZONS:

        return_col = f"Return_{h}D"

        subset = valid_df.dropna(
            subset=[return_col]
        )

        grouped = subset.groupby(
            ["Trend_200", "RSI_Zone"],
            observed=True
        )

        for (trend, rsi), group in grouped:

            values = group[return_col]

            results.append({
                "Valuta": asset,
                "Horisont": f"{h}D",
                "Trend": trend,
                "RSI": str(rsi),
                "Observationer": len(group),
                "Medianavkastning (%)": values.median(),
                "Medelavkastning (%)": values.mean(),
                "Positiva utfall (%)": (
                    values.gt(0).mean() * 100
                ),
                "Förlust över 20 % (%)": (
                    values.lt(-20).mean() * 100
                )
            })

    return pd.DataFrame(results)


eth_pattern_results = summarize_patterns(
    df_eth_patterns, "ETH"
)

btc_pattern_results = summarize_patterns(
    df_btc_patterns, "BTC"
)

df_pattern_results = pd.concat(
    [eth_pattern_results, btc_pattern_results],
    ignore_index=True
)

# Visa 30-dagarsresultaten
for asset in ["ETH", "BTC"]:

    print(f"\n===== {asset} – 30 DAGAR =====")

    result = df_pattern_results[
        (df_pattern_results["Valuta"] == asset)
        & (df_pattern_results["Horisont"] == "30D")
    ].sort_values(
        "Medianavkastning (%)",
        ascending=False
    )

    display(result.round(2))



===== ETH – 30 DAGAR =====


,Valuta,Horisont,Trend,RSI,Observationer,Medianavkastning (%),Medelavkastning (%),Positiva utfall (%),Förlust över 20 % (%)
15,ETH,30D,Över SMA 200,RSI över 70,393,9.01,21.72,60.81,15.78
8,ETH,30D,Under SMA 200,RSI under 30,161,5.74,12.01,59.63,15.53
12,ETH,30D,Över SMA 200,RSI under 30,6,4.40,12.17,50.00,0.00
14,ETH,30D,Över SMA 200,RSI 50–70,1125,3.98,18.23,58.22,10.22
10,ETH,30D,Under SMA 200,RSI 50–70,485,3.86,4.80,56.49,17.32
9,ETH,30D,Under SMA 200,RSI 30–50,962,-0.75,1.56,48.02,21.41
11,ETH,30D,Under SMA 200,RSI över 70,32,-1.30,3.40,37.50,9.38
13,ETH,30D,Över SMA 200,RSI 30–50,710,-2.27,3.78,46.20,16.90



===== BTC – 30 DAGAR =====


,Valuta,Horisont,Trend,RSI,Observationer,Medianavkastning (%),Medelavkastning (%),Positiva utfall (%),Förlust över 20 % (%)
55,BTC,30D,Över SMA 200,RSI över 70,503,9.41,12.93,71.77,2.98
51,BTC,30D,Under SMA 200,RSI över 70,20,7.50,3.39,70.00,5.00
53,BTC,30D,Över SMA 200,RSI 30–50,702,3.41,4.88,56.98,8.69
52,BTC,30D,Över SMA 200,RSI under 30,16,2.82,20.49,62.50,6.25
48,BTC,30D,Under SMA 200,RSI under 30,134,2.54,2.80,65.67,5.22
54,BTC,30D,Över SMA 200,RSI 50–70,1323,2.49,8.98,53.89,6.73
49,BTC,30D,Under SMA 200,RSI 30–50,924,1.95,2.45,56.82,10.39
50,BTC,30D,Under SMA 200,RSI 50–70,471,1.72,3.99,53.50,10.83


In [16]:

# STEG 5.3 – Heatmaps över historisk avkastning

import plotly.express as px

RSI_ORDER = [
    "RSI under 30", "RSI 30–50",
    "RSI 50–70", "RSI över 70"
]

TREND_ORDER = [
    "Under SMA 200", "Över SMA 200"
]

def plot_pattern_heatmap(results, asset, horizon):

    subset = results[
        (results["Valuta"] == asset)
        & (results["Horisont"] == f"{horizon}D")
    ].copy()

    # Markera grupper med färre än 20 observationer
    subset.loc[
        subset["Observationer"] < 20,
        "Medianavkastning (%)"
    ] = np.nan

    pivot = subset.pivot(
        index="RSI",
        columns="Trend",
        values="Medianavkastning (%)"
    ).reindex(
        index=RSI_ORDER,
        columns=TREND_ORDER
    )

    fig = px.imshow(
        pivot,
        text_auto=".1f",
        color_continuous_scale="RdYlGn",
        color_continuous_midpoint=0,
        aspect="auto",
        title=(
            f"{asset} – Medianavkastning efter "
            f"{horizon} dagar (%)"
        ),
        labels={
            "x": "Långsiktig trend",
            "y": "RSI-kategori",
            "color": "Avkastning (%)"
        }
    )

    fig.update_layout(
        template="plotly_dark",
        height=480
    )

    fig.show()


for asset in ["ETH", "BTC"]:
    for horizon in [30, 180]:
        plot_pattern_heatmap(
            df_pattern_results,
            asset,
            horizon
        )


In [17]:

# STEG 5.4 – Andel positiva utfall per mönster

import plotly.express as px

for asset in ["ETH", "BTC"]:

    subset = df_pattern_results[
        (df_pattern_results["Valuta"] == asset)
        & (df_pattern_results["Horisont"] == "30D")
        & (df_pattern_results["Observationer"] >= 20)
    ].copy()

    fig = px.bar(
        subset,
        x="RSI",
        y="Positiva utfall (%)",
        color="Trend",
        barmode="group",
        text_auto=".1f",
        category_orders={"RSI": RSI_ORDER},
        hover_data=["Observationer"],
        title=(
            f"{asset} – Andel positiv avkastning "
            "efter 30 dagar"
        ),
        template="plotly_dark"
    )

    fig.add_hline(
        y=50,
        line_dash="dash",
        line_color="white"
    )

    fig.update_layout(
        height=530,
        yaxis_range=[0, 100],
        yaxis_title="Positiva utfall (%)",
        xaxis_title="RSI-kategori"
    )

    fig.show()



### Resultat och slutsats – Steg 5

#### Historiska mönster

Vi undersökte hur RSI 14 och SMA 200
samvarierar med framtida avkastning för
Ethereum och Bitcoin.

Analysen omfattade 7, 30, 90, 180
och 365 dagars avkastning.

#### Resultat efter 30 dagar

För Ethereum hade kombinationen RSI över 70
och pris över SMA 200:

- Medianavkastning: +9,01 %
- Positiva utfall: 60,81 %
- Observationer: 393

För Bitcoin hade samma kombination:

- Medianavkastning: +9,41 %
- Positiva utfall: 71,77 %
- Observationer: 503

#### Resultat efter 180 dagar

När RSI låg över 70 och priset över SMA 200
var medianavkastningen:

- Ethereum: +25,5 %
- Bitcoin: +40,6 %

#### Tolkning

Resultaten indikerar att starkt momentum
i en positiv långsiktig trend historiskt
ofta har följts av fortsatt prisuppgång.

RSI över 70 behöver därför inte automatiskt
vara en säljsignal.

#### Begränsningar

Resultaten är explorativa och bygger
på överlappande framtida avkastningsperioder.

Samma marknadstrend kan därför påverka
många observationer.

Vi har ännu inte verifierat att mönstren
har prognosvärde på oberoende testdata.

#### Preliminär slutsats

Kombinationen av RSI och SMA 200 visar
intressanta historiska samband för både
Ethereum och Bitcoin.

Resultaten motiverar fortsatt analys
med tidsbaserad validering och
maskininlärning.

Nästa steg är att undersöka om modellerna
kan identifiera gynnsamma köp- och
säljperioder även på senare data.


In [18]:

# STEG 6.1 – Tidsbaserad kontroll av mönster

import pandas as pd
import numpy as np
from IPython.display import display

results = []

for asset, df in [
    ("ETH", df_eth_patterns),
    ("BTC", df_btc_patterns)
]:

    data = df.copy()
    data["time"] = pd.to_datetime(data["time"], utc=True)

    # Mönster att undersöka
    pattern = (
        (data["RSI_Zone"].astype(str) == "RSI över 70")
        & (data["Trend_200"] == "Över SMA 200")
        & data["Valid_Features"]
    )

    for horizon in [30, 180]:

        return_col = f"Return_{horizon}D"

        # Utvecklingsperiod: alla framtida utfall
        # måste vara kända före 2024-01-01
        train_end = (
            pd.Timestamp("2024-01-01", tz="UTC")
            - pd.Timedelta(days=horizon + 1)
        )

        periods = {
            "Utveckling": data["time"] < train_end,
            "Senare period": (
                data["time"] >= pd.Timestamp(
                    "2024-01-01", tz="UTC"
                )
            )
        }

        for period, period_mask in periods.items():

            subset = data.loc[
                pattern & period_mask
            ].dropna(subset=[return_col])

            returns = subset[return_col]

            results.append({
                "Valuta": asset,
                "Horisont": f"{horizon}D",
                "Period": period,
                "Observationer": len(subset),
                "Medianavkastning (%)": (
                    returns.median()
                    if len(returns) else np.nan
                ),
                "Positiva utfall (%)": (
                    (returns > 0).mean() * 100
                    if len(returns) else np.nan
                )
            })

df_pattern_validation = pd.DataFrame(results)

print("TIDSBASERAD MÖNSTERJÄMFÖRELSE")
display(df_pattern_validation.round(2))


TIDSBASERAD MÖNSTERJÄMFÖRELSE


,Valuta,Horisont,Period,Observationer,Medianavkastning (%),Positiva utfall (%)
0,ETH,30D,Utveckling,302,8.57,59.27
1,ETH,30D,Senare period,85,10.03,68.24
2,ETH,180D,Utveckling,286,53.88,71.68
3,ETH,180D,Senare period,73,-20.44,6.85
4,BTC,30D,Utveckling,407,11.22,70.02
5,BTC,30D,Senare period,88,6.39,78.41
6,BTC,180D,Utveckling,381,47.64,81.36
7,BTC,180D,Senare period,73,0.95,53.42


In [19]:

# STEG 6.2 – Visualisera tidsbaserad jämförelse

import plotly.express as px

fig = px.bar(
    df_pattern_validation,
    x="Period",
    y="Medianavkastning (%)",
    color="Valuta",
    facet_col="Horisont",
    barmode="group",
    text_auto=".1f",
    hover_data=["Observationer"],
    title="ETH och BTC – Håller mönstret över tid?",
    template="plotly_dark",
    category_orders={
        "Period": ["Utveckling", "Senare period"]
    }
)

fig.add_hline(y=0, line_dash="dash", line_color="white")

fig.update_layout(height=550)
fig.show()



### Resultat och slutsats – Steg 6

#### Syfte

Vi undersökte om historiska marknadsmönster
behåller sina samband med framtida avkastning
under senare marknadsperioder.

Vi analyserade kombinationen:

- RSI över 70
- Pris över SMA 200

Vi jämförde framtida avkastning efter
30 och 180 dagar för Ethereum och Bitcoin.

#### Resultat – 30 dagar

Ethereum:
- Utvecklingsperiod: +8,57 %
- Senare period: +10,03 %

Bitcoin:
- Utvecklingsperiod: +11,22 %
- Senare period: +6,39 %

Medianavkastningen var positiv under båda
perioderna för båda kryptovalutorna.

#### Resultat – 180 dagar

Ethereum:
- Utvecklingsperiod: +53,88 %
- Senare period: -20,44 %

Bitcoin:
- Utvecklingsperiod: +47,64 %
- Senare period: +0,95 %

Resultaten visar en betydande försämring
på den längre prognoshorisonten.

#### Tolkning

Starkt momentum i en positiv långsiktig
trend har historiskt ofta följts av
positiv avkastning på 30 dagars sikt.

Sambandet är däremot inte lika stabilt
på 180 dagars sikt.

Marknadsförhållandena påverkar därför
hur indikatorerna bör tolkas.

#### Begränsningar

Analysen bygger på överlappande observationer
och en begränsad senare utvärderingsperiod.

Mönstret valdes efter explorativ analys
av den historiska datan.

Resultaten är därför inte ett helt
oberoende test av prognosförmåga.

#### Preliminär slutsats

Resultaten motiverar fortsatt analys
av kortsiktiga och medellånga prognoser.

Vi behöver jämföra identifierade mönster
mot marknadens genomsnittliga utveckling
och testa fler indikatorer.

Nästa steg är att undersöka om
maskininlärning kan bidra med ytterligare
prognosinformation.


In [20]:

# STEG 7.1 – Jämför mönster med marknadsreferens

import pandas as pd
import numpy as np
from IPython.display import display

results = []

for asset, df in [
    ("ETH", df_eth_patterns),
    ("BTC", df_btc_patterns)
]:

    data = df.copy()
    data["time"] = pd.to_datetime(
        data["time"], utc=True
    )

    data["Year"] = data["time"].dt.year

    pattern = (
        (data["RSI_Zone"].astype(str) == "RSI över 70")
        & (data["Trend_200"] == "Över SMA 200")
    )

    for horizon in [30, 180]:

        ret_col = f"Return_{horizon}D"

        valid = (
            data["Valid_Features"]
            & data[ret_col].notna()
        )

        for year in sorted(data["Year"].unique()):

            year_mask = data["Year"] == year

            # Referens: samtliga giltiga dagar samma år
            baseline = data.loc[
                valid & year_mask,
                ret_col
            ]

            # Dagar med identifierat mönster
            selected = data.loc[
                valid & year_mask & pattern,
                ret_col
            ]

            if len(selected) < 10 or len(baseline) < 10:
                continue

            results.append({
                "Valuta": asset,
                "År": year,
                "Horisont": f"{horizon}D",
                "Observationer": len(selected),
                "Median mönster (%)": selected.median(),
                "Median referens (%)": baseline.median(),
                "Skillnad (pp)": (
                    selected.median() - baseline.median()
                ),
                "Positiva utfall (%)": (
                    selected.gt(0).mean() * 100
                )
            })

df_market_comparison = pd.DataFrame(results)

print("STEG 7 – MÖNSTER JÄMFÖRT MED MARKNADEN")

for asset in ["ETH", "BTC"]:

    print(f"\n===== {asset} =====")

    display(
        df_market_comparison[
            df_market_comparison["Valuta"] == asset
        ].round(2)
    )


STEG 7 – MÖNSTER JÄMFÖRT MED MARKNADEN

===== ETH =====


,Valuta,År,Horisont,Observationer,Median mönster (%),Median referens (%),Skillnad (pp),Positiva utfall (%)
0,ETH,2016,30D,28,-11.03,-2.54,-8.49,35.71
1,ETH,2017,30D,97,62.40,45.50,16.90,78.35
2,ETH,2018,30D,16,-27.74,-17.72,-10.02,6.25
3,ETH,2019,30D,20,-0.19,2.16,-2.34,50.00
4,ETH,2020,30D,52,-3.09,20.23,-23.32,44.23
5,ETH,2021,30D,56,5.92,7.88,-1.96,57.14
6,ETH,2023,30D,37,7.64,2.43,5.21,78.38
7,ETH,2024,30D,46,1.03,-1.37,2.40,50.00
8,ETH,2025,30D,27,18.89,-2.99,21.88,92.59
9,ETH,2026,30D,12,9.05,3.93,5.12,83.33



===== BTC =====


,Valuta,År,Horisont,Observationer,Median mönster (%),Median referens (%),Skillnad (pp),Positiva utfall (%)
19,BTC,2015,30D,31,8.21,8.39,-0.18,67.74
20,BTC,2016,30D,52,2.59,6.04,-3.45,53.85
21,BTC,2017,30D,109,15.86,26.08,-10.22,59.63
22,BTC,2019,30D,39,18.37,5.96,12.42,84.62
23,BTC,2020,30D,83,21.55,17.19,4.36,80.72
24,BTC,2021,30D,37,5.91,-1.69,7.60,70.27
25,BTC,2023,30D,64,8.81,5.38,3.43,81.25
26,BTC,2024,30D,52,13.80,5.21,8.59,82.69
27,BTC,2025,30D,21,1.16,-0.85,2.01,61.90
28,BTC,2026,30D,15,7.36,2.00,5.36,86.67


In [21]:

# STEG 7.2 – Mönstrets resultat mot referensen

import plotly.express as px

for asset in ["ETH", "BTC"]:

    subset = df_market_comparison[
        df_market_comparison["Valuta"] == asset
    ].copy()

    fig = px.bar(
        subset,
        x="År",
        y="Skillnad (pp)",
        color="Horisont",
        barmode="group",
        text_auto=".1f",
        hover_data=["Observationer"],
        title=(
            f"{asset} – Mönstrets medianavkastning "
            "jämfört med marknadsreferensen"
        ),
        template="plotly_dark"
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="white"
    )

    fig.update_layout(
        height=550,
        xaxis_title="År",
        yaxis_title="Skillnad i medianavkastning (procentenheter)",
        hovermode="x unified"
    )

    fig.update_xaxes(dtick=1)

    fig.show()



### Resultat och slutsats – Steg 7

#### Syfte

Vi undersökte om ett historiskt marknadsmönster
gav bättre framtida avkastning än marknadens
normala utveckling under samma år.

Mönstret definierades som:

- RSI över 70
- Pris över SMA 200

Vi analyserade Ethereum och Bitcoin separat
med prognoshorisonter på 30 och 180 dagar.

#### Resultat – Ethereum

Resultaten varierade mellan olika år.

År 2025 överträffade mönstret marknadens
medianavkastning med 21,88 procentenheter
på 30 dagars sikt.

På 180 dagars sikt var resultatet däremot
13,09 procentenheter sämre än referensen.

#### Resultat – Bitcoin

Bitcoin visade positiva skillnader på
30 dagars sikt under flera senare år.

År 2024 var skillnaden +8,59 procentenheter
på 30 dagars sikt, men -18,29
procentenheter på 180 dagars sikt.

#### Tolkning

Resultaten visar att ett marknadsmönster
kan fungera olika beroende på marknadsläge
och prognoshorisont.

Positiv historisk avkastning betyder inte
automatiskt att mönstret överträffar
marknadens normala utveckling.

#### Begränsningar

- Resultaten är explorativa.
- Avkastningsperioderna överlappar.
- Vissa år har få observationer.
- Mönstret har undersökts med historiska data
  som redan använts i tidigare analyser.
- Resultaten är inte bevis på framtida
  prognosförmåga.

#### Slutsats

Vi har identifierat historiska mönster
som kan vara relevanta för köp- och
säljbedömningar.

Mönstrens resultat är däremot inte
tillräckligt stabila över tid för att
ensamma användas som handelssignaler.

Nästa steg är att undersöka om
maskininlärning kan kombinera flera
indikatorer och förbättra prognosförmågan.

Vi kommer att utvärdera modellerna med
tidsbaserad validering och jämföra dem
mot enkla referensmodeller.



## Steg 8 – Machine Learning: Prognos av prisriktning

### Syfte

Vi undersöker om maskininlärning kan
identifiera historiska marknadsmönster
som har samband med framtida prisutveckling
för Ethereum och Bitcoin.

Vi vill uppskatta sannolikheten för
positiv avkastning efter 30 och 180 dagar.

### Modeller

Vi planerar att jämföra:

1. Baseline – historisk andel positiva utfall.
2. Logistic Regression – linjär klassificeringsmodell.
3. Random Forest – icke-linjär klassificeringsmodell.

### Variabler

- RSI 14
- Avstånd från SMA 200
- Avstånd från Bull Market Support Band
- Historisk volatilitet 30 dagar
- Historisk avkastning 30 dagar

### Datadelning

- Träning: före 2023
- Validering: 2023–2024
- Test: från 2025

Vi säkerställer att framtida utfallsperioder
inte går över gränsen mellan tränings-
och valideringsdata.

Testperioden sparas för slutlig utvärdering.

### Utvärdering

Vi kommer att använda:

- Brier Score
- ROC-AUC
- Balanced Accuracy
- Resultat jämfört med Baseline

Vi undersöker både prognosprecision
och om modellen kan bidra till
historiskt gynnsamma handelsbeslut.

Resultaten får inte tolkas som
garanterade köp- eller säljsignaler.


In [22]:

# STEG 8.1 – Skapa ML-dataset för ETH och BTC

import numpy as np
import pandas as pd
from IPython.display import display

ML_HORIZONS = [30, 180]

FEATURE_COLUMNS = [
    "RSI_14",
    "SMA_Distance_Pct",
    "BMSB_Distance_Pct",
    "Volatility_30D",
    "Return_30D_History"
]

TRAIN_END = pd.Timestamp("2023-01-01", tz="UTC")
VALID_END = pd.Timestamp("2025-01-01", tz="UTC")


def build_ml_dataset(data, asset):

    df = data.copy().sort_values("time")

    df["time"] = pd.to_datetime(
        df["time"], utc=True
    )

    price_col = f"{asset}_Price"

    price = pd.to_numeric(
        df[price_col], errors="coerce"
    )

    # Historiskt avstånd från SMA 200
    df["SMA_Distance_Pct"] = (
        price / df["SMA_200"] - 1
    ) * 100

    # Historiskt avstånd från BMSB
    band_middle = (
        df["BMSB_Upper"] + df["BMSB_Lower"]
    ) / 2

    df["BMSB_Distance_Pct"] = (
        price / band_middle - 1
    ) * 100

    # Historisk 30-dagarsavkastning
    df["Return_30D_History"] = (
        price / price.shift(30) - 1
    ) * 100

    return df


df_eth_ml = build_ml_dataset(
    df_eth_patterns, "ETH"
)

df_btc_ml = build_ml_dataset(
    df_btc_patterns, "BTC"
)


def split_ml_data(data, horizon):

    df = data.copy()

    target_col = f"Return_{horizon}D"
    label_col = f"Target_Up_{horizon}D"

    # Klassificeringsmål: upp eller ned
    df[label_col] = np.where(
        df[target_col].notna(),
        (df[target_col] > 0).astype(int),
        np.nan
    )

    # Endast kompletta observationer
    df = df.dropna(
        subset=FEATURE_COLUMNS + [label_col]
    ).copy()

    # Känt utfall efter nästa dags köp
    # och h dagars innehav
    df["Outcome_End"] = (
        df["time"]
        + pd.to_timedelta(horizon + 1, unit="D")
    )

    # Träning: hela utfallsperioden före 2023
    train = df[
        df["Outcome_End"] < TRAIN_END
    ].copy()

    # Validering: start 2023, utfall före 2025
    validation = df[
        (df["time"] >= TRAIN_END)
        & (df["Outcome_End"] < VALID_END)
    ].copy()

    # Test: observationer från 2025
    # med känt framtida utfall
    test = df[
        df["time"] >= VALID_END
    ].copy()

    return train, validation, test, label_col


ml_splits = {}
summary_rows = []

for asset, data in [
    ("ETH", df_eth_ml),
    ("BTC", df_btc_ml)
]:

    for horizon in ML_HORIZONS:

        train, val, test, target = split_ml_data(
            data, horizon
        )

        ml_splits[(asset, horizon)] = {
            "train": train,
            "validation": val,
            "test": test,
            "target": target
        }

        for name, subset in [
            ("Träning", train),
            ("Validering", val),
            ("Test", test)
        ]:

            summary_rows.append({
                "Valuta": asset,
                "Horisont": f"{horizon}D",
                "Dataset": name,
                "Observationer": len(subset),
                "Första datum": (
                    subset["time"].min().date()
                    if len(subset) else None
                ),
                "Sista datum": (
                    subset["time"].max().date()
                    if len(subset) else None
                ),
                "Positiva utfall (%)": (
                    subset[target].mean() * 100
                    if len(subset) else np.nan
                )
            })

df_ml_summary = pd.DataFrame(summary_rows)

print("STEG 8.1 – DATAUPPDELNING")
display(df_ml_summary.round(2))


STEG 8.1 – DATAUPPDELNING


,Valuta,Horisont,Dataset,Observationer,Första datum,Sista datum,Positiva utfall (%)
0,ETH,30D,Träning,2473,2016-02-23,2022-11-30,54.47
1,ETH,30D,Validering,700,2023-01-01,2024-11-30,55.14
2,ETH,30D,Test,639,2025-01-01,2026-10-01,48.51
3,ETH,180D,Träning,2323,2016-02-23,2022-07-03,62.81
4,ETH,180D,Validering,550,2023-01-01,2024-07-03,67.64
5,ETH,180D,Test,489,2025-01-01,2026-05-04,50.92
6,BTC,30D,Träning,2692,2015-07-19,2022-11-30,57.32
7,BTC,30D,Validering,700,2023-01-01,2024-11-30,64.29
8,BTC,30D,Test,639,2025-01-01,2026-10-01,51.33
9,BTC,180D,Träning,2542,2015-07-19,2022-07-03,63.34


In [23]:

# STEG 8.2 – Träning och validering

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import (
    brier_score_loss,
    roc_auc_score,
    balanced_accuracy_score
)
from IPython.display import display

model_results = []
trained_models = {}
validation_predictions = []

for (asset, horizon), splits in ml_splits.items():

    train = splits["train"]
    val = splits["validation"]
    target = splits["target"]

    if train.empty or val.empty:
        print(f"Hoppar över {asset} {horizon}D: för lite data")
        continue

    X_train = train[FEATURE_COLUMNS]
    y_train = train[target].astype(int)

    X_val = val[FEATURE_COLUMNS]
    y_val = val[target].astype(int)

    if y_train.nunique() < 2:
        print(f"Hoppar över {asset} {horizon}D: en träningsklass")
        continue

    # Fasta modellinställningar i första experimentet
    models = {
        "Baseline": DummyClassifier(
            strategy="prior"
        ),
        "Logistic Regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                max_iter=2000,
                C=1.0
            )
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=200,
            max_depth=5,
            min_samples_leaf=30,
            random_state=42,
            n_jobs=-1
        )
    }

    for model_name, model in models.items():

        model.fit(X_train, y_train)

        probabilities = model.predict_proba(X_val)[:, 1]
        predictions = (probabilities >= 0.5).astype(int)

        auc = (
            roc_auc_score(y_val, probabilities)
            if y_val.nunique() == 2
            else np.nan
        )

        model_results.append({
            "Valuta": asset,
            "Horisont": f"{horizon}D",
            "Modell": model_name,
            "Observationer": len(val),
            "Brier Score": brier_score_loss(
                y_val, probabilities
            ),
            "ROC-AUC": auc,
            "Balanced Accuracy": balanced_accuracy_score(
                y_val, predictions
            )
        })

        trained_models[(asset, horizon, model_name)] = model

        prediction_df = pd.DataFrame({
            "time": val["time"].values,
            "Valuta": asset,
            "Horisont": f"{horizon}D",
            "Modell": model_name,
            "Sannolikhet": probabilities,
            "Faktiskt_utfall": y_val.to_numpy()
        })

        validation_predictions.append(prediction_df)

df_ml_results = pd.DataFrame(model_results)

df_validation_predictions = pd.concat(
    validation_predictions,
    ignore_index=True
)

print("MODELLJÄMFÖRELSE – VALIDERING 2023–2024")

display(
    df_ml_results.sort_values(
        ["Valuta", "Horisont", "Brier Score"]
    ).round(4)
)


MODELLJÄMFÖRELSE – VALIDERING 2023–2024


,Valuta,Horisont,Modell,Observationer,Brier Score,ROC-AUC,Balanced Accuracy
11,BTC,180D,Random Forest,550,0.1410,0.6830,0.4595
10,BTC,180D,Logistic Regression,550,0.1501,0.5295,0.4925
9,BTC,180D,Baseline,550,0.1737,0.5000,0.5000
7,BTC,30D,Logistic Regression,700,0.2321,0.5413,0.4842
6,BTC,30D,Baseline,700,0.2344,0.5000,0.5000
8,BTC,30D,Random Forest,700,0.2645,0.5365,0.5313
5,ETH,180D,Random Forest,550,0.2026,0.7744,0.5000
3,ETH,180D,Baseline,550,0.2212,0.5000,0.5000
4,ETH,180D,Logistic Regression,550,0.2381,0.2547,0.5000
0,ETH,30D,Baseline,700,0.2474,0.5000,0.5000


In [24]:

# STEG 8.3 – Interaktiva diagram

import plotly.express as px

for asset in ["ETH", "BTC"]:

    subset = df_ml_results[
        df_ml_results["Valuta"] == asset
    ].copy()

    fig = px.bar(
        subset,
        x="Modell",
        y="Brier Score",
        color="Modell",
        facet_col="Horisont",
        text_auto=".4f",
        title=f"{asset} – ML-modeller: Brier Score",
        template="plotly_dark"
    )

    fig.update_layout(
        height=550,
        showlegend=False,
        yaxis_title="Brier Score (lägre är bättre)"
    )

    fig.show()


In [25]:

# STEG 9.1 – Expanding Window Cross-Validation

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import brier_score_loss, roc_auc_score
from IPython.display import display

cv_results = []

for asset, source in [
    ("ETH", df_eth_ml),
    ("BTC", df_btc_ml)
]:

    for horizon in [30, 180]:

        target = f"Return_{horizon}D"

        df = source.copy()
        df["time"] = pd.to_datetime(df["time"], utc=True)

        df = df.dropna(
            subset=FEATURE_COLUMNS + [target]
        ).copy()

        df["Target"] = (df[target] > 0).astype(int)

        df["Outcome_End"] = (
            df["time"] +
            pd.Timedelta(days=horizon + 1)
        )

        for year in [2023, 2024]:

            start = pd.Timestamp(
                f"{year}-01-01", tz="UTC"
            )

            end = pd.Timestamp(
                f"{year+1}-01-01", tz="UTC"
            )

            # Träningsutfallen måste vara kända
            # innan valideringsåret börjar
            train = df[
                df["Outcome_End"] < start
            ].copy()

            # Validering inom aktuellt år
            validation = df[
                (df["time"] >= start)
                & (df["Outcome_End"] < end)
            ].copy()

            if train.empty or validation.empty:
                continue

            X_train = train[FEATURE_COLUMNS]
            y_train = train["Target"]

            X_val = validation[FEATURE_COLUMNS]
            y_val = validation["Target"]

            if y_train.nunique() < 2:
                continue

            models = {
                "Baseline": DummyClassifier(
                    strategy="prior"
                ),
                "Logistic Regression": make_pipeline(
                    StandardScaler(),
                    LogisticRegression(
                        max_iter=2000,
                        C=1.0
                    )
                ),
                "Random Forest": RandomForestClassifier(
                    n_estimators=200,
                    max_depth=5,
                    min_samples_leaf=30,
                    random_state=42,
                    n_jobs=-1
                )
            }

            for name, model in models.items():

                model.fit(X_train, y_train)

                probabilities = model.predict_proba(
                    X_val
                )[:, 1]

                brier = brier_score_loss(
                    y_val, probabilities
                )

                auc = (
                    roc_auc_score(y_val, probabilities)
                    if y_val.nunique() == 2
                    else np.nan
                )

                cv_results.append({
                    "Valuta": asset,
                    "Horisont": f"{horizon}D",
                    "År": year,
                    "Modell": name,
                    "Träning": len(train),
                    "Validering": len(validation),
                    "Positiva utfall (%)": (
                        y_val.mean() * 100
                    ),
                    "Brier Score": brier,
                    "ROC-AUC": auc
                })

df_cv_results = pd.DataFrame(cv_results)

print("EXPANDING WINDOW – RESULTAT")

display(
    df_cv_results.sort_values(
        ["Valuta", "Horisont", "År", "Brier Score"]
    ).round(4)
)


EXPANDING WINDOW – RESULTAT


,Valuta,Horisont,År,Modell,Träning,Validering,Positiva utfall (%),Brier Score,ROC-AUC
20,BTC,180D,2023,Random Forest,2542,184,81.5217,0.1552,0.7747
19,BTC,180D,2023,Logistic Regression,2542,184,81.5217,0.1728,0.4192
18,BTC,180D,2023,Baseline,2542,184,81.5217,0.1837,0.5000
23,BTC,180D,2024,Random Forest,2907,185,74.5946,0.1672,0.8108
22,BTC,180D,2024,Logistic Regression,2907,185,74.5946,0.1806,0.6300
21,BTC,180D,2024,Baseline,2907,185,74.5946,0.1979,0.5000
13,BTC,30D,2023,Logistic Regression,2692,334,65.2695,0.2294,0.5678
12,BTC,30D,2023,Baseline,2692,334,65.2695,0.2330,0.5000
14,BTC,30D,2023,Random Forest,2692,334,65.2695,0.2725,0.4966
16,BTC,30D,2024,Logistic Regression,3057,335,64.7761,0.2290,0.5374


In [26]:

# STEG 9.2 – Modellernas stabilitet över tid

import plotly.express as px

for asset in ["ETH", "BTC"]:

    subset = df_cv_results[
        df_cv_results["Valuta"] == asset
    ].copy()

    subset["År"] = subset["År"].astype(str)

    fig = px.bar(
        subset,
        x="År",
        y="Brier Score",
        color="Modell",
        facet_col="Horisont",
        barmode="group",
        text_auto=".3f",
        title=f"{asset} – Expanding Window Validation",
        template="plotly_dark",
        hover_data=[
            "Träning",
            "Validering",
            "Positiva utfall (%)"
        ]
    )

    fig.update_layout(
        height=600,
        yaxis_title="Brier Score – lägre är bättre"
    )

    fig.show()


In [27]:

# STEG 10.1 – Historisk testutvärdering 2025–2026

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss, roc_auc_score
from IPython.display import display

test_results = []
test_predictions = []

TEST_START = pd.Timestamp("2025-01-01", tz="UTC")

for asset, source in [
    ("ETH", df_eth_ml),
    ("BTC", df_btc_ml)
]:

    for horizon in [30, 180]:

        df = source.copy()

        df["time"] = pd.to_datetime(
            df["time"], utc=True
        )

        target_col = f"Return_{horizon}D"

        df = df.dropna(
            subset=FEATURE_COLUMNS + [target_col]
        ).copy()

        df["Target"] = (
            df[target_col] > 0
        ).astype(int)

        df["Outcome_End"] = (
            df["time"] +
            pd.Timedelta(days=horizon + 1)
        )

        # Endast utfall kända före 2025 används
        train = df[
            df["Outcome_End"] < TEST_START
        ].copy()

        # Testobservationer från 2025 och framåt
        test = df[
            df["time"] >= TEST_START
        ].copy()

        if train.empty or test.empty:
            continue

        X_train = train[FEATURE_COLUMNS]
        y_train = train["Target"]

        X_test = test[FEATURE_COLUMNS]
        y_test = test["Target"]

        if y_train.nunique() < 2:
            continue

        models = {
            "Baseline": DummyClassifier(
                strategy="prior"
            ),
            "Logistic Regression": make_pipeline(
                StandardScaler(),
                LogisticRegression(
                    max_iter=2000,
                    C=1.0
                )
            ),
            "Random Forest": RandomForestClassifier(
                n_estimators=200,
                max_depth=5,
                min_samples_leaf=30,
                random_state=42,
                n_jobs=-1
            )
        }

        for name, model in models.items():

            model.fit(X_train, y_train)

            prob = model.predict_proba(X_test)[:, 1]

            auc = (
                roc_auc_score(y_test, prob)
                if y_test.nunique() == 2
                else np.nan
            )

            test_results.append({
                "Valuta": asset,
                "Horisont": f"{horizon}D",
                "Modell": name,
                "Testobservationer": len(test),
                "Brier Score": brier_score_loss(
                    y_test, prob
                ),
                "ROC-AUC": auc
            })

            test_predictions.append(pd.DataFrame({
                "time": test["time"].to_numpy(),
                "Valuta": asset,
                "Horisont": f"{horizon}D",
                "Modell": name,
                "Sannolikhet": prob,
                "Faktiskt_Utfall": y_test.to_numpy()
            }))

df_final_test = pd.DataFrame(test_results)

df_final_predictions = pd.concat(
    test_predictions, ignore_index=True
)

print("STEG 10 – HISTORISK TESTPERIOD 2025–2026")

display(
    df_final_test.sort_values(
        ["Valuta", "Horisont", "Brier Score"]
    ).round(4)
)


STEG 10 – HISTORISK TESTPERIOD 2025–2026


,Valuta,Horisont,Modell,Testobservationer,Brier Score,ROC-AUC
9,BTC,180D,Baseline,489,0.3137,0.5000
10,BTC,180D,Logistic Regression,489,0.3948,0.2742
11,BTC,180D,Random Forest,489,0.5095,0.2368
6,BTC,30D,Baseline,639,0.2556,0.5000
7,BTC,30D,Logistic Regression,639,0.2637,0.4359
8,BTC,30D,Random Forest,639,0.3084,0.3744
3,ETH,180D,Baseline,489,0.2688,0.5000
4,ETH,180D,Logistic Regression,489,0.2917,0.2033
5,ETH,180D,Random Forest,489,0.2932,0.4693
1,ETH,30D,Logistic Regression,639,0.2443,0.6040


In [28]:

# STEG 10.2 – Jämför modeller på testperioden

import plotly.express as px

for asset in ["ETH", "BTC"]:

    subset = df_final_test[
        df_final_test["Valuta"] == asset
    ].copy()

    fig = px.bar(
        subset,
        x="Modell",
        y="Brier Score",
        color="Modell",
        facet_col="Horisont",
        text_auto=".4f",
        hover_data=["Testobservationer", "ROC-AUC"],
        title=f"{asset} – Modellernas resultat 2025–2026",
        template="plotly_dark"
    )

    fig.update_layout(
        height=550,
        showlegend=False,
        yaxis_title="Brier Score (lägre är bättre)"
    )

    fig.show()



### Resultat och slutsats – Steg 10

#### Syfte

Vi undersökte hur våra maskininlärningsmodeller
presterade på en senare historisk testperiod
under 2025–2026.

Målet var att kontrollera om modellerna kunde
identifiera framtida prisriktning för
Ethereum och Bitcoin.

#### Resultat – Ethereum

På 30 dagars sikt gav Logistic Regression
bäst resultat:

- Brier Score: 0,2443
- ROC-AUC: 0,6040
- Baseline Brier Score: 0,2538

Modellen visade en begränsad förbättring
jämfört med Baseline.

På 180 dagars sikt gav Baseline bäst
Brier Score (0,2688).

Ingen ML-modell överträffade Baseline
på denna längre prognoshorisont.

#### Resultat – Bitcoin

På både 30 och 180 dagars sikt
hade Baseline lägst Brier Score.

Random Forest, som tidigare fungerade
bra på 180 dagar, försämrades kraftigt:

- Validering Brier Score: 0,1410
- Test Brier Score: 0,5095
- Test ROC-AUC: 0,2368

#### Tolkning

Resultaten visar att modellernas
prognosförmåga förändras mellan olika
historiska marknadsperioder.

En modell som fungerar bra under en
tidigare period behöver inte fungera
under senare marknadsförhållanden.

Det kan bero på förändrade samband
mellan indikatorer och prisrörelser.

#### Begränsningar

Framtida avkastningsperioder överlappar,
vilket begränsar observationernas
statistiska oberoende.

Den senare testperioden har också
undersökts i tidigare explorativa analyser
och kan därför inte betraktas som helt
oberoende av projektets modellutveckling.

Resultaten mäter prognoser av prisriktning,
inte faktisk handelsvinst.

#### Slutsats

Logistic Regression för Ethereum på
30 dagar är en kandidat för vidare analys.

För Bitcoin gav ML-modellerna inte
tillräckligt stabila förbättringar.

Vi behöver därför undersöka
modellernas sannolikhetsprognoser,
marknadsförändringar och prognosfel
innan vi utvecklar köp- och säljsignaler
för Streamlit.


In [29]:

# STEG 11.1 – Prognoser jämfört med faktiska utfall

import pandas as pd
import numpy as np
from IPython.display import display

df_diagnostics = df_final_predictions.copy()

df_diagnostics["time"] = pd.to_datetime(
    df_diagnostics["time"], utc=True
)

df_diagnostics["År"] = df_diagnostics["time"].dt.year

diagnostics = (
    df_diagnostics
    .groupby(
        ["Valuta", "Horisont", "Modell", "År"],
        as_index=False
    )
    .agg(
        Observationer=("Faktiskt_Utfall", "size"),
        Prognos=("Sannolikhet", "mean"),
        Faktiskt=("Faktiskt_Utfall", "mean")
    )
)

diagnostics["Prognos (%)"] = (
    diagnostics["Prognos"] * 100
)

diagnostics["Faktiskt utfall (%)"] = (
    diagnostics["Faktiskt"] * 100
)

diagnostics["Skillnad (pp)"] = (
    diagnostics["Prognos (%)"]
    - diagnostics["Faktiskt utfall (%)"]
)

df_diagnostics_summary = diagnostics[
    [
        "Valuta", "Horisont", "Modell", "År",
        "Observationer",
        "Prognos (%)",
        "Faktiskt utfall (%)",
        "Skillnad (pp)"
    ]
]

for asset in ["ETH", "BTC"]:
    print(f"\n===== {asset} =====")
    display(
        df_diagnostics_summary[
            df_diagnostics_summary["Valuta"] == asset
        ].round(2)
    )



===== ETH =====


,Valuta,Horisont,Modell,År,Observationer,Prognos (%),Faktiskt utfall (%),Skillnad (pp)
12,ETH,180D,Baseline,2025,365,64.67,48.77,15.90
13,ETH,180D,Baseline,2026,124,64.67,57.26,7.41
14,ETH,180D,Logistic Regression,2025,365,64.03,48.77,15.26
15,ETH,180D,Logistic Regression,2026,124,61.02,57.26,3.76
16,ETH,180D,Random Forest,2025,365,55.89,48.77,7.12
17,ETH,180D,Random Forest,2026,124,59.65,57.26,2.39
18,ETH,30D,Baseline,2025,365,54.87,41.37,13.50
19,ETH,30D,Baseline,2026,274,54.87,58.03,-3.16
20,ETH,30D,Logistic Regression,2025,365,53.72,41.37,12.35
21,ETH,30D,Logistic Regression,2026,274,52.22,58.03,-5.81



===== BTC =====


,Valuta,Horisont,Modell,År,Observationer,Prognos (%),Faktiskt utfall (%),Skillnad (pp)
0,BTC,180D,Baseline,2025,365,67.86,35.34,32.52
1,BTC,180D,Baseline,2026,124,67.86,58.06,9.79
2,BTC,180D,Logistic Regression,2025,365,76.70,35.34,41.36
3,BTC,180D,Logistic Regression,2026,124,72.27,58.06,14.20
4,BTC,180D,Random Forest,2025,365,80.36,35.34,45.02
5,BTC,180D,Random Forest,2026,124,63.53,58.06,5.47
6,BTC,30D,Baseline,2025,365,58.92,45.75,13.17
7,BTC,30D,Baseline,2026,274,58.92,58.76,0.17
8,BTC,30D,Logistic Regression,2025,365,60.35,45.75,14.60
9,BTC,30D,Logistic Regression,2026,274,59.72,58.76,0.96


In [30]:

# STEG 11.2 – Modellernas prognoser kontra verkligheten

import plotly.express as px

for asset in ["ETH", "BTC"]:

    subset = df_diagnostics_summary[
        df_diagnostics_summary["Valuta"] == asset
    ].copy()

    plot_data = subset.melt(
        id_vars=[
            "Horisont", "Modell", "År",
            "Observationer"
        ],
        value_vars=[
            "Prognos (%)",
            "Faktiskt utfall (%)"
        ],
        var_name="Typ",
        value_name="Andel (%)"
    )

    plot_data["År"] = plot_data["År"].astype(str)

    fig = px.bar(
        plot_data,
        x="Modell",
        y="Andel (%)",
        color="Typ",
        facet_row="Horisont",
        facet_col="År",
        barmode="group",
        text_auto=".1f",
        hover_data=["Observationer"],
        title=(
            f"{asset} – Prognostiserad kontra "
            "faktisk andel prisuppgångar"
        ),
        template="plotly_dark"
    )

    fig.update_layout(
        height=850,
        yaxis_title="Andel (%)"
    )

    fig.update_yaxes(range=[0, 100])

    fig.show()



### Resultat och slutsats – Steg 11

#### Syfte

Vi undersökte om modellernas uppskattade
sannolikheter för prisuppgång motsvarade
de faktiska historiska utfallen.

#### Resultat – Ethereum

På 30 dagars sikt uppskattade
Logistic Regression följande:

2025:
- Prognostiserad andel: 53,72 %
- Faktisk andel: 41,37 %

2026:
- Prognostiserad andel: 52,22 %
- Faktisk andel: 58,03 %

Modellen överskattade uppgångsfrekvensen
under 2025 men underskattade den under 2026.

#### Resultat – Bitcoin

Random Forest visade stora avvikelser
på 180 dagars sikt.

2025:
- Prognostiserad andel: 80,36 %
- Faktisk andel: 35,34 %
- Skillnad: 45,02 procentenheter

2026:
- Prognostiserad andel: 63,53 %
- Faktisk andel: 58,06 %

#### Tolkning

Modellernas sannolikhetsprognoser
förändrades inte tillräckligt väl
i takt med marknadsförhållandena.

Särskilt Random Forest för Bitcoin
överskattade sannolikheten för uppgång
under 2025.

#### Begränsningar

Analysen visar genomsnittliga skillnader
mellan prognoser och utfall.

Den undersöker ännu inte om prognoser
på exempelvis 60 eller 80 procent
är korrekt kalibrerade.

Utfallsperioderna överlappar även här.

#### Slutsats

Modellerna behöver ytterligare utvärdering
innan deras sannolikheter kan användas
för köp- och säljbedömningar.

Nästa steg är att analysera
sannolikhetskalibrering och därefter
undersöka hur prognoser kan omvandlas
till historiskt utvärderade handelssignaler.


In [31]:

# STEG 12.1 – Sannolikhetskalibrering

import pandas as pd
import numpy as np
from IPython.display import display

calibration_rows = []

for (asset, horizon, model), group in (
    df_final_predictions.groupby(
        ["Valuta", "Horisont", "Modell"]
    )
):
    if model == "Baseline":
        continue

    data = group.copy()

    # Dela sannolikheterna i upp till 5 grupper
    data["Grupp"] = pd.qcut(
        data["Sannolikhet"],
        q=5,
        labels=False,
        duplicates="drop"
    )

    grouped = data.groupby("Grupp")

    for bin_id, subset in grouped:

        calibration_rows.append({
            "Valuta": asset,
            "Horisont": horizon,
            "Modell": model,
            "Grupp": int(bin_id) + 1,
            "Observationer": len(subset),
            "Prognos (%)": (
                subset["Sannolikhet"].mean() * 100
            ),
            "Faktiskt (%)": (
                subset["Faktiskt_Utfall"].mean() * 100
            )
        })

df_calibration = pd.DataFrame(calibration_rows)

df_calibration["Skillnad (pp)"] = (
    df_calibration["Prognos (%)"]
    - df_calibration["Faktiskt (%)"]
)

for asset in ["ETH", "BTC"]:
    print(f"\n===== {asset} – KALIBRERING =====")
    display(
        df_calibration[
            df_calibration["Valuta"] == asset
        ].round(2)
    )



===== ETH – KALIBRERING =====


,Valuta,Horisont,Modell,Grupp,Observationer,Prognos (%),Faktiskt (%),Skillnad (pp)
20,ETH,180D,Logistic Regression,1,98,56.60,86.73,-30.13
21,ETH,180D,Logistic Regression,2,98,60.71,69.39,-8.68
22,ETH,180D,Logistic Regression,3,97,63.46,36.08,27.38
23,ETH,180D,Logistic Regression,4,98,65.91,59.18,6.73
24,ETH,180D,Logistic Regression,5,98,69.66,3.06,66.60
25,ETH,180D,Random Forest,1,98,38.11,71.43,-33.32
26,ETH,180D,Random Forest,2,98,51.52,35.71,15.81
27,ETH,180D,Random Forest,3,97,57.35,34.02,23.33
28,ETH,180D,Random Forest,4,98,62.61,59.18,3.43
29,ETH,180D,Random Forest,5,98,74.61,54.08,20.53



===== BTC – KALIBRERING =====


,Valuta,Horisont,Modell,Grupp,Observationer,Prognos (%),Faktiskt (%),Skillnad (pp)
0,BTC,180D,Logistic Regression,1,98,63.57,69.39,-5.81
1,BTC,180D,Logistic Regression,2,98,71.86,44.90,26.96
2,BTC,180D,Logistic Regression,3,97,76.43,48.45,27.98
3,BTC,180D,Logistic Regression,4,98,81.06,36.73,44.32
4,BTC,180D,Logistic Regression,5,98,84.96,6.12,78.83
5,BTC,180D,Random Forest,1,98,37.26,71.43,-34.17
6,BTC,180D,Random Forest,2,98,63.73,48.98,14.75
7,BTC,180D,Random Forest,3,97,84.23,48.45,35.78
8,BTC,180D,Random Forest,4,98,96.58,34.69,61.89
9,BTC,180D,Random Forest,5,98,98.74,2.04,96.70


In [32]:

# STEG 12.2 – Reliability diagrams

import plotly.graph_objects as go
from plotly.subplots import make_subplots

for asset in ["ETH", "BTC"]:

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=["30 dagar", "180 dagar"]
    )

    for col, horizon in enumerate(
        ["30D", "180D"], start=1
    ):

        subset = df_calibration[
            (df_calibration["Valuta"] == asset)
            & (df_calibration["Horisont"] == horizon)
        ]

        for model in subset["Modell"].unique():

            m = subset[
                subset["Modell"] == model
            ].sort_values("Prognos (%)")

            fig.add_trace(
                go.Scatter(
                    x=m["Prognos (%)"],
                    y=m["Faktiskt (%)"],
                    mode="lines+markers",
                    name=model,
                    legendgroup=model,
                    showlegend=(col == 1),
                    customdata=m[["Observationer"]],
                    hovertemplate=(
                        "Prognos: %{x:.1f}%"
                        "<br>Faktiskt: %{y:.1f}%"
                        "<br>Observationer: %{customdata[0]}"
                        "<extra></extra>"
                    )
                ),
                row=1, col=col
            )

        # Perfekt kalibrering
        fig.add_trace(
            go.Scatter(
                x=[0, 100],
                y=[0, 100],
                mode="lines",
                line=dict(
                    dash="dash",
                    color="white"
                ),
                name="Perfekt kalibrering",
                showlegend=(col == 1)
            ),
            row=1, col=col
        )

        fig.update_xaxes(
            title_text="Prognostiserad sannolikhet (%)",
            range=[0, 100],
            row=1, col=col
        )

        fig.update_yaxes(
            title_text="Faktisk andel uppgångar (%)",
            range=[0, 100],
            row=1, col=col
        )

    fig.update_layout(
        title=f"{asset} – Kalibrering av ML-prognoser",
        template="plotly_dark",
        height=550
    )

    fig.show()



### Resultat och slutsats – Steg 12

#### Syfte

Vi undersökte om modellernas uppskattade
sannolikheter för framtida prisuppgång
överensstämde med faktiska historiska utfall.

Vi analyserade Ethereum och Bitcoin med
prognoshorisonter på 30 och 180 dagar.

#### Resultat – Ethereum

Logistic Regression visade relativt god
kalibrering för de högre sannolikhetsgrupperna
på 30 dagars sikt.

Exempel:

- Prognos 55,84 % – faktiskt 55,47 %
- Prognos 61,20 % – faktiskt 59,38 %

De lägre sannolikhetsgrupperna visade
större avvikelser.

På 180 dagars sikt var kalibreringen
betydligt mer instabil.

#### Resultat – Bitcoin

Random Forest visade stora kalibreringsfel,
särskilt på 180 dagars sikt.

I gruppen med högst uppskattad sannolikhet:

- Prognos: 98,74 %
- Faktiskt positiva utfall: 2,04 %

Modellen överskattade därmed kraftigt
sannolikheten för prisuppgång.

#### Tolkning

Resultaten visar att en hög uppskattad
sannolikhet inte automatiskt innebär
en hög faktisk sannolikhet för uppgång.

Logistic Regression för Ethereum på
30 dagar är den mest intressanta kandidaten
för fortsatt undersökning.

#### Begränsningar

- Utfallen överlappar i tiden.
- Kalibreringen bygger på historiska data.
- Resultaten påverkas av marknadsperioden.
- Sannolikhetsgrupperna innehåller
  beroende observationer.
- En bra sannolikhetsprognos garanterar
  inte ekonomiskt lönsamma affärer.

#### Slutsats

Vi har identifierat betydande skillnader
i modellernas tillförlitlighet.

Ethereum på 30 dagars sikt visar
viss prognosförmåga som motiverar
ytterligare analys.

Bitcoin-modellerna behöver förbättras
och valideras på nya data innan deras
prognoser används för handelsbeslut.

Nästa steg är att undersöka sambandet
mellan modellernas sannolikheter och
den faktiska framtida avkastningen.


In [33]:

# STEG 13.1 – Modellprognoser och framtida avkastning

import pandas as pd
import numpy as np
from IPython.display import display

# Tidigare skapade modellprognoser
pred = df_final_predictions[
    (df_final_predictions["Valuta"] == "ETH")
    & (df_final_predictions["Horisont"] == "30D")
    & (df_final_predictions["Modell"] == "Logistic Regression")
].copy()

pred["time"] = pd.to_datetime(
    pred["time"], utc=True
)

# Framtida avkastning från steg 5
actual = df_eth_ml[
    ["time", "Return_30D"]
].copy()

actual["time"] = pd.to_datetime(
    actual["time"], utc=True
)

# Slå ihop på datum
analysis = pred.merge(
    actual,
    on="time",
    how="inner",
    validate="one_to_one"
)

# Förutbestämda explorativa sannolikhetsgrupper
analysis["Prognosgrupp"] = pd.cut(
    analysis["Sannolikhet"],
    bins=[0, 0.4, 0.5, 0.6, 1.0],
    labels=[
        "0–40 %",
        "40–50 %",
        "50–60 %",
        "60–100 %"
    ],
    include_lowest=True
)

summary = (
    analysis.groupby("Prognosgrupp", observed=True)
    .agg(
        Observationer=("Return_30D", "size"),
        Medianavkastning=("Return_30D", "median"),
        Medelavkastning=("Return_30D", "mean"),
        Positiva_Andel=(
            "Faktiskt_Utfall", "mean"
        )
    )
    .reset_index()
)

summary["Positiva utfall (%)"] = (
    summary["Positiva_Andel"] * 100
)

summary = summary.drop(
    columns=["Positiva_Andel"]
)

print("ETH – PROGNOSSANNOLIKHET OCH AVKASTNING")
display(summary.round(2))


ETH – PROGNOSSANNOLIKHET OCH AVKASTNING


,Prognosgrupp,Observationer,Medianavkastning,Medelavkastning,Positiva utfall (%)
0,0–40 %,5,5.44,6.29,80.00
1,40–50 %,180,-5.19,-3.68,37.78
2,50–60 %,386,-0.39,1.14,48.45
3,60–100 %,68,9.23,11.64,75.00


In [34]:

# STEG 13.2 – Avkastning per prognosgrupp

import plotly.express as px

fig = px.bar(
    summary,
    x="Prognosgrupp",
    y="Medianavkastning",
    text_auto=".2f",
    hover_data=[
        "Observationer",
        "Positiva utfall (%)"
    ],
    title="ETH – Medianavkastning efter 30 dagar per prognosgrupp",
    template="plotly_dark",
    labels={
        "Medianavkastning": "Medianavkastning (%)",
        "Prognosgrupp": "Prognostiserad sannolikhet"
    }
)

fig.add_hline(
    y=0,
    line_dash="dash",
    line_color="white"
)

fig.update_layout(height=550)
fig.show()


fig2 = px.bar(
    summary,
    x="Prognosgrupp",
    y="Positiva utfall (%)",
    text_auto=".1f",
    hover_data=["Observationer"],
    title="ETH – Andel positiva utfall efter 30 dagar",
    template="plotly_dark"
)

fig2.add_hline(
    y=50,
    line_dash="dash",
    line_color="white"
)

fig2.update_layout(
    height=550,
    yaxis_range=[0, 100]
)

fig2.show()



### Resultat och slutsats – Steg 13

#### Syfte

Vi undersökte om Logistic Regressions
uppskattade sannolikhet för prisuppgång
hade samband med Ethereums faktiska
avkastning efter 30 dagar.

#### Resultat

Prognosgruppen 60–100 % visade:

- 68 historiska observationer
- Medianavkastning: +9,23 %
- Medelavkastning: +11,64 %
- Positiva utfall: 75,00 %

Prognosgruppen 40–50 % visade:

- 180 observationer
- Medianavkastning: -5,19 %
- Positiva utfall: 37,78 %

#### Tolkning

Resultaten visar att högre prognostiserad
sannolikhet i vissa grupper sammanföll
med högre framtida avkastning.

Särskilt gruppen över 60 % visade
historiskt gynnsamma resultat.

Detta indikerar att modellen kan
innehålla användbar information
för identifiering av potentiella köpperioder.

#### Begränsningar

- Prognosperioderna överlappar.
- Resultaten bygger på historiska data.
- Gruppen under 40 % har endast fem observationer.
- Sannolikhetsgränserna har undersökts
  med data från testperioden.
- Ingen handelsstrategi har ännu testats
  med dessa sannolikhetsgränser.
- Transaktionskostnader och risk behöver
  inkluderas i framtida utvärdering.

#### Slutsats

Logistic Regression för Ethereum på
30 dagar visar ett intressant samband
mellan hög prognossannolikhet och
positiv framtida avkastning.

Resultaten motiverar fortsatt analys
av historiska köpperioder.

Vi behöver nu undersöka om observationerna
representerar flera oberoende marknadsperioder
eller främst samma sammanhängande prisrörelser.

Därefter kan vi utveckla och utvärdera
en preliminär handelsstrategi.


In [35]:

# STEG 14.1 – Identifiera historiska signalperioder

import pandas as pd
import numpy as np
from IPython.display import display

signals = analysis.copy()

signals["time"] = pd.to_datetime(
    signals["time"], utc=True
)

signals = signals.sort_values("time").reset_index(drop=True)

# Explorativ gräns, inte en validerad handelsregel
signals["Hög_Sannolikhet"] = (
    signals["Sannolikhet"] >= 0.60
)

# Ny period vid nytt tillfälle eller tidsavbrott
new_period = (
    signals["Hög_Sannolikhet"].ne(
        signals["Hög_Sannolikhet"].shift()
    )
    | signals["time"].diff().gt(
        pd.Timedelta(days=1)
    )
)

signals["Period_ID"] = new_period.cumsum()

selected = signals[
    signals["Hög_Sannolikhet"]
].copy()

period_summary = (
    selected.groupby("Period_ID")
    .agg(
        Start=("time", "min"),
        Slut=("time", "max"),
        Antal_dagar=("time", "size"),
        Genomsnittlig_prognos=("Sannolikhet", "mean"),
        Medianavkastning_30D=("Return_30D", "median"),
        Positiva_30D=("Faktiskt_Utfall", "mean")
    )
    .reset_index(drop=True)
)

period_summary["Genomsnittlig_prognos"] *= 100
period_summary["Positiva_30D"] *= 100

print("ANTAL HISTORISKA SIGNALPERIODER:")
print(len(period_summary))

print("\nHISTORISKA PERIODER MED PROGNOS ≥ 60 %")

display(period_summary.round(2))


ANTAL HISTORISKA SIGNALPERIODER:
16

HISTORISKA PERIODER MED PROGNOS ≥ 60 %


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\4016767678.py:57: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(period_summary.round(2))


,Start,Slut,Antal_dagar,Genomsnittlig_prognos,Medianavkastning_30D,Positiva_30D
0,2025-04-23 00:00:00+00:00,2025-04-23 00:00:00+00:00,1,60.12,42.96,100.00
1,2025-05-08 00:00:00+00:00,2025-05-14 00:00:00+00:00,7,63.49,3.56,57.14
2,2025-05-22 00:00:00+00:00,2025-05-22 00:00:00+00:00,1,62.17,-11.68,0.00
3,2025-05-26 00:00:00+00:00,2025-05-28 00:00:00+00:00,3,61.00,-9.25,0.00
4,2025-06-09 00:00:00+00:00,2025-06-12 00:00:00+00:00,4,62.20,8.73,100.00
5,2025-07-02 00:00:00+00:00,2025-07-03 00:00:00+00:00,2,60.75,35.31,100.00
6,2025-07-09 00:00:00+00:00,2025-07-27 00:00:00+00:00,19,65.68,28.60,100.00
7,2025-07-29 00:00:00+00:00,2025-07-30 00:00:00+00:00,2,60.08,16.50,100.00
8,2025-08-09 00:00:00+00:00,2025-08-10 00:00:00+00:00,2,61.94,2.15,100.00
9,2025-08-12 00:00:00+00:00,2025-08-17 00:00:00+00:00,6,61.66,2.30,83.33


In [36]:

# STEG 14.2 – Visualisera prognoser och köpperioder

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=signals["time"],
    y=signals["Sannolikhet"] * 100,
    mode="lines",
    name="ML-prognos",
    line=dict(color="#5DADE2", width=2)
))

fig.add_trace(go.Scatter(
    x=selected["time"],
    y=selected["Sannolikhet"] * 100,
    mode="markers",
    name="Potentiell köpperiod",
    marker=dict(
        color="#2ECC71",
        size=7
    )
))

fig.add_hline(
    y=60,
    line_dash="dash",
    line_color="orange",
    annotation_text="Explorativ gräns 60 %"
)

fig.update_layout(
    title="ETH – Historiska ML-signaler över 60 %",
    xaxis_title="Datum",
    yaxis_title="Prognostiserad sannolikhet (%)",
    yaxis_range=[0, 100],
    template="plotly_dark",
    height=550,
    hovermode="x unified"
)

fig.update_xaxes(rangeslider_visible=True)

fig.show()



### Resultat och slutsats – Steg 14

#### Syfte

Vi undersökte när Logistic Regression
gav minst 60 % sannolikhet för positiv
Ethereum-avkastning efter 30 dagar.

#### Resultat

Under testperioden 2025–2026 identifierades:

- 68 dagar med prognos på minst 60 %
- 16 sammanhängande signalperioder
- Längsta perioden: 19 dagar

Signalerna var särskilt koncentrerade
till delar av 2025.

Vissa signalperioder följdes av positiv
avkastning, medan andra följdes av förlust.

#### Tolkning

Resultaten visar att modellens höga
sannolikhetsprognoser inte är jämnt
fördelade över tiden.

Flera signalperioder ligger nära varandra,
vilket innebär att framtida avkastning
kan överlappa mellan observationerna.

#### Begränsningar

- Gränsen 60 % har undersökts på testdata.
- Signalerna är inte oberoende affärer.
- Resultaten påverkas av marknadstrender.
- Vi har ännu inte simulerat faktisk handel.

#### Slutsats

Vi har identifierat historiska perioder
där ML-modellen signalerade hög
sannolikhet för prisuppgång.

Nästa steg är att simulera separata
affärer med en bestämd innehavstid.

Vi ska undersöka faktisk avkastning,
antal lönsamma affärer och risk,
utan att återinvestera kapital i
överlappande positioner.


In [37]:

# STEG 15.1 – Simulera köp utan överlappning

import pandas as pd
import numpy as np
from IPython.display import display

HOLD_DAYS = 30
FEE = 0.001
THRESHOLD = 0.60

prices = df_eth_ml[["time", "ETH_Price"]].copy()
prices["time"] = pd.to_datetime(prices["time"], utc=True)

prices = (
    prices.drop_duplicates("time")
    .set_index("time")
    .sort_index()
)

candidate_signals = signals[
    signals["Sannolikhet"] >= THRESHOLD
].copy().sort_values("time")

trades = []
last_exit = pd.Timestamp.min.tz_localize("UTC")

for _, row in candidate_signals.iterrows():

    signal_date = row["time"]

    # Ingen ny affär förrän föregående stängts
    if signal_date < last_exit:
        continue

    entry_date = signal_date + pd.Timedelta(days=1)
    exit_date = entry_date + pd.Timedelta(days=HOLD_DAYS)

    if (
        entry_date not in prices.index
        or exit_date not in prices.index
    ):
        continue

    entry_price = prices.loc[entry_date, "ETH_Price"]
    exit_price = prices.loc[exit_date, "ETH_Price"]

    if (
        pd.isna(entry_price)
        or pd.isna(exit_price)
        or entry_price <= 0
    ):
        continue

    gross_return = exit_price / entry_price - 1

    # Nettoutfall efter köp- och säljavgift
    net_return = (
        (1 - FEE)
        * (exit_price / entry_price)
        * (1 - FEE)
        - 1
    )

    trades.append({
        "Signaldatum": signal_date,
        "Köpdatum": entry_date,
        "Säljdatum": exit_date,
        "Prognos (%)": row["Sannolikhet"] * 100,
        "Köppris USD": entry_price,
        "Säljpris USD": exit_price,
        "Bruttoavkastning (%)": gross_return * 100,
        "Nettoavkastning (%)": net_return * 100
    })

    last_exit = exit_date

df_trades = pd.DataFrame(trades)

if df_trades.empty:
    print("Inga giltiga affärer kunde simuleras.")
else:
    print("ANTAL SEPARATA AFFÄRER:", len(df_trades))

    print(
        "Andel lönsamma affärer:",
        round(
            (df_trades["Nettoavkastning (%)"] > 0).mean()
            * 100, 2
        ),
        "%"
    )

    print(
        "Medianavkastning:",
        round(
            df_trades["Nettoavkastning (%)"].median(), 2
        ),
        "%"
    )

    print("\nAFFÄRER")
    display(df_trades.round(2))


ANTAL SEPARATA AFFÄRER: 8
Andel lönsamma affärer: 62.5 %
Medianavkastning: 1.08 %

AFFÄRER


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\182554642.py:102: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  display(df_trades.round(2))


,Signaldatum,Köpdatum,Säljdatum,Prognos (%),Köppris USD,Säljpris USD,Bruttoavkastning (%),Nettoavkastning (%)
0,2025-04-23 00:00:00+00:00,2025-04-24 00:00:00+00:00,2025-05-24 00:00:00+00:00,60.12,1769.48,2529.65,42.96,42.67
1,2025-05-26 00:00:00+00:00,2025-05-27 00:00:00+00:00,2025-06-26 00:00:00+00:00,60.17,2661.42,2415.25,-9.25,-9.43
2,2025-07-02 00:00:00+00:00,2025-07-03 00:00:00+00:00,2025-08-02 00:00:00+00:00,60.71,2590.32,3393.52,31.01,30.75
3,2025-08-09 00:00:00+00:00,2025-08-10 00:00:00+00:00,2025-09-09 00:00:00+00:00,62.00,4248.71,4309.73,1.44,1.23
4,2025-09-12 00:00:00+00:00,2025-09-13 00:00:00+00:00,2025-10-13 00:00:00+00:00,62.01,4666.35,4249.55,-8.93,-9.11
5,2026-03-16 00:00:00+00:00,2026-03-17 00:00:00+00:00,2026-04-16 00:00:00+00:00,60.55,2323.32,2349.63,1.13,0.93
6,2026-08-19 00:00:00+00:00,2026-08-20 00:00:00+00:00,2026-09-19 00:00:00+00:00,66.06,2326.67,2632.92,13.16,12.94
7,2026-09-20 00:00:00+00:00,2026-09-21 00:00:00+00:00,2026-10-21 00:00:00+00:00,60.27,2773.80,2473.58,-10.82,-11.00


In [38]:

# STEG 15.2 – Visualisera avkastning per affär

import plotly.express as px

if not df_trades.empty:

    plot_data = df_trades.copy()

    plot_data["Affär"] = [
        f"Affär {i+1}"
        for i in range(len(plot_data))
    ]

    plot_data["Resultat"] = np.where(
        plot_data["Nettoavkastning (%)"] >= 0,
        "Vinst",
        "Förlust"
    )

    fig = px.bar(
        plot_data,
        x="Affär",
        y="Nettoavkastning (%)",
        color="Resultat",
        color_discrete_map={
            "Vinst": "#2ECC71",
            "Förlust": "#FF5252"
        },
        text_auto=".1f",
        hover_data=[
            "Köpdatum",
            "Säljdatum",
            "Prognos (%)"
        ],
        title="Ethereum – Avkastning per simulerad ML-affär",
        template="plotly_dark"
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="white"
    )

    fig.update_layout(
        height=550,
        yaxis_title="Nettoavkastning (%)"
    )

    fig.show()



### Resultat och slutsats – Steg 15

#### Syfte

Vi undersökte om Logistic Regression
kunde identifiera historiskt gynnsamma
köptillfällen för Ethereum.

#### Metod

Vi simulerade köp när modellen uppskattade
minst 60 % sannolikhet för prisuppgång.

Handelsregler:

- Startsignal: ML-prognos minst 60 %
- Köp: nästa dags stängningspris
- Innehavstid: 30 dagar
- Transaktionskostnad: 0,1 % vid köp och sälj
- Ingen överlappning mellan affärerna

#### Resultat

Totalt genomfördes 8 simulerade affärer.

- Lönsamma affärer: 5 av 8
- Andel lönsamma affärer: 62,5 %
- Medianavkastning: +1,08 %

Bästa affären gav +42,67 %.

Sämsta affären gav -11,00 %.

#### Tolkning

Strategin identifierade flera historiska
perioder med positiv efterföljande avkastning.

Resultaten visar samtidigt att hög
prognossannolikhet inte garanterar vinst.

Ett fåtal mycket lönsamma affärer
påverkade resultatet betydligt.

#### Begränsningar

Endast åtta separata affärer genomfördes.

Sannolikhetsgränsen 60 % undersöktes
efter att testresultaten blivit kända.

Resultaten kan därför vara överanpassade
till den undersökta marknadsperioden.

Vi har ännu inte jämfört en fullständig
kapitalutveckling med Buy & Hold under
exakt samma period.

#### Preliminär slutsats

Modellen har identifierat vissa historiskt
gynnsamma köptillfällen för Ethereum.

Det begränsade antalet affärer innebär
att resultatet inte räcker för att
fastställa framtida lönsamhet.

Nästa steg är att jämföra kapitalutveckling,
risk och avkastning mot Buy & Hold.



## Steg 16 – Kapitalutveckling: ML mot Buy & Hold

### Syfte

Vi jämför vår ML-baserade handelsstrategi
med en passiv Buy & Hold-strategi.

### Metod

Båda strategierna får:

- Startkapital: 10 000 USD
- Samma startdatum och slutdatum
- Transaktionsavgift: 0,1 % per köp och sälj
- Ingen hävstång

ML-strategin köper endast vid de åtta
identifierade affärerna och håller
kontanter mellan positionerna.

Buy & Hold äger Ethereum under hela perioden.

### Utvärderingsmått

1. Slutkapital
2. Total avkastning
3. Maximal drawdown
4. Andel dagar investerad
5. Antal genomförda affärer

Detta är ett explorativt historiskt backtest.
Resultaten utgör inte ett oberoende test
av 60-procentsgränsen.


In [39]:

# STEG 16.2 – ML-strategi mot Buy & Hold

import pandas as pd
import numpy as np
from IPython.display import display

START_CAPITAL = 10000
FEE = 0.001

# Tidigare simulerade affärer
trades_bt = df_trades.copy().sort_values("Köpdatum")

# Historiska Ethereum-priser
price_data = df_eth_ml[["time", "ETH_Price"]].copy()
price_data["time"] = pd.to_datetime(
    price_data["time"], utc=True
)

price_data = (
    price_data.drop_duplicates("time")
    .set_index("time")
    .sort_index()
)

start_date = trades_bt["Köpdatum"].min()
end_date = trades_bt["Säljdatum"].max()

price_data = price_data.loc[
    start_date:end_date
].copy()

if price_data["ETH_Price"].isna().any():
    raise ValueError("Saknade ETH-priser i testperioden")

buy_dates = set(trades_bt["Köpdatum"])
sell_dates = set(trades_bt["Säljdatum"])

# ML-strategi
cash = START_CAPITAL
units = 0.0

# Buy & Hold
bh_units = 0.0
bh_cash = START_CAPITAL

records = []

for date, row in price_data.iterrows():

    price = float(row["ETH_Price"])

    # Avsluta position först
    if date in sell_dates and units > 0:
        cash = units * price * (1 - FEE)
        units = 0.0

    # Öppna ny position
    if date in buy_dates and units == 0:
        units = cash * (1 - FEE) / price
        cash = 0.0

    # Buy & Hold köper på första dagen
    if date == start_date:
        bh_units = bh_cash * (1 - FEE) / price
        bh_cash = 0.0

    # Buy & Hold säljer på sista dagen
    if date == end_date:
        bh_cash = bh_units * price * (1 - FEE)
        bh_units = 0.0

    strategy_value = cash + units * price
    buyhold_value = bh_cash + bh_units * price

    records.append({
        "time": date,
        "ML_Strategy": strategy_value,
        "Buy_Hold": buyhold_value,
        "Invested": int(units > 0)
    })

df_equity = pd.DataFrame(records)

# Maximal drawdown
for col in ["ML_Strategy", "Buy_Hold"]:

    df_equity[f"{col}_DD"] = (
        df_equity[col] /
        df_equity[col].cummax() - 1
    ) * 100

summary = []

for col, name in [
    ("ML_Strategy", "ML-strategi"),
    ("Buy_Hold", "Buy & Hold")
]:

    final_capital = df_equity[col].iloc[-1]

    summary.append({
        "Strategi": name,
        "Startkapital": START_CAPITAL,
        "Slutkapital": final_capital,
        "Total avkastning (%)": (
            final_capital / START_CAPITAL - 1
        ) * 100,
        "Max Drawdown (%)": (
            df_equity[f"{col}_DD"].min()
        ),
        "Investerad tid (%)": (
            df_equity["Invested"].mean() * 100
            if col == "ML_Strategy" else 100.0
        ),
        "Antal affärer": (
            len(trades_bt)
            if col == "ML_Strategy" else 1
        )
    })

df_equity_summary = pd.DataFrame(summary)

print("STEG 16 – ML-STRATEGI MOT BUY & HOLD")
print("Period:", start_date.date(), "till", end_date.date())

display(df_equity_summary.round(2))


STEG 16 – ML-STRATEGI MOT BUY & HOLD
Period: 2025-04-24 till 2026-10-21


,Strategi,Startkapital,Slutkapital,Total avkastning (%),Max Drawdown (%),Investerad tid (%),Antal affärer
0,ML-strategi,10000,15769.23,57.69,-31.17,43.96,8
1,Buy & Hold,10000,13951.22,39.51,-67.62,100.00,1


In [40]:

# STEG 16.3 – Kapitalutveckling och drawdown

import plotly.graph_objects as go

# Diagram 1 – Kapitalutveckling
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_equity["time"],
    y=df_equity["ML_Strategy"],
    name="ML-strategi",
    line=dict(color="#2ECC71", width=2.5)
))

fig.add_trace(go.Scatter(
    x=df_equity["time"],
    y=df_equity["Buy_Hold"],
    name="Buy & Hold",
    line=dict(color="#5DADE2", width=2.5)
))

fig.add_hline(
    y=START_CAPITAL,
    line_dash="dash",
    line_color="gray"
)

fig.update_layout(
    title="Ethereum – Kapitalutveckling: ML vs Buy & Hold",
    xaxis_title="Datum",
    yaxis_title="Portföljvärde (USD)",
    template="plotly_dark",
    height=600,
    hovermode="x unified"
)

fig.update_xaxes(rangeslider_visible=True)
fig.show()


# Diagram 2 – Drawdown
fig2 = go.Figure()

fig2.add_trace(go.Scatter(
    x=df_equity["time"],
    y=df_equity["ML_Strategy_DD"],
    name="ML-strategi",
    line=dict(color="#2ECC71", width=2)
))

fig2.add_trace(go.Scatter(
    x=df_equity["time"],
    y=df_equity["Buy_Hold_DD"],
    name="Buy & Hold",
    line=dict(color="#FF5252", width=2)
))

fig2.update_layout(
    title="Ethereum – Historisk drawdown",
    xaxis_title="Datum",
    yaxis_title="Drawdown (%)",
    template="plotly_dark",
    height=550,
    hovermode="x unified"
)

fig2.update_xaxes(rangeslider_visible=True)
fig2.show()



### Resultat och slutsats – Steg 16

#### Syfte

Vi undersökte om en handelsstrategi
baserad på Logistic Regression kunde
överträffa Buy & Hold för Ethereum.

#### Metod

Vi simulerade handel under perioden
24 april 2025 till 21 oktober 2026.

Båda strategierna hade:

- Startkapital: 10 000 USD
- Samma jämförelseperiod
- Transaktionsavgift: 0,1 % per affärssida
- Ingen hävstång

ML-strategin köpte vid en uppskattad
sannolikhet på minst 60 % och sålde
efter 30 dagar.

#### Resultat

**ML-strategi**

- Slutkapital: 15 769,23 USD
- Total avkastning: +57,69 %
- Max drawdown: -31,17 %
- Investerad tid: 43,96 %
- Antal affärer: 8

**Buy & Hold**

- Slutkapital: 13 951,22 USD
- Total avkastning: +39,51 %
- Max drawdown: -67,62 %
- Investerad tid: 100 %

#### Tolkning

ML-strategin gav 18,18 procentenheter
högre totalavkastning än Buy & Hold
under den simulerade perioden.

Strategin uppvisade också lägre
maximal drawdown.

En möjlig förklaring är att modellen
periodvis höll kapitalet utanför marknaden
och därmed undvek vissa prisnedgångar.

#### Begränsningar

- Endast åtta affärer genomfördes.
- Sannolikhetsgränsen valdes efter
  tidigare analys av testperioden.
- Strategin är inte utvärderad på
  helt oberoende framtida data.
- Slippage och skatter ingår inte.
- Exekvering och tidsförskjutning
  behöver kontrolleras ytterligare.

#### Slutsats

Den simulerade ML-strategin överträffade
Buy & Hold under den undersökta perioden.

Resultaten visar att modellen kan ha
identifierat historiskt gynnsamma
handelsperioder.

Det är däremot ännu inte fastställt
att strategin har en stabil fördel
på framtida marknadsdata.

Nästa steg är att kontrollera backtestets
tidsordning och jämföra strategin
mot ytterligare referensmetoder.


In [41]:

# STEG 17.1 – Kvalitetskontroll av ML-backtest

import pandas as pd
import numpy as np
from IPython.display import display

audit = df_trades.copy().sort_values("Köpdatum")

for col in ["Signaldatum", "Köpdatum", "Säljdatum"]:
    audit[col] = pd.to_datetime(audit[col], utc=True)

# Kontrollera tidsordning
audit["Signal_före_köp"] = (
    audit["Signaldatum"] < audit["Köpdatum"]
)

audit["Korrekt_innehavstid"] = (
    (audit["Säljdatum"] - audit["Köpdatum"]).dt.days
    == 30
)

audit["Ingen_överlappning"] = (
    audit["Köpdatum"] >= audit["Säljdatum"].shift(1)
)

audit.loc[
    audit.index[0], "Ingen_överlappning"
] = True

# Beräkna slutkapital direkt från affärernas nettoavkastning
capital_from_trades = (
    10000 *
    np.prod(1 + audit["Nettoavkastning (%)"] / 100)
)

capital_from_equity = df_equity["ML_Strategy"].iloc[-1]

difference = abs(
    capital_from_trades - capital_from_equity
)

checks = pd.DataFrame({
    "Kontroll": [
        "Signal före köp",
        "30 dagars innehav",
        "Inga överlappande positioner",
        "Slutkapital överensstämmer"
    ],
    "Godkänd": [
        bool(audit["Signal_före_köp"].all()),
        bool(audit["Korrekt_innehavstid"].all()),
        bool(audit["Ingen_överlappning"].all()),
        bool(difference < 0.01)
    ]
})

print("STEG 17 – KVALITETSKONTROLL")

display(checks)

print("\nSlutkapital från affärerna:",
      round(capital_from_trades, 2))

print("Slutkapital från portföljkurvan:",
      round(capital_from_equity, 2))

print("Skillnad:", round(difference, 6))

if checks["Godkänd"].all():
    print("\nAlla automatiska kontroller godkända.")
else:
    print("\nNågon kontroll behöver undersökas.")


STEG 17 – KVALITETSKONTROLL


,Kontroll,Godkänd
0,Signal före köp,True
1,30 dagars innehav,True
2,Inga överlappande positioner,True
3,Slutkapital överensstämmer,True



Slutkapital från affärerna: 15769.23
Slutkapital från portföljkurvan: 15769.23
Skillnad: 0.0

Alla automatiska kontroller godkända.



## Steg 18 – Kontroll av dataläckage

### Syfte

Vi undersöker om de tekniska indikatorerna
har beräknats med information som var
tillgänglig vid respektive historiskt datum.

### Kontrollområden

- SMA 200
- Historisk avkastning under 30 dagar
- Bull Market Support Band
- Datum och tidsordning

### Varför är detta viktigt?

Om framtida priser används för att beräkna
en historisk indikator kan modellen få
orealistiskt bra resultat.

Vi behöver därför kontrollera indikatorernas
beräkningar innan vi kan dra slutsatser
om strategins verkliga prognosförmåga.


In [42]:

# STEG 18.1 – Kontrollera historiska indikatorer

import pandas as pd
import numpy as np
from IPython.display import display

audit_results = []

for asset, source in [
    ("ETH", df_eth_ml),
    ("BTC", df_btc_ml)
]:

    df = source.copy().sort_values("time")
    price_col = f"{asset}_Price"

    price = pd.to_numeric(
        df[price_col], errors="coerce"
    )

    # Oberoende beräkning av SMA 200
    sma_check = price.rolling(
        window=200,
        min_periods=200
    ).mean()

    # Kontrollera tidigare sparad SMA 200
    sma_diff = (
        df["SMA_200"] - sma_check
    ).abs()

    valid_sma = (
        df["SMA_200"].notna()
        & sma_check.notna()
    )

    # Historisk 30-dagarsavkastning
    return_check = (
        price / price.shift(30) - 1
    ) * 100

    return_diff = (
        df["Return_30D_History"] - return_check
    ).abs()

    audit_results.append({
        "Valuta": asset,
        "Kontroll": "SMA 200",
        "Jämförda rader": int(valid_sma.sum()),
        "Max avvikelse": (
            float(sma_diff[valid_sma].max())
            if valid_sma.any() else np.nan
        )
    })

    audit_results.append({
        "Valuta": asset,
        "Kontroll": "Historisk avkastning 30D",
        "Jämförda rader": int(return_diff.notna().sum()),
        "Max avvikelse": (
            float(return_diff.max())
            if return_diff.notna().any() else np.nan
        )
    })

df_leakage_audit = pd.DataFrame(audit_results)

df_leakage_audit["Numeriskt godkänd"] = (
    df_leakage_audit["Jämförda rader"].gt(0)
    & df_leakage_audit["Max avvikelse"].le(1e-6)
)

print("STEG 18 – KONTROLL AV INDIKATORBERÄKNINGAR")

display(df_leakage_audit.round(8))

print("\nBMSB-kolumner – exempel från ETH:")
display(
    df_eth_ml[
        ["time", "ETH_Price",
         "BMSB_Upper", "BMSB_Lower"]
    ].tail(10)
)


STEG 18 – KONTROLL AV INDIKATORBERÄKNINGAR


,Valuta,Kontroll,Jämförda rader,Max avvikelse,Numeriskt godkänd
0,ETH,SMA 200,3905,0.0,True
1,ETH,Historisk avkastning 30D,4074,0.0,True
2,BTC,SMA 200,4124,0.0,True
3,BTC,Historisk avkastning 30D,4293,0.0,True



BMSB-kolumner – exempel från ETH:


,time,ETH_Price,BMSB_Upper,BMSB_Lower
4094,2026-10-23 00:00:00+00:00,2378.260399,2318.522944,2131.194108
4095,2026-10-24 00:00:00+00:00,2351.985153,2318.522944,2131.194108
4096,2026-10-25 00:00:00+00:00,2325.796597,2318.522944,2131.194108
4097,2026-10-26 00:00:00+00:00,2316.598625,2319.184185,2163.381441
4098,2026-10-27 00:00:00+00:00,2347.562225,2319.184185,2163.381441
4099,2026-10-28 00:00:00+00:00,2356.988509,2319.184185,2163.381441
4100,2026-10-29 00:00:00+00:00,2256.031994,2319.184185,2163.381441
4101,2026-10-30 00:00:00+00:00,2240.229980,2319.184185,2163.381441
4102,2026-10-31 00:00:00+00:00,2252.073358,2319.184185,2163.381441
4103,2026-11-01 00:00:00+00:00,2269.988406,2319.184185,2163.381441


In [43]:

# STEG 18.2 – Kontrollera BMSB mot avslutade veckor

import pandas as pd
import numpy as np
from IPython.display import display

bmsb_results = []

for asset, source in [
    ("ETH", df_eth_ml),
    ("BTC", df_btc_ml)
]:

    df = source.copy()
    df["time"] = pd.to_datetime(df["time"], utc=True)

    df = (
        df.sort_values("time")
        .drop_duplicates("time")
        .set_index("time")
    )

    price = pd.to_numeric(
        df[f"{asset}_Price"], errors="coerce"
    )

    # Veckan avslutas på söndag
    weekly_close = price.resample("W-SUN").last()

    # 20 veckors SMA
    weekly_sma = weekly_close.rolling(
        20, min_periods=20
    ).mean()

    # 21 veckors EMA
    weekly_ema = weekly_close.ewm(
        span=21,
        adjust=False,
        min_periods=21
    ).mean()

    # Först nästa måndag blir värdet tillgängligt
    weekly_band = pd.DataFrame({
        "SMA20": weekly_sma,
        "EMA21": weekly_ema
    })

    weekly_band.index += pd.Timedelta(days=1)

    # Lägg in veckovärden på dagliga datum
    aligned = weekly_band.reindex(
        df.index, method="ffill"
    )

    calculated_upper = aligned[
        ["SMA20", "EMA21"]
    ].max(axis=1)

    calculated_lower = aligned[
        ["SMA20", "EMA21"]
    ].min(axis=1)

    valid = (
        df["BMSB_Upper"].notna()
        & df["BMSB_Lower"].notna()
        & aligned["SMA20"].notna()
        & aligned["EMA21"].notna()
    )

    upper_error = (
        df.loc[valid, "BMSB_Upper"]
        - calculated_upper.loc[valid]
    ).abs()

    lower_error = (
        df.loc[valid, "BMSB_Lower"]
        - calculated_lower.loc[valid]
    ).abs()

    max_error = max(
        upper_error.max(),
        lower_error.max()
    ) if valid.any() else np.nan

    bmsb_results.append({
        "Valuta": asset,
        "Jämförda dagar": int(valid.sum()),
        "Max avvikelse": max_error,
        "Matchar referens": (
            bool(max_error < 0.01)
            if pd.notna(max_error) else False
        )
    })

    print(f"\n===== {asset} – SENASTE BMSB =====")

    comparison = pd.DataFrame({
        "Befintlig övre": df["BMSB_Upper"],
        "Beräknad övre": calculated_upper,
        "Befintlig nedre": df["BMSB_Lower"],
        "Beräknad nedre": calculated_lower
    })

    display(comparison.tail(10).round(2))

print("\nBMSB – SAMMANFATTNING")
display(pd.DataFrame(bmsb_results).round(6))



===== ETH – SENASTE BMSB =====


,Befintlig övre,Beräknad övre,Befintlig nedre,Beräknad nedre
time,,,,
2026-10-23 00:00:00+00:00,2318.52,2318.52,2131.19,2131.19
2026-10-24 00:00:00+00:00,2318.52,2318.52,2131.19,2131.19
2026-10-25 00:00:00+00:00,2318.52,2318.52,2131.19,2131.19
2026-10-26 00:00:00+00:00,2319.18,2319.18,2163.38,2163.38
2026-10-27 00:00:00+00:00,2319.18,2319.18,2163.38,2163.38
2026-10-28 00:00:00+00:00,2319.18,2319.18,2163.38,2163.38
2026-10-29 00:00:00+00:00,2319.18,2319.18,2163.38,2163.38
2026-10-30 00:00:00+00:00,2319.18,2319.18,2163.38,2163.38
2026-10-31 00:00:00+00:00,2319.18,2319.18,2163.38,2163.38



===== BTC – SENASTE BMSB =====


,Befintlig övre,Beräknad övre,Befintlig nedre,Beräknad nedre
time,,,,
2026-10-23 00:00:00+00:00,76192.48,76192.48,71509.00,71509.00
2026-10-24 00:00:00+00:00,76192.48,76192.48,71509.00,71509.00
2026-10-25 00:00:00+00:00,76192.48,76192.48,71509.00,71509.00
2026-10-26 00:00:00+00:00,76542.16,76542.16,72353.03,72353.03
2026-10-27 00:00:00+00:00,76542.16,76542.16,72353.03,72353.03
2026-10-28 00:00:00+00:00,76542.16,76542.16,72353.03,72353.03
2026-10-29 00:00:00+00:00,76542.16,76542.16,72353.03,72353.03
2026-10-30 00:00:00+00:00,76542.16,76542.16,72353.03,72353.03
2026-10-31 00:00:00+00:00,76542.16,76542.16,72353.03,72353.03



BMSB – SAMMANFATTNING


,Valuta,Jämförda dagar,Max avvikelse,Matchar referens
0,ETH,3962,0.0,True
1,BTC,4179,0.0,True



### Resultat och slutsats – Steg 18

#### Syfte

Vi undersökte om våra tekniska indikatorer
var korrekt beräknade och om det fanns
tecken på dataläckage.

#### Genomförda kontroller

Vi kontrollerade:

- SMA 200
- Historisk avkastning under 30 dagar
- Bull Market Support Band
- Veckobaserad uppdatering av BMSB

#### Resultat

Samtliga numeriska kontroller godkändes.

**Ethereum**
- SMA 200: godkänd
- Historisk 30-dagarsavkastning: godkänd
- BMSB: 3 962 jämförda dagar
- Maximal BMSB-avvikelse: 0,00

**Bitcoin**
- SMA 200: godkänd
- Historisk 30-dagarsavkastning: godkänd
- BMSB: 4 179 jämförda dagar
- Maximal BMSB-avvikelse: 0,00

#### Tolkning

Indikatorerna överensstämmer med
de oberoende kontrollberäkningarna.

BMSB-värdena motsvarar en beräkning
med avslutade veckor som fördelas
på efterföljande dagar.

#### Begränsningar

Numerisk överensstämmelse bevisar inte
att alla historiska indata var tillgängliga
vid respektive signalögonblick.

Vi behöver även verifiera RSI, volatilitet,
pristidsstämplar och modellens exekvering.

Den tidigare valda köpgränsen 60 %
har undersökts på testperiodens data.

#### Slutsats

Vi har inte hittat några numeriska fel
i de kontrollerade indikatorerna.

Resultaten stärker den tekniska
kvaliteten på vår datahantering.

Ytterligare tidskontroller behövs
innan handelsstrategin kan betraktas
som fullständigt verifierad.


In [44]:

# STEG 19.1 – Kontroll av datum och tidsordning

import pandas as pd
import numpy as np
from IPython.display import display

rows = []

for asset, source in [
    ("ETH", df_eth_ml),
    ("BTC", df_btc_ml)
]:

    data = source.copy()
    data["time"] = pd.to_datetime(
        data["time"], utc=True
    )

    data = data.sort_values("time")

    for horizon in [30, 180]:

        return_col = f"Return_{horizon}D"

        valid = data.dropna(
            subset=FEATURE_COLUMNS + [return_col]
        ).copy()

        valid["Utfallsslut"] = (
            valid["time"]
            + pd.Timedelta(days=horizon + 1)
        )

        # Kontrollera att framtida utfall är
        # kända senast vid datasetets sista datum
        future_errors = (
            valid["Utfallsslut"] >
            data["time"].max()
        ).sum()

        # Dubbla observationer
        duplicate_dates = (
            valid["time"].duplicated().sum()
        )

        # Kontroll av tidsordning
        sorted_dates = (
            valid["time"].is_monotonic_increasing
        )

        # Luckor i den giltiga tidsserien
        gaps = (
            valid["time"].diff()
            > pd.Timedelta(days=1)
        ).sum()

        rows.append({
            "Valuta": asset,
            "Horisont": f"{horizon}D",
            "Observationer": len(valid),
            "Utfall efter datans slut": int(future_errors),
            "Dubbla datum": int(duplicate_dates),
            "Datum sorterade": bool(sorted_dates),
            "Tidsluckor": int(gaps)
        })

df_time_audit = pd.DataFrame(rows)

print("STEG 19 – TIDSKONTROLL")
display(df_time_audit)


STEG 19 – TIDSKONTROLL


,Valuta,Horisont,Observationer,Utfall efter datans slut,Dubbla datum,Datum sorterade,Tidsluckor
0,ETH,30D,3874,0,0,True,0
1,ETH,180D,3724,0,0,True,0
2,BTC,30D,4093,0,0,True,0
3,BTC,180D,3943,0,0,True,0



### Resultat och slutsats – Steg 19

#### Syfte

Vi kontrollerade datumstrukturen och
tidsordningen i våra dataset för
Ethereum och Bitcoin.

#### Resultat

Kontrollen omfattade fyra dataset:

- Ethereum 30 dagar: 3 874 observationer
- Ethereum 180 dagar: 3 724 observationer
- Bitcoin 30 dagar: 4 093 observationer
- Bitcoin 180 dagar: 3 943 observationer

Samtliga dataset klarade kontrollerna:

- Inga dubbla datum
- Inga tidsluckor bland giltiga observationer
- Korrekt sorterade datum
- Inga utfall som sträcker sig
  förbi datamaterialets slutdatum

#### Tolkning

Resultaten visar att våra tidsserier
har en konsekvent datumstruktur.

Detta är en viktig förutsättning för
tidsbaserad modellvalidering och backtesting.

#### Begränsningar

Kontrollen verifierar inte i sig
att samtliga historiska indikatorer
var tillgängliga vid signalögonblicket.

Modellernas tidsordning, RSI,
volatilitet och exekvering behöver
fortfarande granskas.

#### Slutsats

Inga strukturella datumproblem
identifierades i de kontrollerade dataseten.

Vi har därmed slutfört den grundläggande
tidsseriekontrollen.

Nästa steg är att förbereda resultaten
för visualisering i Streamlit.


In [45]:

# STEG 20.1 – Exportera historiska ML-resultat

from pathlib import Path
import pandas as pd

# Hitta projektets data-mapp
current = Path.cwd()

candidates = [
    current / "data",
    current / "Ethereum_Bitcoin_ML" / "data",
    current / "milena_crypto_ml" / "data"
]

output_dir = next(
    (p for p in candidates if p.is_dir()),
    current / "data"
)

output_dir.mkdir(parents=True, exist_ok=True)

# Historiska ML-prognoser
df_final_predictions.to_csv(
    output_dir / "ml_historical_predictions.csv",
    index=False,
    encoding="utf-8-sig"
)

# Simulerade Ethereum-affärer
df_trades.to_csv(
    output_dir / "eth_ml_trades.csv",
    index=False,
    encoding="utf-8-sig"
)

# Kapitalutveckling
df_equity.to_csv(
    output_dir / "eth_ml_equity.csv",
    index=False,
    encoding="utf-8-sig"
)

# Jämförelse med Buy & Hold
df_equity_summary.to_csv(
    output_dir / "eth_ml_strategy_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("EXPORT KLAR!")
print("Sparat i:", output_dir.resolve())

for filename in [
    "ml_historical_predictions.csv",
    "eth_ml_trades.csv",
    "eth_ml_equity.csv",
    "eth_ml_strategy_summary.csv"
]:
    print("✓", filename)


EXPORT KLAR!
Sparat i: C:\Users\milen\source\repos\Statistisk_programmering_ML\Ethereum_Bitcoin_ML\data
✓ ml_historical_predictions.csv
✓ eth_ml_trades.csv
✓ eth_ml_equity.csv
✓ eth_ml_strategy_summary.csv


In [46]:

# STEG 20.2 – Streamlit: Historisk ML-utvärdering

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

st.set_page_config(
    page_title="ML Backtesting",
    page_icon="📊",
    layout="wide"
)

DATA_DIR = Path(r"C:\Users\milen\source\repos\Statistisk_programmering_ML\Ethereum_Bitcoin_ML\data")

st.title("Ethereum & Bitcoin – Machine Learning")
st.markdown(
    "Historiska ML-prognoser, modellutvärdering "
    "och simulerade handelsresultat."
)

st.warning(
    "Detta är en explorativ historisk analys, "
    "inte en liveprognos eller investeringsrekommendation. "
    "Handelsgränsen valdes efter analys av testdata."
)


@st.cache_data
def load_data():
    predictions = pd.read_csv(
        DATA_DIR / "ml_historical_predictions.csv",
        parse_dates=["time"]
    )
    trades = pd.read_csv(
        DATA_DIR / "eth_ml_trades.csv",
        parse_dates=["Signaldatum", "Köpdatum", "Säljdatum"]
    )
    equity = pd.read_csv(
        DATA_DIR / "eth_ml_equity.csv",
        parse_dates=["time"]
    )
    summary = pd.read_csv(
        DATA_DIR / "eth_ml_strategy_summary.csv"
    )
    return predictions, trades, equity, summary


try:
    predictions, trades, equity, summary = load_data()
except FileNotFoundError as e:
    st.error(f"Saknad datafil: {e.filename}")
    st.stop()


# SIDOMENY
asset = st.sidebar.selectbox(
    "Välj kryptovaluta",
    ["ETH", "BTC"]
)

horizon = st.sidebar.selectbox(
    "Prognoshorisont",
    ["30D", "180D"]
)

model = st.sidebar.selectbox(
    "Välj modell",
    [
        "Logistic Regression",
        "Random Forest",
        "Baseline"
    ]
)


# 1 – HISTORISKA PROGNOSER
st.header("1. Historiska ML-prognoser")

filtered = predictions[
    (predictions["Valuta"] == asset)
    & (predictions["Horisont"] == horizon)
    & (predictions["Modell"] == model)
].copy().sort_values("time")

if filtered.empty:
    st.info("Inga prognoser för valt alternativ.")
    st.stop()

avg_prediction = filtered["Sannolikhet"].mean() * 100
actual_up = filtered["Faktiskt_Utfall"].mean() * 100

c1, c2, c3 = st.columns(3)

c1.metric(
    "Genomsnittlig prognos",
    f"{avg_prediction:.1f} %"
)

c2.metric(
    "Faktisk andel uppgångar",
    f"{actual_up:.1f} %"
)

c3.metric(
    "Historiska observationer",
    f"{len(filtered):,}"
)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=filtered["time"],
    y=filtered["Sannolikhet"] * 100,
    mode="lines",
    name="ML-prognos"
))

fig.add_hline(
    y=50,
    line_dash="dash",
    annotation_text="50 %"
)

fig.update_layout(
    title=f"{asset} – {model} ({horizon})",
    xaxis_title="Datum",
    yaxis_title="Sannolikhet för uppgång (%)",
    yaxis_range=[0, 100],
    template="plotly_dark",
    height=480
)

st.plotly_chart(fig, use_container_width=True)


# 2 – KALIBRERING
st.header("2. Prognosernas historiska tillförlitlighet")

calibration = filtered.copy()

calibration["Grupp"] = pd.qcut(
    calibration["Sannolikhet"],
    q=5,
    labels=False,
    duplicates="drop"
)

cal = (
    calibration.groupby("Grupp")
    .agg(
        Prognos=("Sannolikhet", "mean"),
        Faktiskt=("Faktiskt_Utfall", "mean"),
        Observationer=("Faktiskt_Utfall", "size")
    )
    .reset_index()
)

cal["Prognos"] *= 100
cal["Faktiskt"] *= 100

fig_cal = go.Figure()

fig_cal.add_trace(go.Scatter(
    x=cal["Prognos"],
    y=cal["Faktiskt"],
    mode="lines+markers",
    name="Historiskt utfall"
))

fig_cal.add_trace(go.Scatter(
    x=[0, 100],
    y=[0, 100],
    mode="lines",
    line=dict(dash="dash", color="white"),
    name="Perfekt kalibrering"
))

fig_cal.update_layout(
    title="Prognostiserat kontra faktiskt utfall",
    xaxis_title="Prognos (%)",
    yaxis_title="Faktiskt positiva utfall (%)",
    xaxis_range=[0, 100],
    yaxis_range=[0, 100],
    template="plotly_dark",
    height=480
)

st.plotly_chart(fig_cal, use_container_width=True)


# 3 – BACKTESTING, ENDAST ETH 30D
st.header("3. Historisk handelsstrategi")

if asset == "ETH" and horizon == "30D":

    st.caption(
        "Explorativ ETH-strategi med Logistic Regression, "
        "30 dagars innehav och 60 % signalgräns. "
        "Backtestet förändras inte av modellväljaren ovan."
    )

    ml = summary[
        summary["Strategi"] == "ML-strategi"
    ].iloc[0]

    bh = summary[
        summary["Strategi"] == "Buy & Hold"
    ].iloc[0]

    col1, col2, col3 = st.columns(3)

    col1.metric(
        "ML-avkastning",
        f"{ml['Total avkastning (%)']:.2f} %"
    )

    col2.metric(
        "Buy & Hold",
        f"{bh['Total avkastning (%)']:.2f} %"
    )

    col3.metric(
        "Simulerade affärer",
        len(trades)
    )

    fig_equity = go.Figure()

    fig_equity.add_trace(go.Scatter(
        x=equity["time"],
        y=equity["ML_Strategy"],
        name="ML-strategi"
    ))

    fig_equity.add_trace(go.Scatter(
        x=equity["time"],
        y=equity["Buy_Hold"],
        name="Buy & Hold"
    ))

    fig_equity.update_layout(
        title="Kapitalutveckling från 10 000 USD",
        xaxis_title="Datum",
        yaxis_title="Portföljvärde (USD)",
        template="plotly_dark",
        height=500
    )

    st.plotly_chart(
        fig_equity,
        use_container_width=True
    )

    st.subheader("Simulerade affärer")

    st.dataframe(
        trades.round(2),
        use_container_width=True,
        hide_index=True
    )

    st.subheader("Risk – Maximal drawdown")

    fig_dd = go.Figure()

    fig_dd.add_trace(go.Scatter(
        x=equity["time"],
        y=equity["ML_Strategy_DD"],
        name="ML-strategi"
    ))

    fig_dd.add_trace(go.Scatter(
        x=equity["time"],
        y=equity["Buy_Hold_DD"],
        name="Buy & Hold"
    ))

    fig_dd.update_layout(
        yaxis_title="Drawdown (%)",
        template="plotly_dark",
        height=450
    )

    st.plotly_chart(
        fig_dd,
        use_container_width=True
    )

else:
    st.info(
        "Den nuvarande handelsstrategin är endast "
        "backtestad för ETH på 30 dagars sikt."
    )


# 4 – SLUTSATS
st.header("4. Slutsats")

st.markdown("""
- ML-modellernas kvalitet varierar mellan olika perioder.
- Hög prognossannolikhet garanterar inte positiv avkastning.
- ETH Logistic Regression 30D är en kandidat för fortsatt analys.
- Den historiska strategin har inte bekräftats på helt nya data.
- Nya modeller och signalgränser måste valideras innan praktisk användning.
""")


2026-11-12 09:01:53.694 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-12 09:01:53.696 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-12 09:01:54.129 
  command:

    streamlit run c:\Users\milen\AppData\Local\Programs\Python\Python314\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-11-12 09:01:54.130 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-12 09:01:54.130 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-12 09:01:54.131 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-12 09:01:54.131 Thread 'MainThread': missing ScriptRunContext! This wa

DeltaGenerator()


# Steg 21 – Marknadscykler: Bull & Bear Markets

## Syfte

Syftet med analysen är att identifiera
historiska marknadscykler för Ethereum (ETH)
och Bitcoin (BTC).

Vi undersöker:

1. När marknaden befunnit sig i en bull market.
2. När marknaden befunnit sig i en bear market.
3. Hur länge varje marknadsfas varat.
4. Hur stora prisförändringarna varit.
5. Hur marknadscyklerna kan visualiseras
   i vår gemensamma Streamlit-app.

## Metod

Vi börjar med en regelbaserad klassificering
utifrån två indikatorer:

- SMA 200: glidande medelvärde för 200 dagar.
- SMA 200:s riktning under de senaste 30 dagarna.

Vi använder tre marknadsfaser:

**Bull Market**
Priset ligger över SMA 200 samtidigt som
SMA 200 stiger jämfört med 30 dagar tidigare.

**Bear Market**
Priset ligger under SMA 200 samtidigt som
SMA 200 faller jämfört med 30 dagar tidigare.

**Övergångsfas**
Indikatorerna ger motstridiga signaler.

Detta är en indikatorbaserad definition
och inte en universell definition av
historiska bull- och bear-marknader.

## Viktig metodbegränsning

Klassificeringen använder enbart aktuella
och tidigare observationer.

Vi kommer senare att jämföra den med
en separat historisk cykeldefinition
baserad på pristoppar och nedgångar.

Vi skiljer mellan marknadsfaser som kan
identifieras löpande och historiska
cykler som fastställs i efterhand.


In [47]:

# =====================================================
# STEG 21.2 – Identifiera Bull & Bear Markets
# =====================================================

import pandas as pd
import numpy as np
from IPython.display import display

def classify_market_cycles(data, asset):

    df = data.copy()

    df["time"] = pd.to_datetime(
        df["time"], utc=True
    )

    df = (
        df.sort_values("time")
        .drop_duplicates("time")
        .reset_index(drop=True)
    )

    price_col = f"{asset}_Price"

    df[price_col] = pd.to_numeric(
        df[price_col],
        errors="coerce"
    )

    # SMA 200
    df["SMA_200"] = (
        df[price_col]
        .rolling(200, min_periods=200)
        .mean()
    )

    # Utvecklingen i SMA 200 över 30 dagar
    df["SMA_Trend_30D"] = (
        df["SMA_200"]
        - df["SMA_200"].shift(30)
    )

    # Tre marknadsfaser
    bull = (
        (df[price_col] > df["SMA_200"])
        & (df["SMA_Trend_30D"] > 0)
    )

    bear = (
        (df[price_col] < df["SMA_200"])
        & (df["SMA_Trend_30D"] < 0)
    )

    df["Market_Phase"] = np.select(
        [bull, bear],
        ["Bull Market", "Bear Market"],
        default="Övergångsfas"
    )

    # Perioder innan tillräcklig historik finns
    missing = (
        df["SMA_200"].isna()
        | df["SMA_Trend_30D"].isna()
        | df[price_col].isna()
    )

    df.loc[
        missing, "Market_Phase"
    ] = "Otillräcklig data"

    return df


df_eth_cycles = classify_market_cycles(
    df_eth_ml, "ETH"
)

df_btc_cycles = classify_market_cycles(
    df_btc_ml, "BTC"
)


# Visa fördelning av marknadsfaser
for asset, df in [
    ("ETH", df_eth_cycles),
    ("BTC", df_btc_cycles)
]:

    print(f"\n===== {asset} – MARKNADSFASER =====")

    summary = (
        df["Market_Phase"]
        .value_counts()
        .rename_axis("Marknadsfas")
        .reset_index(name="Antal dagar")
    )

    summary["Andel (%)"] = (
        summary["Antal dagar"] / len(df) * 100
    ).round(2)

    display(summary)

    print("\nSenaste observationerna:")

    display(
        df[
            [
                "time",
                f"{asset}_Price",
                "SMA_200",
                "SMA_Trend_30D",
                "Market_Phase"
            ]
        ].tail(10).round(2)
    )



===== ETH – MARKNADSFASER =====


,Marknadsfas,Antal dagar,Andel (%)
0,Bull Market,1937,47.20
1,Bear Market,1138,27.73
2,Övergångsfas,800,19.49
3,Otillräcklig data,229,5.58



Senaste observationerna:


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\2138800630.py:115: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,ETH_Price,SMA_200,SMA_Trend_30D,Market_Phase
4094,2026-10-23 00:00:00+00:00,2378.26,2155.88,68.62,Bull Market
4095,2026-10-24 00:00:00+00:00,2351.99,2156.40,65.41,Bull Market
4096,2026-10-25 00:00:00+00:00,2325.80,2157.09,62.62,Bull Market
4097,2026-10-26 00:00:00+00:00,2316.60,2157.72,59.96,Bull Market
4098,2026-10-27 00:00:00+00:00,2347.56,2158.24,57.32,Bull Market
4099,2026-10-28 00:00:00+00:00,2356.99,2158.59,54.62,Bull Market
4100,2026-10-29 00:00:00+00:00,2256.03,2158.92,52.02,Bull Market
4101,2026-10-30 00:00:00+00:00,2240.23,2158.25,48.41,Bull Market
4102,2026-10-31 00:00:00+00:00,2252.07,2157.90,45.42,Bull Market
4103,2026-11-01 00:00:00+00:00,2269.99,2157.46,43.41,Bull Market



===== BTC – MARKNADSFASER =====


,Marknadsfas,Antal dagar,Andel (%)
0,Bull Market,2322,53.71
1,Bear Market,1006,23.27
2,Övergångsfas,766,17.72
3,Otillräcklig data,229,5.30



Senaste observationerna:


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\2138800630.py:115: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,BTC_Price,SMA_200,SMA_Trend_30D,Market_Phase
4313,2026-10-23 00:00:00+00:00,80924.18,72849.49,2092.29,Bull Market
4314,2026-10-24 00:00:00+00:00,80699.31,72892.70,2044.58,Bull Market
4315,2026-10-25 00:00:00+00:00,80038.91,72937.47,2011.68,Bull Market
4316,2026-10-26 00:00:00+00:00,79949.93,72978.27,1980.01,Bull Market
4317,2026-10-27 00:00:00+00:00,80800.54,73017.55,1948.82,Bull Market
4318,2026-10-28 00:00:00+00:00,81237.44,73058.14,1924.52,Bull Market
4319,2026-10-29 00:00:00+00:00,78193.43,73095.68,1898.34,Bull Market
4320,2026-10-30 00:00:00+00:00,77109.59,73108.06,1848.51,Bull Market
4321,2026-10-31 00:00:00+00:00,77401.02,73124.20,1804.32,Bull Market
4322,2026-11-01 00:00:00+00:00,78135.08,73141.07,1772.22,Bull Market


In [48]:

# =====================================================
# STEG 21.3 – Visualisera historiska marknadsfaser
# =====================================================

import plotly.graph_objects as go

phase_colors = {
    "Bull Market": "rgba(46, 204, 113, 0.15)",
    "Bear Market": "rgba(255, 82, 82, 0.15)",
    "Övergångsfas": "rgba(160, 160, 160, 0.10)"
}

def plot_market_cycles(data, asset):

    df = data.copy().sort_values("time")
    price_col = f"{asset}_Price"

    # Identifiera sammanhängande perioder
    # inklusive eventuella datumluckor
    new_phase = (
        df["Market_Phase"].ne(
            df["Market_Phase"].shift()
        )
        | df["time"].diff().gt(
            pd.Timedelta(days=1)
        )
    )

    df["Phase_ID"] = new_phase.cumsum()

    fig = go.Figure()

    # Färgade bakgrunder för varje marknadsperiod
    for _, period in df.groupby("Phase_ID"):

        phase = period["Market_Phase"].iloc[0]

        if phase not in phase_colors:
            continue

        start = period["time"].iloc[0]
        end = period["time"].iloc[-1] + pd.Timedelta(days=1)

        fig.add_vrect(
            x0=start,
            x1=end,
            fillcolor=phase_colors[phase],
            opacity=1,
            line_width=0,
            layer="below"
        )

    # Historisk prisutveckling
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df[price_col],
        mode="lines",
        name=f"{asset}-pris",
        line=dict(
            color="#5DADE2",
            width=1.8
        )
    ))

    # SMA 200
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["SMA_200"],
        mode="lines",
        name="SMA 200",
        line=dict(
            color="#F5B041",
            width=2
        )
    ))

    fig.update_layout(
        title=f"{asset} – Historiska Bull & Bear Markets",
        xaxis_title="Datum",
        yaxis_title="Pris (USD, logaritmisk skala)",
        yaxis_type="log",
        template="plotly_dark",
        height=650,
        hovermode="x unified"
    )

    fig.update_xaxes(
        rangeslider_visible=True
    )

    fig.show()


plot_market_cycles(df_eth_cycles, "ETH")
plot_market_cycles(df_btc_cycles, "BTC")


In [49]:

# =====================================================
# STEG 21.4 – Sammanfattning per marknadsperiod
# =====================================================

def summarize_market_cycles(data, asset):

    df = data.copy().sort_values("time")

    price_col = f"{asset}_Price"

    new_phase = (
        df["Market_Phase"].ne(
            df["Market_Phase"].shift()
        )
        | df["time"].diff().gt(
            pd.Timedelta(days=1)
        )
    )

    df["Phase_ID"] = new_phase.cumsum()

    # Exkludera perioden med otillräcklig data
    valid = df[
        df["Market_Phase"] != "Otillräcklig data"
    ].copy()

    summary = (
        valid.groupby("Phase_ID")
        .agg(
            Marknadsfas=("Market_Phase", "first"),
            Start=("time", "min"),
            Slut=("time", "max"),
            Antal_dagar=("time", "size"),
            Startpris=(price_col, "first"),
            Slutpris=(price_col, "last")
        )
        .reset_index(drop=True)
    )

    summary["Prisförändring (%)"] = (
        (
            summary["Slutpris"]
            / summary["Startpris"]
        ) - 1
    ) * 100

    summary["Valuta"] = asset

    return summary


df_eth_periods = summarize_market_cycles(
    df_eth_cycles, "ETH"
)

df_btc_periods = summarize_market_cycles(
    df_btc_cycles, "BTC"
)

for asset, df in [
    ("ETH", df_eth_periods),
    ("BTC", df_btc_periods)
]:

    print(f"\n===== {asset} – HISTORISKA MARKNADSPERIODER =====")

    display(
        df[
            [
                "Marknadsfas",
                "Start",
                "Slut",
                "Antal_dagar",
                "Prisförändring (%)"
            ]
        ].tail(20).round(2)
    )



===== ETH – HISTORISKA MARKNADSPERIODER =====


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\3688592355.py:76: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(20).round(2)


,Marknadsfas,Start,Slut,Antal_dagar,Prisförändring (%)
64,Bear Market,2024-09-10 00:00:00+00:00,2024-11-07 00:00:00+00:00,59,21.02
65,Övergångsfas,2024-11-08 00:00:00+00:00,2024-11-29 00:00:00+00:00,22,21.46
66,Bull Market,2024-11-30 00:00:00+00:00,2024-12-31 00:00:00+00:00,32,-10.25
67,Övergångsfas,2025-01-01 00:00:00+00:00,2025-01-26 00:00:00+00:00,26,-3.28
68,Bull Market,2025-01-27 00:00:00+00:00,2025-02-01 00:00:00+00:00,6,-1.59
69,Bear Market,2025-02-02 00:00:00+00:00,2025-06-08 00:00:00+00:00,127,-11.77
70,Övergångsfas,2025-06-09 00:00:00+00:00,2025-06-12 00:00:00+00:00,4,-0.89
71,Bear Market,2025-06-13 00:00:00+00:00,2025-07-01 00:00:00+00:00,19,-6.88
72,Övergångsfas,2025-07-02 00:00:00+00:00,2025-08-04 00:00:00+00:00,34,44.15
73,Bull Market,2025-08-05 00:00:00+00:00,2025-11-03 00:00:00+00:00,91,-0.51



===== BTC – HISTORISKA MARKNADSPERIODER =====


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\3688592355.py:76: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(20).round(2)


,Marknadsfas,Start,Slut,Antal_dagar,Prisförändring (%)
66,Övergångsfas,2025-03-09 00:00:00+00:00,2025-03-11 00:00:00+00:00,3,2.60
67,Bull Market,2025-03-12 00:00:00+00:00,2025-03-12 00:00:00+00:00,1,0.00
68,Övergångsfas,2025-03-13 00:00:00+00:00,2025-03-13 00:00:00+00:00,1,0.00
69,Bull Market,2025-03-14 00:00:00+00:00,2025-03-15 00:00:00+00:00,2,0.29
70,Övergångsfas,2025-03-16 00:00:00+00:00,2025-03-18 00:00:00+00:00,3,0.23
71,Bull Market,2025-03-19 00:00:00+00:00,2025-03-19 00:00:00+00:00,1,0.00
72,Övergångsfas,2025-03-20 00:00:00+00:00,2025-03-22 00:00:00+00:00,3,-0.36
73,Bull Market,2025-03-23 00:00:00+00:00,2025-03-27 00:00:00+00:00,5,1.69
74,Övergångsfas,2025-03-28 00:00:00+00:00,2025-04-21 00:00:00+00:00,25,3.62
75,Bull Market,2025-04-22 00:00:00+00:00,2025-10-16 00:00:00+00:00,178,15.66



## Steg 22 – Stabilare klassificering av marknadsfaser

### Syfte

I steg 21 identifierade vi Bull Markets,
Bear Markets och övergångsfaser med SMA 200.

Resultaten visade flera mycket korta perioder
och snabba växlingar mellan marknadsfaser.

Vi undersöker därför om klassificeringen
kan bli stabilare med ett bekräftelsekrav.

### Metod

För att bekräfta en Bull Market krävs
14 dagar i följd med:

- Pris över SMA 200.
- SMA 200 högre än 30 dagar tidigare.

För en Bear Market krävs motsvarande
14 dagar med motsatta villkor.

Övriga dagar klassificeras som
övergångsfas.

Vi jämför den ursprungliga klassificeringen
med den nya modellen.

### Begränsning

Ett bekräftelsekrav kan minska tillfälliga
signaler men innebär samtidigt en fördröjning.

14 dagar är en experimentell parameter,
inte en optimerad eller bevisat bästa gräns.


In [50]:

# STEG 22.2 – Bekräfta Bull och Bear efter 14 dagar

import pandas as pd
import numpy as np
from IPython.display import display

CONFIRM_DAYS = 14

def confirm_market_phases(source, asset, confirm_days=14):

    df = source.copy().sort_values("time").reset_index(drop=True)
    price_col = f"{asset}_Price"

    bull = df["Market_Phase"].eq("Bull Market")
    bear = df["Market_Phase"].eq("Bear Market")

    # Kräver 14 konsekutiva dagar med samma status
    bull_confirmed = (
        bull.astype(int)
        .rolling(confirm_days, min_periods=confirm_days)
        .sum()
        .eq(confirm_days)
    )

    bear_confirmed = (
        bear.astype(int)
        .rolling(confirm_days, min_periods=confirm_days)
        .sum()
        .eq(confirm_days)
    )

    df["Confirmed_Phase"] = np.select(
        [bull_confirmed, bear_confirmed],
        ["Bull Market", "Bear Market"],
        default="Övergångsfas"
    )

    df.loc[
        df["Market_Phase"].eq("Otillräcklig data"),
        "Confirmed_Phase"
    ] = "Otillräcklig data"

    return df


df_eth_confirmed = confirm_market_phases(
    df_eth_cycles, "ETH", CONFIRM_DAYS
)

df_btc_confirmed = confirm_market_phases(
    df_btc_cycles, "BTC", CONFIRM_DAYS
)


for asset, df in [
    ("ETH", df_eth_confirmed),
    ("BTC", df_btc_confirmed)
]:

    print(f"\n===== {asset} – BEKRÄFTADE MARKNADSFASER =====")

    result = (
        df["Confirmed_Phase"]
        .value_counts()
        .rename_axis("Marknadsfas")
        .reset_index(name="Antal dagar")
    )

    result["Andel (%)"] = (
        result["Antal dagar"] / len(df) * 100
    ).round(2)

    display(result)

    print("Senaste observationerna:")

    display(
        df[
            ["time", f"{asset}_Price",
             "Market_Phase", "Confirmed_Phase"]
        ].tail(10)
    )



===== ETH – BEKRÄFTADE MARKNADSFASER =====


,Marknadsfas,Antal dagar,Andel (%)
0,Bull Market,1698,41.37
1,Övergångsfas,1216,29.63
2,Bear Market,961,23.42
3,Otillräcklig data,229,5.58


Senaste observationerna:


,time,ETH_Price,Market_Phase,Confirmed_Phase
4094,2026-10-23 00:00:00+00:00,2378.260399,Bull Market,Bull Market
4095,2026-10-24 00:00:00+00:00,2351.985153,Bull Market,Bull Market
4096,2026-10-25 00:00:00+00:00,2325.796597,Bull Market,Bull Market
4097,2026-10-26 00:00:00+00:00,2316.598625,Bull Market,Bull Market
4098,2026-10-27 00:00:00+00:00,2347.562225,Bull Market,Bull Market
4099,2026-10-28 00:00:00+00:00,2356.988509,Bull Market,Bull Market
4100,2026-10-29 00:00:00+00:00,2256.031994,Bull Market,Bull Market
4101,2026-10-30 00:00:00+00:00,2240.229980,Bull Market,Bull Market
4102,2026-10-31 00:00:00+00:00,2252.073358,Bull Market,Bull Market
4103,2026-11-01 00:00:00+00:00,2269.988406,Bull Market,Bull Market



===== BTC – BEKRÄFTADE MARKNADSFASER =====


,Marknadsfas,Antal dagar,Andel (%)
0,Bull Market,2083,48.18
1,Övergångsfas,1113,25.75
2,Bear Market,898,20.77
3,Otillräcklig data,229,5.30


Senaste observationerna:


,time,BTC_Price,Market_Phase,Confirmed_Phase
4313,2026-10-23 00:00:00+00:00,80924.183766,Bull Market,Bull Market
4314,2026-10-24 00:00:00+00:00,80699.313127,Bull Market,Bull Market
4315,2026-10-25 00:00:00+00:00,80038.905803,Bull Market,Bull Market
4316,2026-10-26 00:00:00+00:00,79949.934329,Bull Market,Bull Market
4317,2026-10-27 00:00:00+00:00,80800.537372,Bull Market,Bull Market
4318,2026-10-28 00:00:00+00:00,81237.437826,Bull Market,Bull Market
4319,2026-10-29 00:00:00+00:00,78193.426404,Bull Market,Bull Market
4320,2026-10-30 00:00:00+00:00,77109.590747,Bull Market,Bull Market
4321,2026-10-31 00:00:00+00:00,77401.017287,Bull Market,Bull Market
4322,2026-11-01 00:00:00+00:00,78135.080289,Bull Market,Bull Market


In [51]:

# STEG 22.3 – Interaktiva diagram

eth_plot = df_eth_confirmed.copy()
btc_plot = df_btc_confirmed.copy()

# Diagramfunktionen använder Market_Phase
eth_plot["Market_Phase"] = eth_plot["Confirmed_Phase"]
btc_plot["Market_Phase"] = btc_plot["Confirmed_Phase"]

print("ETH – Bekräftade Bull & Bear Markets")
plot_market_cycles(eth_plot, "ETH")

print("BTC – Bekräftade Bull & Bear Markets")
plot_market_cycles(btc_plot, "BTC")


ETH – Bekräftade Bull & Bear Markets


BTC – Bekräftade Bull & Bear Markets


In [52]:

# STEG 22.4 – Jämförelse av antalet fasbyten

results = []

for asset, df in [
    ("ETH", df_eth_confirmed),
    ("BTC", df_btc_confirmed)
]:

    for column, method in [
        ("Market_Phase", "Ursprunglig"),
        ("Confirmed_Phase", "14 dagars bekräftelse")
    ]:

        valid = df[
            df[column] != "Otillräcklig data"
        ].copy()

        changes = (
            valid[column]
            .ne(valid[column].shift())
            .sum() - 1
        )

        results.append({
            "Valuta": asset,
            "Metod": method,
            "Antal fasbyten": max(int(changes), 0),
            "Bull-dagar": int(
                valid[column].eq("Bull Market").sum()
            ),
            "Bear-dagar": int(
                valid[column].eq("Bear Market").sum()
            ),
            "Övergångsdagar": int(
                valid[column].eq("Övergångsfas").sum()
            )
        })

df_phase_comparison = pd.DataFrame(results)

print("JÄMFÖRELSE AV MARKNADSKLASSIFICERING")
display(df_phase_comparison)


JÄMFÖRELSE AV MARKNADSKLASSIFICERING


,Valuta,Metod,Antal fasbyten,Bull-dagar,Bear-dagar,Övergångsdagar
0,ETH,Ursprunglig,83,1937,1138,800
1,ETH,14 dagars bekräftelse,53,1698,961,1216
2,BTC,Ursprunglig,85,2322,1006,766
3,BTC,14 dagars bekräftelse,37,2083,898,1113



### Resultat och slutsats – Steg 22

#### Syfte

Vi undersökte om klassificeringen av
Bull och Bear Markets kunde förbättras
genom att kräva 14 dagars bekräftelse.

#### Metod

Den ursprungliga klassificeringen
baserades på:

- Prisets position relativt SMA 200
- SMA 200:s riktning under 30 dagar

I den nya metoden krävs att villkoren
uppfylls under 14 dagar i följd.

#### Resultat – Ethereum

- Ursprungligt antal fasbyten: 83
- Efter bekräftelse: 53
- Minskning: 36,1 %

Fördelning efter bekräftelse:

- Bull Market: 41,37 %
- Bear Market: 23,42 %
- Övergångsfas: 29,63 %

#### Resultat – Bitcoin

- Ursprungligt antal fasbyten: 85
- Efter bekräftelse: 37
- Minskning: 56,5 %

Fördelning efter bekräftelse:

- Bull Market: 48,18 %
- Bear Market: 20,77 %
- Övergångsfas: 25,75 %

#### Tolkning

Bekräftelsekravet minskade antalet
växlingar mellan marknadsfaserna.

Effekten var särskilt tydlig för Bitcoin.

Metoden ger en stabilare visualisering
men innebär även att förändringar
i marknadstrenden bekräftas senare.

#### Begränsningar

- 14 dagar är en experimentell parameter.
- Metoden kan missa kortare marknadsrörelser.
- En bekräftad fas kan snabbt återgå
  till övergångsfas.
- Färre fasbyten innebär inte automatiskt
  bättre prognoser eller högre avkastning.
- Indikatorbaserade trendfaser är inte
  samma sak som långa historiska cykler.

#### Slutsats

Bekräftelsekravet minskade antalet
fasbyten för både Ethereum och Bitcoin.

Metoden är ett intressant alternativ
för visualisering av marknadstrender.

Nästa steg är att komplettera
klassificeringen med Moving Averages
och Bull Market Support Band.

Vi ska även jämföra indikatorbaserade
trendfaser med längre historiska cykler.



## Steg 23 – Moving Averages och BMSB

### Syfte

Vi undersöker hur olika glidande
medelvärden kan användas för att
identifiera långsiktiga marknadstrender.

Vi analyserar tre indikatorer:

**SMA 200**

Det enkla genomsnittet av de senaste
200 dagarnas priser.

**20-week SMA**

Det enkla genomsnittet av de senaste
20 veckornas stängningspriser.

**21-week EMA**

Ett exponentiellt viktat medelvärde
av 21 veckors stängningspriser.

**Bull Market Support Band (BMSB)**

Området mellan 20-week SMA och
21-week EMA.

### Forskningsfrågor

1. Hur skiljer sig indikatorerna åt?
2. När ligger priset över eller under BMSB?
3. Hur samvarierar indikatorerna med Bull
   och Bear Markets?
4. Kan indikatorerna bidra till en
   tydligare presentation av marknadscykler?

### Metod

Vi använder historiska dagspriser
och beräknar veckobaserade indikatorer
utifrån avslutade veckor.

Indikatorerna visualiseras tillsammans
med prisutvecklingen.

Resultaten är deskriptiva och innebär
inte att indikatorerna har bevisad
förmåga att förutsäga marknadsrörelser.


In [53]:

# STEG 23.2 – SMA 200, 20W SMA och 21W EMA

import pandas as pd
import numpy as np
from IPython.display import display

def build_moving_averages(source, asset):

    df = source.copy()

    df["time"] = pd.to_datetime(df["time"], utc=True)

    df = (
        df.sort_values("time")
        .drop_duplicates("time")
        .set_index("time")
    )

    price_col = f"{asset}_Price"
    price = pd.to_numeric(df[price_col], errors="coerce")

    # SMA 200
    df["SMA_200"] = price.rolling(
        window=200, min_periods=200
    ).mean()

    # Veckopriser, söndag som avslut
    weekly = price.resample("W-SUN").last()

    # 20 veckors SMA
    sma20 = weekly.rolling(
        window=20, min_periods=20
    ).mean()

    # 21 veckors EMA
    ema21 = weekly.ewm(
        span=21,
        adjust=False,
        min_periods=21
    ).mean()

    # Tillgängliga först följande måndag
    weekly_indicators = pd.DataFrame({
        "SMA_20W": sma20,
        "EMA_21W": ema21
    })

    weekly_indicators.index += pd.Timedelta(days=1)

    aligned = weekly_indicators.reindex(
        df.index,
        method="ffill"
    )

    df["SMA_20W"] = aligned["SMA_20W"]
    df["EMA_21W"] = aligned["EMA_21W"]

    df["BMSB_Upper"] = df[
        ["SMA_20W", "EMA_21W"]
    ].max(axis=1)

    df["BMSB_Lower"] = df[
        ["SMA_20W", "EMA_21W"]
    ].min(axis=1)

    # Klassificera prisets position
    df["BMSB_Position"] = np.select(
        [
            price > df["BMSB_Upper"],
            price < df["BMSB_Lower"]
        ],
        [
            "Över BMSB",
            "Under BMSB"
        ],
        default="Inom BMSB"
    )

    missing = (
        df["BMSB_Upper"].isna()
        | df["BMSB_Lower"].isna()
        | price.isna()
    )

    df.loc[
        missing, "BMSB_Position"
    ] = "Otillräcklig data"

    return df.reset_index()


df_eth_ma = build_moving_averages(df_eth_ml, "ETH")
df_btc_ma = build_moving_averages(df_btc_ml, "BTC")

for asset, df in [
    ("ETH", df_eth_ma),
    ("BTC", df_btc_ma)
]:

    print(f"\n===== {asset} – MOVING AVERAGES =====")

    display(
        df[
            [
                "time",
                f"{asset}_Price",
                "SMA_200",
                "SMA_20W",
                "EMA_21W",
                "BMSB_Position"
            ]
        ].tail(10).round(2)
    )

    print("\nFördelning av prisets position:")

    display(
        df["BMSB_Position"]
        .value_counts()
        .rename_axis("Position")
        .reset_index(name="Antal dagar")
    )



===== ETH – MOVING AVERAGES =====


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\3447741415.py:112: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,ETH_Price,SMA_200,SMA_20W,EMA_21W,BMSB_Position
4094,2026-10-23 00:00:00+00:00,2378.26,2155.88,2131.19,2318.52,Över BMSB
4095,2026-10-24 00:00:00+00:00,2351.99,2156.40,2131.19,2318.52,Över BMSB
4096,2026-10-25 00:00:00+00:00,2325.80,2157.09,2131.19,2318.52,Över BMSB
4097,2026-10-26 00:00:00+00:00,2316.60,2157.72,2163.38,2319.18,Inom BMSB
4098,2026-10-27 00:00:00+00:00,2347.56,2158.24,2163.38,2319.18,Över BMSB
4099,2026-10-28 00:00:00+00:00,2356.99,2158.59,2163.38,2319.18,Över BMSB
4100,2026-10-29 00:00:00+00:00,2256.03,2158.92,2163.38,2319.18,Inom BMSB
4101,2026-10-30 00:00:00+00:00,2240.23,2158.25,2163.38,2319.18,Inom BMSB
4102,2026-10-31 00:00:00+00:00,2252.07,2157.90,2163.38,2319.18,Inom BMSB
4103,2026-11-01 00:00:00+00:00,2269.99,2157.46,2163.38,2319.18,Inom BMSB



Fördelning av prisets position:


,Position,Antal dagar
0,Över BMSB,2181
1,Under BMSB,1575
2,Inom BMSB,213
3,Otillräcklig data,135



===== BTC – MOVING AVERAGES =====


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\3447741415.py:112: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ].tail(10).round(2)


,time,BTC_Price,SMA_200,SMA_20W,EMA_21W,BMSB_Position
4313,2026-10-23 00:00:00+00:00,80924.18,72849.49,71509.00,76192.48,Över BMSB
4314,2026-10-24 00:00:00+00:00,80699.31,72892.70,71509.00,76192.48,Över BMSB
4315,2026-10-25 00:00:00+00:00,80038.91,72937.47,71509.00,76192.48,Över BMSB
4316,2026-10-26 00:00:00+00:00,79949.93,72978.27,72353.03,76542.16,Över BMSB
4317,2026-10-27 00:00:00+00:00,80800.54,73017.55,72353.03,76542.16,Över BMSB
4318,2026-10-28 00:00:00+00:00,81237.44,73058.14,72353.03,76542.16,Över BMSB
4319,2026-10-29 00:00:00+00:00,78193.43,73095.68,72353.03,76542.16,Över BMSB
4320,2026-10-30 00:00:00+00:00,77109.59,73108.06,72353.03,76542.16,Över BMSB
4321,2026-10-31 00:00:00+00:00,77401.02,73124.20,72353.03,76542.16,Över BMSB
4322,2026-11-01 00:00:00+00:00,78135.08,73141.07,72353.03,76542.16,Över BMSB



Fördelning av prisets position:


,Position,Antal dagar
0,Över BMSB,2512
1,Under BMSB,1520
2,Inom BMSB,154
3,Otillräcklig data,137


In [54]:

# STEG 23.3 – Visualisera Moving Averages

import plotly.graph_objects as go

def plot_moving_averages(data, asset):

    df = data.copy().sort_values("time")

    price_col = f"{asset}_Price"

    fig = go.Figure()

    # Nedre BMSB
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["BMSB_Lower"],
        mode="lines",
        line=dict(color="#2ECC71", width=1),
        name="Nedre BMSB"
    ))

    # Övre BMSB + färgat område
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["BMSB_Upper"],
        mode="lines",
        line=dict(color="#2ECC71", width=1),
        fill="tonexty",
        fillcolor="rgba(46,204,113,0.15)",
        name="Övre BMSB"
    ))

    # Pris
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df[price_col],
        mode="lines",
        name=f"{asset}-pris",
        line=dict(color="#5DADE2", width=2)
    ))

    # SMA 200
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["SMA_200"],
        mode="lines",
        name="SMA 200",
        line=dict(color="#F39C12", width=2)
    ))

    # 20W SMA
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["SMA_20W"],
        mode="lines",
        name="20W SMA",
        line=dict(color="#A569BD", width=1.8)
    ))

    # 21W EMA
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["EMA_21W"],
        mode="lines",
        name="21W EMA",
        line=dict(color="#F4D03F", width=1.8)
    ))

    fig.update_layout(
        title=f"{asset} – Moving Averages och BMSB",
        xaxis_title="Datum",
        yaxis_title="Pris (USD, logaritmisk skala)",
        yaxis_type="log",
        template="plotly_dark",
        height=650,
        hovermode="x unified"
    )

    fig.update_xaxes(rangeslider_visible=True)

    fig.show()


plot_moving_averages(df_eth_ma, "ETH")
plot_moving_averages(df_btc_ma, "BTC")



### Resultat och slutsats – Steg 23

#### Syfte

Vi undersökte hur olika glidande
medelvärden kan användas för att analysera
marknadstrender för Ethereum och Bitcoin.

#### Metod

Följande indikatorer användes:

- SMA 200 – 200 dagars glidande medelvärde
- 20W SMA – 20 veckors glidande medelvärde
- 21W EMA – 21 veckors exponentiellt medelvärde
- BMSB – området mellan 20W SMA och 21W EMA

Vi klassificerade prisets position som:

- Över BMSB
- Inom BMSB
- Under BMSB

#### Resultat – Ethereum

- Över BMSB: 2 181 dagar
- Under BMSB: 1 575 dagar
- Inom BMSB: 213 dagar
- Otillräcklig data: 135 dagar

#### Resultat – Bitcoin

- Över BMSB: 2 512 dagar
- Under BMSB: 1 520 dagar
- Inom BMSB: 154 dagar
- Otillräcklig data: 137 dagar

#### Tolkning

Bitcoin befann sig över BMSB under
en större andel av observationsperioden
än Ethereum.

Diagrammen visar att de veckobaserade
indikatorerna följer långsiktiga
prisförändringar men reagerar med
viss fördröjning.

SMA 200 ger en kompletterande bild
av marknadens långsiktiga trend.

#### Begränsningar

Indikatorerna baseras på historiska priser
och kan inte ensamma förutsäga framtiden.

Att priset ligger över BMSB innebär
inte automatiskt att priset kommer stiga.

Vi behöver därför undersöka sambandet
mellan indikatorernas position och
faktisk framtida avkastning.

#### Slutsats

Moving Averages och BMSB är användbara
för visualisering av historiska
marknadstrender.

De kan komplettera gruppens Shared
Curvature Quantile Regression genom
att visa trendinformation tillsammans
med regressionsbanden.

Nästa steg är att statistiskt undersöka
hur framtida avkastning har varierat
beroende på prisets position mot BMSB.



## Steg 24 – BMSB och framtida avkastning

### Syfte

Vi undersöker om prisets position
relativt Bull Market Support Band
har samband med framtida avkastning.

### Forskningsfråga

Har Ethereum och Bitcoin historiskt
utvecklats olika beroende på om priset
befunnit sig över, inom eller under BMSB?

### Metod

Vi delar in observationerna i tre grupper:

1. Över BMSB
2. Inom BMSB
3. Under BMSB

För varje grupp undersöker vi avkastningen
efter 30 respektive 180 kalenderdagar.

Vi beräknar:

- Antal observationer
- Medianavkastning
- Medelavkastning
- Andel positiva utfall
- Andel förluster större än 20 %

### Viktiga begränsningar

Analysen är deskriptiv.

Framtida avkastningsperioder överlappar,
vilket innebär att observationerna
inte är statistiskt oberoende.

Resultaten beskriver historiska samband
och är inte en validerad handelsstrategi.


In [55]:

# STEG 24.2 – BMSB och framtida avkastning

import pandas as pd
import numpy as np
from IPython.display import display

HORIZONS = [30, 180]

def analyze_bmsb_returns(source, asset):

    df = source.copy()

    df["time"] = pd.to_datetime(
        df["time"], utc=True
    )

    df = (
        df.sort_values("time")
        .drop_duplicates("time")
        .set_index("time")
    )

    price_col = f"{asset}_Price"

    price = pd.to_numeric(
        df[price_col],
        errors="coerce"
    )

    # Beräkna framtida priser med exakta datum
    results = []

    for days in HORIZONS:

        future_prices = pd.DataFrame({
            "time": df.index,
            "Future_Price": price.values
        })

        # Koppla framtida pris till dagens datum
        future_prices["time"] -= pd.Timedelta(days=days)

        future_prices = future_prices.set_index("time")

        future_price = future_prices[
            "Future_Price"
        ].reindex(df.index)

        df["Future_Return"] = (
            future_price / price - 1
        ) * 100

        valid = df[
            df["BMSB_Position"].isin([
                "Över BMSB",
                "Inom BMSB",
                "Under BMSB"
            ])
            & df["Future_Return"].notna()
        ].copy()

        for position, group in valid.groupby("BMSB_Position"):

            returns = group["Future_Return"]

            results.append({
                "Valuta": asset,
                "Horisont": f"{days}D",
                "BMSB_Position": position,
                "Observationer": len(group),
                "Medianavkastning (%)": returns.median(),
                "Medelavkastning (%)": returns.mean(),
                "Positiva utfall (%)": (
                    returns.gt(0).mean() * 100
                ),
                "Förlust över 20 % (%)": (
                    returns.lt(-20).mean() * 100
                )
            })

    return pd.DataFrame(results)


df_eth_bmsb_returns = analyze_bmsb_returns(
    df_eth_ma, "ETH"
)

df_btc_bmsb_returns = analyze_bmsb_returns(
    df_btc_ma, "BTC"
)

df_bmsb_returns = pd.concat(
    [
        df_eth_bmsb_returns,
        df_btc_bmsb_returns
    ],
    ignore_index=True
)

print("STEG 24 – BMSB OCH FRAMTIDA AVKASTNING")

for asset in ["ETH", "BTC"]:

    print(f"\n===== {asset} =====")

    display(
        df_bmsb_returns[
            df_bmsb_returns["Valuta"] == asset
        ].round(2)
    )


STEG 24 – BMSB OCH FRAMTIDA AVKASTNING

===== ETH =====


,Valuta,Horisont,BMSB_Position,Observationer,Medianavkastning (%),Medelavkastning (%),Positiva utfall (%),Förlust över 20 % (%)
0,ETH,30D,Inom BMSB,208,-11.13,-0.04,41.35,21.15
1,ETH,30D,Under BMSB,1575,0.37,3.50,50.48,19.94
2,ETH,30D,Över BMSB,2156,4.43,20.80,58.12,11.92
3,ETH,180D,Inom BMSB,208,14.26,120.20,60.58,27.88
4,ETH,180D,Under BMSB,1470,9.38,141.92,57.07,34.01
5,ETH,180D,Över BMSB,2111,24.30,139.62,64.42,24.49



===== BTC =====


,Valuta,Horisont,BMSB_Position,Observationer,Medianavkastning (%),Medelavkastning (%),Positiva utfall (%),Förlust över 20 % (%)
6,BTC,30D,Inom BMSB,154,-0.33,2.54,50.00,18.83
7,BTC,30D,Under BMSB,1520,1.78,2.68,56.64,9.34
8,BTC,30D,Över BMSB,2482,4.69,9.20,59.51,6.08
9,BTC,180D,Inom BMSB,144,26.97,29.29,61.81,20.83
10,BTC,180D,Under BMSB,1435,15.61,24.88,55.19,32.33
11,BTC,180D,Över BMSB,2427,37.10,71.88,73.47,12.86


In [56]:

# STEG 24.3 – Diagram över BMSB och framtida avkastning

import plotly.express as px

position_order = [
    "Under BMSB",
    "Inom BMSB",
    "Över BMSB"
]

for asset in ["ETH", "BTC"]:

    data = df_bmsb_returns[
        df_bmsb_returns["Valuta"] == asset
    ].copy()

    fig = px.bar(
        data,
        x="BMSB_Position",
        y="Medianavkastning (%)",
        color="Horisont",
        barmode="group",
        text_auto=".1f",
        category_orders={
            "BMSB_Position": position_order,
            "Horisont": ["30D", "180D"]
        },
        hover_data=[
            "Observationer",
            "Positiva utfall (%)",
            "Förlust över 20 % (%)"
        ],
        title=(
            f"{asset} – Framtida medianavkastning "
            "efter BMSB-position"
        ),
        template="plotly_dark"
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="white"
    )

    fig.update_layout(
        xaxis_title="Prisets position",
        yaxis_title="Medianavkastning (%)",
        height=550
    )

    fig.show()



### Resultat och slutsats – Steg 24

#### Syfte

Vi undersökte sambandet mellan prisets position
relativt Bull Market Support Band (BMSB)
och efterföljande avkastning för ETH och BTC.

#### Metod

Observationerna delades in i tre grupper:

- Över BMSB
- Inom BMSB
- Under BMSB

Vi jämförde medianavkastning, medelavkastning,
andel positiva utfall och risken för
förlust större än 20 % efter 30 och 180 dagar.

#### Resultat – Ethereum

När priset låg över BMSB var medianavkastningen:

- Efter 30 dagar: +4,43 %
- Efter 180 dagar: +24,30 %

När priset låg under BMSB:

- Efter 30 dagar: +0,37 %
- Efter 180 dagar: +9,38 %

Förluster större än 20 % efter 180 dagar
förekom i 24,49 % av observationerna över
BMSB och 34,01 % under BMSB.

#### Resultat – Bitcoin

När priset låg över BMSB var medianavkastningen:

- Efter 30 dagar: +4,69 %
- Efter 180 dagar: +37,10 %

När priset låg under BMSB:

- Efter 30 dagar: +1,78 %
- Efter 180 dagar: +15,61 %

Förluster större än 20 % efter 180 dagar
förekom i 12,86 % av observationerna över
BMSB och 32,33 % under BMSB.

#### Tolkning

Både Ethereum och Bitcoin uppvisade högre
historisk medianavkastning när priset
befann sig över BMSB.

Samma grupp hade även lägst observerad
andel stora förluster på båda tidshorisonterna.

Resultaten tyder på att BMSB kan vara en
användbar indikator för att beskriva
marknadsförhållanden.

#### Begränsningar

- Observationernas framtida perioder överlappar.
- Resultaten kan påverkas av enskilda marknadscykler.
- Sambanden behöver undersökas per år.
- Ingen oberoende statistisk signifikans
  har fastställts.
- Historiska samband garanterar inte
  framtida avkastning.

#### Slutsats

BMSB uppvisar ett intressant historiskt
samband med framtida avkastning och risk.

Indikatorn är relevant att inkludera
i gruppens Streamlit-app tillsammans med
Moving Averages och regressionsband.

Nästa steg är att undersöka om sambanden
är stabila under olika marknadscykler.



## Steg 25 – BMSB:s stabilitet över tid

### Syfte

Vi undersöker om sambandet mellan BMSB
och framtida avkastning är stabilt
under olika historiska marknadsperioder.

### Forskningsfråga

Har priset över BMSB följts av högre
avkastning även under svagare marknadsår?

### Metod

Vi delar upp observationerna per kalenderår
och jämför framtida avkastning efter
30 respektive 180 dagar.

Vi analyserar:

- Antal observationer
- Medianavkastning per år
- Skillnad mellan pris över och under BMSB

### Begränsningar

Historiska avkastningsperioder överlappar.
Resultaten är deskriptiva och ska inte
tolkas som oberoende statistiska tester.


In [57]:

# STEG 25.2 – BMSB och framtida avkastning per år

import pandas as pd
import numpy as np
from IPython.display import display

def analyze_bmsb_yearly(source, asset):

    df = source.copy()

    df["time"] = pd.to_datetime(
        df["time"], utc=True
    )

    df = (
        df.sort_values("time")
        .drop_duplicates("time")
        .set_index("time")
    )

    price = pd.to_numeric(
        df[f"{asset}_Price"],
        errors="coerce"
    )

    results = []

    for horizon in [30, 180]:

        # Exakt framtida kalenderdatum
        future = price.copy()
        future.index -= pd.Timedelta(days=horizon)

        future_price = future.reindex(df.index)

        df["Future_Return"] = (
            future_price / price - 1
        ) * 100

        valid = df[
            df["BMSB_Position"].isin([
                "Över BMSB",
                "Under BMSB"
            ])
            & df["Future_Return"].notna()
        ].copy()

        valid["År"] = valid.index.year

        summary = (
            valid.groupby(["År", "BMSB_Position"])
            .agg(
                Observationer=("Future_Return", "size"),
                Median=("Future_Return", "median"),
                Positiva=("Future_Return",
                          lambda x: (x > 0).mean() * 100)
            )
            .reset_index()
        )

        summary["Valuta"] = asset
        summary["Horisont"] = f"{horizon}D"

        results.append(summary)

    return pd.concat(results, ignore_index=True)


df_yearly_bmsb = pd.concat(
    [
        analyze_bmsb_yearly(df_eth_ma, "ETH"),
        analyze_bmsb_yearly(df_btc_ma, "BTC")
    ],
    ignore_index=True
)

for asset in ["ETH", "BTC"]:

    print(f"\n===== {asset} – BMSB PER ÅR =====")

    display(
        df_yearly_bmsb[
            df_yearly_bmsb["Valuta"] == asset
        ].round(2)
    )



===== ETH – BMSB PER ÅR =====


,År,BMSB_Position,Observationer,Median,Positiva,Valuta,Horisont
0,2015,Under BMSB,10,158.32,100.00,ETH,30D
1,2015,Över BMSB,1,157.23,100.00,ETH,30D
2,2016,Under BMSB,74,-1.13,50.00,ETH,30D
3,2016,Över BMSB,244,3.26,53.69,ETH,30D
4,2017,Under BMSB,8,22.75,100.00,ETH,30D
5,2017,Över BMSB,346,48.43,78.03,ETH,30D
6,2018,Under BMSB,269,-13.15,31.23,ETH,30D
7,2018,Över BMSB,70,-26.99,5.71,ETH,30D
8,2019,Under BMSB,217,0.94,54.84,ETH,30D
9,2019,Över BMSB,105,6.30,58.10,ETH,30D



===== BTC – BMSB PER ÅR =====


,År,BMSB_Position,Observationer,Median,Positiva,Valuta,Horisont
47,2015,Under BMSB,85,13.00,95.29,BTC,30D
48,2015,Över BMSB,137,5.45,56.20,BTC,30D
49,2016,Under BMSB,1,8.58,100.00,BTC,30D
50,2016,Över BMSB,352,5.90,73.30,BTC,30D
51,2017,Över BMSB,364,26.00,74.45,BTC,30D
52,2018,Under BMSB,296,-3.58,36.15,BTC,30D
53,2018,Över BMSB,49,-22.02,0.00,BTC,30D
54,2019,Under BMSB,170,7.76,72.94,BTC,30D
55,2019,Över BMSB,174,3.53,57.47,BTC,30D
56,2020,Under BMSB,63,27.04,93.65,BTC,30D


In [58]:

# STEG 25.3 – Årlig medianavkastning för BMSB

import plotly.express as px

for asset in ["ETH", "BTC"]:

    data = df_yearly_bmsb[
        df_yearly_bmsb["Valuta"] == asset
    ].copy()

    data["År"] = data["År"].astype(str)

    fig = px.bar(
        data,
        x="År",
        y="Median",
        color="BMSB_Position",
        facet_col="Horisont",
        barmode="group",
        text_auto=".1f",
        hover_data=[
            "Observationer",
            "Positiva"
        ],
        color_discrete_map={
            "Över BMSB": "#2ECC71",
            "Under BMSB": "#FF5252"
        },
        title=f"{asset} – BMSB:s historiska resultat per år",
        template="plotly_dark"
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="white"
    )

    fig.update_layout(
        height=600,
        yaxis_title="Medianavkastning (%)"
    )

    fig.show()



## Steg 26 – BMSB:s stabilitet mellan marknadsår

### Syfte

Vi undersöker om prisets position relativt
BMSB har ett stabilt samband med framtida
avkastning under olika marknadsår.

### Metod

Vi jämför medianavkastningen när priset ligger
över respektive under BMSB.

Vi analyserar:

- Ethereum och Bitcoin separat
- 30 och 180 dagars framtida avkastning
- Skillnaden i medianavkastning per år
- Hur många år respektive position haft
  högst observerad medianavkastning

Som ett första beskrivande filter kräver
vi minst 30 observationer i vardera gruppen.

### Begränsningar

Observationerna kan överlappa kraftigt.
30 observationer innebär därför inte
30 statistiskt oberoende observationer.

Resultaten ska inte användas för att
optimera en handelsstrategi utan
ytterligare tidsbaserad validering.


In [59]:

# STEG 26.2 – Jämförelse av BMSB mellan marknadsår

import pandas as pd
import numpy as np
from IPython.display import display

MIN_OBS = 30

df_compare = df_yearly_bmsb.copy()

# Separera över och under BMSB
pivot = df_compare.pivot_table(
    index=["Valuta", "Horisont", "År"],
    columns="BMSB_Position",
    values=["Median", "Observationer"],
    aggfunc="first"
)

pivot.columns = [
    f"{metric}_{position}"
    for metric, position in pivot.columns
]

pivot = pivot.reset_index()

# Kontrollera att båda grupper finns
required = [
    "Median_Över BMSB",
    "Median_Under BMSB",
    "Observationer_Över BMSB",
    "Observationer_Under BMSB"
]

pivot = pivot.dropna(subset=required).copy()

# Minst 30 observationer i varje grupp
pivot = pivot[
    (pivot["Observationer_Över BMSB"] >= MIN_OBS)
    & (pivot["Observationer_Under BMSB"] >= MIN_OBS)
].copy()

# Skillnad i procentenheter
pivot["Skillnad (pp)"] = (
    pivot["Median_Över BMSB"]
    - pivot["Median_Under BMSB"]
)

pivot["Bäst historiskt"] = np.select(
    [
        pivot["Skillnad (pp)"] > 0,
        pivot["Skillnad (pp)"] < 0
    ],
    [
        "Över BMSB",
        "Under BMSB"
    ],
    default="Lika"
)

df_bmsb_comparison = pivot.sort_values(
    ["Valuta", "Horisont", "År"]
)

print("STEG 26 – JÄMFÖRELSE MELLAN MARKNADSÅR")

for asset in ["ETH", "BTC"]:

    print(f"\n===== {asset} =====")

    display(
        df_bmsb_comparison[
            df_bmsb_comparison["Valuta"] == asset
        ].round(2)
    )

# Sammanfattning
summary = (
    df_bmsb_comparison
    .groupby(["Valuta", "Horisont", "Bäst historiskt"])
    .size()
    .reset_index(name="Antal år")
)

print("\nSAMMANFATTNING – HISTORISKT BÄSTA POSITION")

display(summary)


STEG 26 – JÄMFÖRELSE MELLAN MARKNADSÅR

===== ETH =====


,Valuta,Horisont,År,Median_Under BMSB,Median_Över BMSB,Observationer_Under BMSB,Observationer_Över BMSB,Skillnad (pp),Bäst historiskt
25,ETH,180D,2016,1713.63,27.38,74.0,244.0,-1686.24,Under BMSB
27,ETH,180D,2018,-46.58,-61.49,269.0,70.0,-14.91,Under BMSB
28,ETH,180D,2019,34.54,-31.29,217.0,105.0,-65.83,Under BMSB
29,ETH,180D,2020,141.87,259.91,49.0,311.0,118.05,Över BMSB
30,ETH,180D,2021,71.49,31.02,31.0,325.0,-40.47,Under BMSB
32,ETH,180D,2023,101.69,17.52,88.0,258.0,-84.17,Under BMSB
33,ETH,180D,2024,-12.99,-16.02,115.0,240.0,-3.04,Under BMSB
34,ETH,180D,2025,64.87,-22.96,158.0,201.0,-87.83,Under BMSB
37,ETH,30D,2016,-1.13,3.26,74.0,244.0,4.39,Över BMSB
39,ETH,30D,2018,-13.15,-26.99,269.0,70.0,-13.85,Under BMSB



===== BTC =====


,Valuta,Horisont,År,Median_Under BMSB,Median_Över BMSB,Observationer_Under BMSB,Observationer_Över BMSB,Skillnad (pp),Bäst historiskt
0,BTC,180D,2015,74.56,45.26,85.0,137.0,-29.31,Under BMSB
3,BTC,180D,2018,-27.35,-42.55,296.0,49.0,-15.21,Under BMSB
4,BTC,180D,2019,93.79,-11.42,170.0,174.0,-105.21,Under BMSB
5,BTC,180D,2020,59.30,146.58,63.0,296.0,87.28,Över BMSB
6,BTC,180D,2021,34.22,-12.66,104.0,244.0,-46.89,Under BMSB
8,BTC,180D,2023,140.67,40.73,64.0,287.0,-99.94,Under BMSB
9,BTC,180D,2024,59.06,21.77,62.0,279.0,-37.29,Under BMSB
10,BTC,180D,2025,-15.71,-17.38,125.0,230.0,-1.67,Under BMSB
12,BTC,30D,2015,13.00,5.45,85.0,137.0,-7.54,Under BMSB
15,BTC,30D,2018,-3.58,-22.02,296.0,49.0,-18.43,Under BMSB



SAMMANFATTNING – HISTORISKT BÄSTA POSITION


,Valuta,Horisont,Bäst historiskt,Antal år
0,BTC,180D,Under BMSB,7
1,BTC,180D,Över BMSB,1
2,BTC,30D,Under BMSB,7
3,BTC,30D,Över BMSB,2
4,ETH,180D,Under BMSB,7
5,ETH,180D,Över BMSB,1
6,ETH,30D,Under BMSB,6
7,ETH,30D,Över BMSB,3


In [60]:

# STEG 26.3 – BMSB-stabilitet per år

import plotly.express as px

for asset in ["ETH", "BTC"]:

    data = df_bmsb_comparison[
        df_bmsb_comparison["Valuta"] == asset
    ].copy()

    data = data.sort_values(["Horisont", "År"])

    data["År"] = data["År"].astype(str)

    year_order = sorted(
        data["År"].unique(),
        key=int
    )

    fig = px.bar(
        data,
        x="År",
        y="Skillnad (pp)",
        color="Bäst historiskt",
        facet_col="Horisont",
        text_auto=".1f",
        category_orders={
            "År": year_order,
            "Horisont": ["30D", "180D"]
        },
        color_discrete_map={
            "Över BMSB": "#2ECC71",
            "Under BMSB": "#FF5252",
            "Lika": "#A0A0A0"
        },
        hover_data=[
            "Observationer_Över BMSB",
            "Observationer_Under BMSB"
        ],
        title=(
            f"{asset} – BMSB:s samband "
            "med framtida avkastning per år"
        ),
        template="plotly_dark"
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="white"
    )

    fig.update_layout(
        height=600,
        xaxis_title="År",
        yaxis_title="Skillnad i medianavkastning (pp)"
    )

    fig.for_each_xaxis(
        lambda axis: axis.update(
            categoryorder="array",
            categoryarray=year_order
        )
    )

    fig.show()



### Resultat och slutsats – Steg 26

#### Syfte

Vi undersökte om sambandet mellan BMSB och
framtida avkastning var stabilt mellan olika år.

#### Metod

Vi jämförde medianavkastningen över och under
BMSB för Ethereum och Bitcoin.

Analysen omfattade 30 och 180 dagars
framtida avkastning.

Endast år med minst 30 observationer
i vardera gruppen inkluderades.

#### Resultat

**Ethereum – 30 dagar**
- Under BMSB gav högre medianavkastning under 6 år.
- Över BMSB gav högre medianavkastning under 3 år.

**Ethereum – 180 dagar**
- Under BMSB: 7 år.
- Över BMSB: 1 år.

**Bitcoin – 30 dagar**
- Under BMSB: 7 år.
- Över BMSB: 2 år.

**Bitcoin – 180 dagar**
- Under BMSB: 7 år.
- Över BMSB: 1 år.

#### Tolkning

BMSB:s samband med framtida avkastning
varierade betydligt mellan marknadsåren.

Resultaten skiljer sig från den sammanslagna
analysen där pris över BMSB ofta hade
högre medianavkastning.

Det visar hur valet av jämförelsemetod
och historisk period påverkar slutsatserna.

#### Begränsningar

- Framtida avkastningsperioder överlappar.
- Grupperna kan ha olika stora urval.
- Enskilda år innehåller extrema prisrörelser.
- Årsresultaten utgör inte oberoende tester.
- Inget statistiskt säkerställt samband
  med framtida lönsamhet har fastställts.

#### Slutsats

BMSB fungerar som en beskrivande trendindikator,
men dess historiska samband med framtida
avkastning är inte stabilt mellan olika år.

Indikatorn är därför relevant för att
visualisera marknadsförhållanden, men bör
inte ensam användas för köp- och säljbeslut.

BMSB kan komplettera gruppens huvudmodell,
Shared Curvature Quantile Regression.



## Steg 27 – Känslighetsanalys av överlappande observationer

### Syfte

I tidigare analyser undersökte vi sambandet
mellan BMSB och framtida avkastning för
Ethereum och Bitcoin.

När framtida avkastning beräknas för varje
dag överlappar observationsperioderna.

Exempelvis delar två 180-dagarsutfall,
med startdatum en dag ifrån varandra,
nästan hela sin framtida tidsperiod.

Det kan göra att resultaten ser mer stabila
ut än de faktiskt är.

### Forskningsfråga

Förändras sambandet mellan BMSB och framtida
avkastning om vi använder observationer
som ligger längre ifrån varandra?

### Metod

Vi jämför:

1. Samtliga dagliga observationer.
2. Observationer med minst 30 dagars avstånd.
3. Observationer med minst 180 dagars avstånd.

Vi undersöker flera olika startpunkter
för varje urval.

För varje urval beräknas:

- Antal observationer per BMSB-position.
- Medianavkastning efter 30 och 180 dagar.
- Skillnaden mellan över och under BMSB.
- Andel positiva framtida utfall.

### Begränsningar

Ett glesare urval minskar överlappningen
men leder också till färre observationer.

Resultaten kan därför bli känsliga
för vilka datum som väljs.

Vi betraktar analysen som en
känslighetskontroll, inte ett
statistiskt bevis för prognosförmåga.


In [61]:

# ============================================================
# STEG 27.2 – KÄNSLIGHETSANALYS AV ÖVERLAPPANDE OBSERVATIONER
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

HORIZONS = [30, 180]
POSITIONS = ["Under BMSB", "Över BMSB"]


def prepare_future_returns(source, asset, horizon):

    df = source.copy()

    df["time"] = pd.to_datetime(
        df["time"], utc=True
    )

    df = (
        df.sort_values("time")
        .drop_duplicates("time")
        .set_index("time")
    )

    price = pd.to_numeric(
        df[f"{asset}_Price"],
        errors="coerce"
    )

    # Matcha framtida pris på exakt kalenderdatum
    future = price.copy()
    future.index = (
        future.index - pd.Timedelta(days=horizon)
    )

    future_price = future.reindex(df.index)

    df["Future_Return"] = (
        future_price / price - 1
    ) * 100

    # Behåll bara kompletta, giltiga observationer
    df = df[
        df["BMSB_Position"].isin(POSITIONS)
        & df["Future_Return"].notna()
        & np.isfinite(df["Future_Return"])
        & price.gt(0)
    ].copy()

    return df


def evaluate_sample(df, asset, horizon, step, offset):

    # Välj kalenderdatum med samma avstånd,
    # oberoende av hur många rader som filtrerats bort.
    day_number = (
        df.index.normalize()
        - pd.Timestamp("1970-01-01", tz="UTC")
    ).days

    if step == 1:
        selected = df
    else:
        selected = df[
            (day_number % step) == offset
        ]

    result = []

    for position in POSITIONS:

        group = selected[
            selected["BMSB_Position"] == position
        ]

        returns = group["Future_Return"]

        result.append({
            "Valuta": asset,
            "Horisont": f"{horizon}D",
            "Urval": "Alla dagar" if step == 1
                     else f"Var {step}:e dag",
            "Avstånd": step,
            "Startförskjutning": offset,
            "Position": position,
            "Observationer": len(group),
            "Medianavkastning (%)": (
                returns.median() if len(group) else np.nan
            ),
            "Positiva utfall (%)": (
                returns.gt(0).mean() * 100
                if len(group) else np.nan
            )
        })

    return result


all_results = []

for asset, source in [
    ("ETH", df_eth_ma),
    ("BTC", df_btc_ma)
]:

    for horizon in HORIZONS:

        df = prepare_future_returns(
            source, asset, horizon
        )

        # Alla dagliga observationer
        all_results.extend(
            evaluate_sample(df, asset, horizon, 1, 0)
        )

        # Testa samtliga kalenderbaserade startpunkter
        for step in [30, 180]:

            for offset in range(step):

                all_results.extend(
                    evaluate_sample(
                        df, asset, horizon, step, offset
                    )
                )


df_sensitivity = pd.DataFrame(all_results)

# Jämför över och under för varje urval
comparison = df_sensitivity.pivot_table(
    index=[
        "Valuta",
        "Horisont",
        "Urval",
        "Avstånd",
        "Startförskjutning"
    ],
    columns="Position",
    values=[
        "Observationer",
        "Medianavkastning (%)"
    ],
    aggfunc="first"
)

comparison.columns = [
    f"{metric}_{position}"
    for metric, position in comparison.columns
]

comparison = comparison.reset_index()

comparison["Skillnad (pp)"] = (
    comparison["Medianavkastning (%)_Över BMSB"]
    - comparison["Medianavkastning (%)_Under BMSB"]
)

comparison["Bäst historiskt"] = np.select(
    [
        comparison["Skillnad (pp)"] > 0,
        comparison["Skillnad (pp)"] < 0
    ],
    [
        "Över BMSB",
        "Under BMSB"
    ],
    default="Lika / saknas"
)

# Minst fem observationer i vardera gruppen
# för att redovisa ett urval som jämförbart.
comparison["Tillräckligt urval"] = (
    (comparison["Observationer_Över BMSB"] >= 5)
    & (comparison["Observationer_Under BMSB"] >= 5)
)

df_sensitivity_comparison = comparison.copy()

print("STEG 27 – KÄNSLIGHETSANALYS")

for asset in ["ETH", "BTC"]:

    print(f"\n===== {asset} =====")

    daily = comparison[
        (comparison["Valuta"] == asset)
        & (comparison["Avstånd"] == 1)
    ]

    print("\nREFERENS – ALLA DAGAR")
    display(daily.round(2))

    for step in [30, 180]:

        subset = comparison[
            (comparison["Valuta"] == asset)
            & (comparison["Avstånd"] == step)
            & comparison["Tillräckligt urval"]
        ]

        print(f"\nURVAL VAR {step}:E DAG")

        if subset.empty:
            print("Inga urval klarade minimikravet.")
            continue

        summary = (
            subset.groupby("Horisont")
            .agg(
                Giltiga_startpunkter=(
                    "Startförskjutning", "count"
                ),
                Median_skillnad_pp=(
                    "Skillnad (pp)", "median"
                ),
                Min_skillnad_pp=(
                    "Skillnad (pp)", "min"
                ),
                Max_skillnad_pp=(
                    "Skillnad (pp)", "max"
                ),
                Andel_över_bättre=(
                    "Skillnad (pp)",
                    lambda x: (x > 0).mean() * 100
                )
            )
            .reset_index()
        )

        display(summary.round(2))


STEG 27 – KÄNSLIGHETSANALYS

===== ETH =====

REFERENS – ALLA DAGAR


,Valuta,Horisont,Urval,Avstånd,Startförskjutning,Medianavkastning (%)_Under BMSB,Medianavkastning (%)_Över BMSB,Observationer_Under BMSB,Observationer_Över BMSB,Skillnad (pp),Bäst historiskt,Tillräckligt urval
422,ETH,180D,Alla dagar,1,0,9.38,24.30,1470,2111,14.92,Över BMSB,True
633,ETH,30D,Alla dagar,1,0,0.37,4.43,1575,2156,4.06,Över BMSB,True



URVAL VAR 30:E DAG


,Horisont,Giltiga_startpunkter,Median_skillnad_pp,Min_skillnad_pp,Max_skillnad_pp,Andel_över_bättre
0,180D,30,16.67,-1.10,30.46,96.67
1,30D,30,4.79,-0.69,10.58,90.00



URVAL VAR 180:E DAG


,Horisont,Giltiga_startpunkter,Median_skillnad_pp,Min_skillnad_pp,Max_skillnad_pp,Andel_över_bättre
0,180D,180,6.87,-69.13,107.39,55.00
1,30D,180,6.76,-25.39,37.50,76.67


... (2 outputs left out)


URVAL VAR 30:E DAG


,Horisont,Giltiga_startpunkter,Median_skillnad_pp,Min_skillnad_pp,Max_skillnad_pp,Andel_över_bättre
0,180D,30,22.07,13.64,41.18,100.0
1,30D,30,2.62,0.48,5.56,100.0



URVAL VAR 180:E DAG


,Horisont,Giltiga_startpunkter,Median_skillnad_pp,Min_skillnad_pp,Max_skillnad_pp,Andel_över_bättre
0,180D,180,23.30,-27.21,70.88,85.56
1,30D,180,2.85,-10.03,35.07,68.89


In [62]:

# ============================================================
# STEG 27.3 – VISUALISERA KÄNSLIGHETSANALYSEN
# ============================================================

import plotly.express as px
import plotly.graph_objects as go

valid = df_sensitivity_comparison[
    df_sensitivity_comparison["Tillräckligt urval"]
].copy()

valid["Metod"] = valid["Avstånd"].map({
    1: "Alla dagar",
    30: "Var 30:e dag",
    180: "Var 180:e dag"
})

for asset in ["ETH", "BTC"]:

    data = valid[
        valid["Valuta"] == asset
    ].copy()

    fig = px.box(
        data,
        x="Metod",
        y="Skillnad (pp)",
        color="Horisont",
        points="all",
        category_orders={
            "Metod": [
                "Alla dagar",
                "Var 30:e dag",
                "Var 180:e dag"
            ]
        },
        title=(
            f"{asset} – Känslighet för "
            "överlappande observationer"
        ),
        hover_data=[
            "Startförskjutning",
            "Observationer_Över BMSB",
            "Observationer_Under BMSB"
        ],
        template="plotly_dark"
    )

    fig.add_hline(
        y=0,
        line_dash="dash",
        line_color="white"
    )

    fig.update_layout(
        xaxis_title="Urvalsmetod",
        yaxis_title=(
            "Skillnad i medianavkastning (pp)"
        ),
        height=550
    )

    fig.show()


In [63]:

# ============================================================
# STEG 28.1 – EXPORT TILL STREAMLIT
# Marknadscykler + Moving Averages + BMSB
# ============================================================

import pandas as pd
from pathlib import Path
from IPython.display import display

OUTPUT_DIR = Path(
    r"C:\Users\milen\source\repos"
    r"\Statistisk_programmering_ML"
    r"\Ethereum_Bitcoin_ML\data"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def export_market_data(ma_data, cycle_data, asset):

    # Moving Averages och BMSB
    ma = ma_data.copy()

    # Marknadsklassificering
    cycles = cycle_data.copy()

    ma["time"] = pd.to_datetime(
        ma["time"], utc=True
    )

    cycles["time"] = pd.to_datetime(
        cycles["time"], utc=True
    )

    # Sammanfoga på datum
    result = ma.merge(
        cycles[
            [
                "time",
                "Market_Phase",
                "Confirmed_Phase"
            ]
        ],
        on="time",
        how="left",
        validate="one_to_one"
    )

    columns = [
        "time",
        f"{asset}_Price",
        "SMA_200",
        "SMA_20W",
        "EMA_21W",
        "BMSB_Upper",
        "BMSB_Lower",
        "BMSB_Position",
        "Market_Phase",
        "Confirmed_Phase"
    ]

    result = (
        result[columns]
        .sort_values("time")
        .reset_index(drop=True)
    )

    filename = f"{asset.lower()}_market_analysis.csv"

    result.to_csv(
        OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

    print(f"\nExporterad: {filename}")
    print(f"Observationer: {len(result)}")
    print(
        f"Period: {result['time'].min().date()} "
        f"till {result['time'].max().date()}"
    )

    display(result.tail(5))

    return result


eth_market_export = export_market_data(
    df_eth_ma,
    df_eth_confirmed,
    "ETH"
)

btc_market_export = export_market_data(
    df_btc_ma,
    df_btc_confirmed,
    "BTC"
)

print("\nEXPORT KLAR!")



Exporterad: eth_market_analysis.csv
Observationer: 4104
Period: 2015-08-08 till 2026-11-01


,time,ETH_Price,SMA_200,SMA_20W,EMA_21W,BMSB_Upper,BMSB_Lower,BMSB_Position,Market_Phase,Confirmed_Phase
4099,2026-10-28 00:00:00+00:00,2356.988509,2158.586902,2163.381441,2319.184185,2319.184185,2163.381441,Över BMSB,Bull Market,Bull Market
4100,2026-10-29 00:00:00+00:00,2256.031994,2158.916361,2163.381441,2319.184185,2319.184185,2163.381441,Inom BMSB,Bull Market,Bull Market
4101,2026-10-30 00:00:00+00:00,2240.229980,2158.251587,2163.381441,2319.184185,2319.184185,2163.381441,Inom BMSB,Bull Market,Bull Market
4102,2026-10-31 00:00:00+00:00,2252.073358,2157.895898,2163.381441,2319.184185,2319.184185,2163.381441,Inom BMSB,Bull Market,Bull Market
4103,2026-11-01 00:00:00+00:00,2269.988406,2157.461958,2163.381441,2319.184185,2319.184185,2163.381441,Inom BMSB,Bull Market,Bull Market



Exporterad: btc_market_analysis.csv
Observationer: 4323
Period: 2015-01-01 till 2026-11-01


,time,BTC_Price,SMA_200,SMA_20W,EMA_21W,BMSB_Upper,BMSB_Lower,BMSB_Position,Market_Phase,Confirmed_Phase
4318,2026-10-28 00:00:00+00:00,81237.437826,73058.140475,72353.029895,76542.155043,76542.155043,72353.029895,Över BMSB,Bull Market,Bull Market
4319,2026-10-29 00:00:00+00:00,78193.426404,73095.680759,72353.029895,76542.155043,76542.155043,72353.029895,Över BMSB,Bull Market,Bull Market
4320,2026-10-30 00:00:00+00:00,77109.590747,73108.064985,72353.029895,76542.155043,76542.155043,72353.029895,Över BMSB,Bull Market,Bull Market
4321,2026-10-31 00:00:00+00:00,77401.017287,73124.202529,72353.029895,76542.155043,76542.155043,72353.029895,Över BMSB,Bull Market,Bull Market
4322,2026-11-01 00:00:00+00:00,78135.080289,73141.068107,72353.029895,76542.155043,76542.155043,72353.029895,Över BMSB,Bull Market,Bull Market



EXPORT KLAR!



# Steg 29 – Historiska Bull & Bear Markets

## Syfte

Vi jämför vår tidigare SMA-baserade
marknadsklassificering med en alternativ
metod för att identifiera större
historiska marknadscykler.

Analysen genomförs för Ethereum och Bitcoin.

## Forskningsfrågor

1. Vilka större historiska nedgångsperioder
   kan vi identifiera?

2. Hur skiljer sig dessa från vår
   tidigare SMA-baserade klassificering?

3. Kan vi visualisera längre marknadscykler
   tillsammans med SMA 200 och BMSB?

## Metod A – SMA-baserade trendfaser

Den tidigare metoden använder:

- Prisets position relativt SMA 200.
- SMA 200:s riktning under 30 dagar.
- 14 dagars bekräftelse av trendvillkoren.

Metoden använder endast historisk
och aktuell prisinformation.

## Metod B – Historiska priscykler

Vi identifierar nedgångsperioder från
tidigare historiska pristoppar.

En period ingår i vår jämförelse om:

- Nedgången från topp till botten
  är minst 50 %.
- Minst 90 dagar passerar mellan
  pristoppen och den lägsta kursen.

Vi registrerar:

- Datum för pristoppen.
- Datum för botten.
- Största nedgången.
- Datum då tidigare topp återhämtades.
- Om återhämtningen fortfarande saknas.

En historisk bear-episod markeras från
pristoppen till dess att toppen återtas,
eller till sista tillgängliga datum.

## Begränsningar

50 % och 90 dagar är experimentella
gränser för denna jämförelse.

De är inspirerade av en publicerad
Bitcoin-analys, men är inte en
universell definition för kryptovalutor.

Metoden identifierar cykler i efterhand.
Den är inte en handels- eller prognosmodell.

## Källor

- SEC Investor.gov:
  https://www.investor.gov/introduction-investing/investing-basics/glossary/bear-market

- Coinbase Institutional – April 2025:
  https://www.coinbase.com/en-au/institutional/research-insights/research/monthly-outlook/monthly-outlook-apr-2025

- Galaxy Research – Bitcoin Bear Markets:
  https://www.galaxy.com/insights/research/bitcoin-50-week-moving-average-bear-market-ceiling-200-week-floor-analysis


In [64]:

# ============================================================
# STEG 29.3 – HISTORISKA BEAR-MARKET-EPISODER
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display

DRAWDOWN_THRESHOLD = 50
MIN_PEAK_TO_TROUGH_DAYS = 90


def find_historical_bear_cycles(source, asset):

    df = source.copy()

    df["time"] = pd.to_datetime(
        df["time"], utc=True
    )

    df = (
        df.sort_values("time")
        .drop_duplicates("time")
        .reset_index(drop=True)
    )

    price_col = f"{asset}_Price"

    df[price_col] = pd.to_numeric(
        df[price_col], errors="coerce"
    )

    df = df[
        df[price_col].notna()
        & (df[price_col] > 0)
    ].reset_index(drop=True)

    if df.empty:
        raise ValueError(f"Inga giltiga priser för {asset}")

    # Högsta tidigare observerade pris
    df["Historical_High"] = df[price_col].cummax()

    # Nedgång från historiskt högsta pris
    df["Drawdown_Pct"] = (
        df[price_col] / df["Historical_High"] - 1
    ) * 100

    # Varje nytt historiskt högsta pris
    # startar ett nytt topp-till-återhämtning-intervall.
    previous_high = df["Historical_High"].shift()

    is_new_high = (
        df[price_col].ge(previous_high)
        | previous_high.isna()
    )

    peak_indices = np.flatnonzero(is_new_high.to_numpy())

    episodes = []
    episode_id = 0

    for i, peak_idx in enumerate(peak_indices):

        if i + 1 < len(peak_indices):
            recovery_idx = int(peak_indices[i + 1])
            end_idx = recovery_idx - 1
        else:
            recovery_idx = None
            end_idx = len(df) - 1

        if end_idx <= peak_idx:
            continue

        segment = df.iloc[peak_idx:end_idx + 1]

        trough_idx = int(
            segment[price_col].idxmin()
        )

        peak_price = df.loc[peak_idx, price_col]
        trough_price = df.loc[trough_idx, price_col]

        peak_date = df.loc[peak_idx, "time"]
        trough_date = df.loc[trough_idx, "time"]

        drawdown = (
            trough_price / peak_price - 1
        ) * 100

        decline_days = (
            trough_date - peak_date
        ).days

        # Kvalificerar perioden som större nedgång?
        if (
            drawdown > -DRAWDOWN_THRESHOLD
            or decline_days < MIN_PEAK_TO_TROUGH_DAYS
        ):
            continue

        episode_id += 1

        recovered = recovery_idx is not None

        if recovered:
            end_date = df.loc[recovery_idx, "time"]
            status = "Återhämtad"
        else:
            end_date = df["time"].iloc[-1]
            status = "Ej återhämtad vid dataslut"

        episodes.append({
            "Valuta": asset,
            "Episod": episode_id,
            "Start": peak_date,
            "Botten": trough_date,
            "Slut": end_date,
            "Topppris": peak_price,
            "Bottenpris": trough_price,
            "Max nedgång (%)": drawdown,
            "Dagar till botten": decline_days,
            "Dagar till återhämtning/dataslut": (
                end_date - peak_date
            ).days,
            "Status": status
        })

    episodes_df = pd.DataFrame(
        episodes,
        columns=[
            "Valuta", "Episod", "Start", "Botten",
            "Slut", "Topppris", "Bottenpris",
            "Max nedgång (%)", "Dagar till botten",
            "Dagar till återhämtning/dataslut",
            "Status"
        ]
    )

    return df, episodes_df


df_eth_longcycles, df_eth_episodes = (
    find_historical_bear_cycles(df_eth_ma, "ETH")
)

df_btc_longcycles, df_btc_episodes = (
    find_historical_bear_cycles(df_btc_ma, "BTC")
)


for asset, episodes in [
    ("ETH", df_eth_episodes),
    ("BTC", df_btc_episodes)
]:

    print(f"\n===== {asset} – STÖRRE BEAR-EPISODER =====")

    if episodes.empty:
        print("Inga episoder uppfyllde definitionen.")
    else:
        display(
            episodes.round(2)
        )

print("\nBERÄKNING KLAR!")



===== ETH – STÖRRE BEAR-EPISODER =====


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\3907096669.py:163: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  episodes.round(2)


,Valuta,Episod,Start,Botten,Slut,Topppris,Bottenpris,Max nedgång (%),Dagar till botten,Dagar till återhämtning/dataslut,Status
0,ETH,1,2016-06-16 00:00:00+00:00,2016-12-05 00:00:00+00:00,2017-03-11 00:00:00+00:00,20.73,6.73,-67.52,172,268,Återhämtad
1,ETH,2,2018-01-13 00:00:00+00:00,2018-12-15 00:00:00+00:00,2021-01-19 00:00:00+00:00,1380.14,82.83,-94.00,336,1102,Återhämtad
2,ETH,3,2021-11-08 00:00:00+00:00,2022-06-18 00:00:00+00:00,2025-08-22 00:00:00+00:00,4811.16,992.79,-79.36,222,1383,Återhämtad
3,ETH,4,2025-08-22 00:00:00+00:00,2026-06-06 00:00:00+00:00,2026-11-01 00:00:00+00:00,4831.10,1564.52,-67.62,288,436,Ej återhämtad vid dataslut



===== BTC – STÖRRE BEAR-EPISODER =====


C:\Users\milen\AppData\Local\Temp\ipykernel_31888\3907096669.py:163: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  episodes.round(2)


,Valuta,Episod,Start,Botten,Slut,Topppris,Bottenpris,Max nedgång (%),Dagar till botten,Dagar till återhämtning/dataslut,Status
0,BTC,1,2017-12-16 00:00:00+00:00,2018-12-15 00:00:00+00:00,2020-11-30 00:00:00+00:00,19640.51,3185.07,-83.78,364,1080,Återhämtad
1,BTC,2,2021-04-13 00:00:00+00:00,2021-07-20 00:00:00+00:00,2021-10-19 00:00:00+00:00,63445.64,29766.66,-53.08,98,189,Återhämtad
2,BTC,3,2021-11-08 00:00:00+00:00,2022-11-09 00:00:00+00:00,2024-03-04 00:00:00+00:00,67541.76,15758.29,-76.67,366,847,Återhämtad
3,BTC,4,2025-10-06 00:00:00+00:00,2026-06-30 00:00:00+00:00,2026-11-01 00:00:00+00:00,124824.45,58525.08,-53.11,267,391,Ej återhämtad vid dataslut



BERÄKNING KLAR!


In [65]:

# ============================================================
# STEG 29.4 – VISUALISERA HISTORISKA BEAR-EPISODER
# ============================================================

import plotly.graph_objects as go


def plot_historical_cycles(data, episodes, asset):

    df = data.copy().sort_values("time")
    price_col = f"{asset}_Price"

    fig = go.Figure()

    # Markera större historiska nedgångsepisoder
    for _, episode in episodes.iterrows():

        fig.add_vrect(
            x0=episode["Start"],
            x1=episode["Slut"],
            fillcolor="rgba(255,82,82,0.17)",
            line_width=0,
            layer="below"
        )

        # Historisk topp
        fig.add_trace(go.Scatter(
            x=[episode["Start"]],
            y=[episode["Topppris"]],
            mode="markers",
            marker=dict(
                color="#FF5252",
                size=10,
                symbol="triangle-down"
            ),
            name="Historisk topp",
            legendgroup="peak",
            showlegend=False
        ))

        # Historisk botten
        fig.add_trace(go.Scatter(
            x=[episode["Botten"]],
            y=[episode["Bottenpris"]],
            mode="markers",
            marker=dict(
                color="#00CC96",
                size=10,
                symbol="triangle-up"
            ),
            name="Historisk botten",
            legendgroup="trough",
            showlegend=False
        ))

    # Pris
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df[price_col],
        mode="lines",
        name=f"{asset}-pris",
        line=dict(color="#5DADE2", width=2)
    ))

    # SMA 200
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["SMA_200"],
        mode="lines",
        name="SMA 200",
        line=dict(color="#F39C12", width=2)
    ))

    # Bull Market Support Band
    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["BMSB_Lower"],
        mode="lines",
        name="Nedre BMSB",
        line=dict(color="#2ECC71", width=1)
    ))

    fig.add_trace(go.Scatter(
        x=df["time"],
        y=df["BMSB_Upper"],
        mode="lines",
        name="Övre BMSB",
        fill="tonexty",
        fillcolor="rgba(46,204,113,0.12)",
        line=dict(color="#2ECC71", width=1)
    ))

    fig.update_layout(
        title=(
            f"{asset} – Större historiska Bear Markets "
            "(50 % / 90 dagar)"
        ),
        xaxis_title="Datum",
        yaxis_title="Pris (USD, logaritmisk skala)",
        yaxis_type="log",
        template="plotly_dark",
        height=650,
        hovermode="x unified"
    )

    fig.update_xaxes(
        rangeslider_visible=True
    )

    fig.show()


plot_historical_cycles(
    df_eth_longcycles,
    df_eth_episodes,
    "ETH"
)

plot_historical_cycles(
    df_btc_longcycles,
    df_btc_episodes,
    "BTC"
)


In [66]:

# ============================================================
# STEG 29.5 – JÄMFÖR SMA-TREND OCH HISTORISKA CYKLER
# ============================================================

from IPython.display import display

comparison_results = []

for asset, source, episodes in [
    ("ETH", df_eth_confirmed, df_eth_episodes),
    ("BTC", df_btc_confirmed, df_btc_episodes)
]:

    df = source.copy()
    df["time"] = pd.to_datetime(df["time"], utc=True)

    df["Historical_Bear_Episode"] = False

    # Markera bara perioder som kvalificerat
    # sig som stora historiska nedgångsepisoder.
    for _, ep in episodes.iterrows():

        mask = (
            (df["time"] >= ep["Start"])
            & (df["time"] <= ep["Slut"])
        )

        df.loc[
            mask, "Historical_Bear_Episode"
        ] = True

    # Exkludera dagar utan giltig SMA-status
    valid = df[
        df["Confirmed_Phase"] != "Otillräcklig data"
    ].copy()

    for historical_state, group in valid.groupby(
        "Historical_Bear_Episode"
    ):

        comparison_results.append({
            "Valuta": asset,
            "Historisk stor bear-episod": (
                "Ja" if historical_state else "Nej"
            ),
            "Observationer": len(group),
            "SMA Bull (%)": (
                group["Confirmed_Phase"]
                .eq("Bull Market")
                .mean() * 100
            ),
            "SMA Bear (%)": (
                group["Confirmed_Phase"]
                .eq("Bear Market")
                .mean() * 100
            ),
            "SMA Övergång (%)": (
                group["Confirmed_Phase"]
                .eq("Övergångsfas")
                .mean() * 100
            )
        })


df_cycle_comparison = pd.DataFrame(
    comparison_results
)

print("JÄMFÖRELSE – HISTORISKA CYKLER OCH SMA")

display(
    df_cycle_comparison.round(2)
)


JÄMFÖRELSE – HISTORISKA CYKLER OCH SMA


,Valuta,Historisk stor bear-episod,Observationer,SMA Bull (%),SMA Bear (%),SMA Övergång (%)
0,ETH,Nej,683,92.09,0.00,7.91
1,ETH,Ja,3192,33.49,30.11,36.40
2,BTC,Nej,1583,81.68,0.51,17.81
3,BTC,Ja,2511,31.46,35.44,33.09



### Resultat och slutsats – Steg 29

#### Syfte

Vi jämförde två metoder för att analysera
historiska marknadscykler för Ethereum
och Bitcoin:

1. SMA 200 med 14 dagars bekräftelse.
2. Större historiska nedgångsepisoder.

#### Metod

Vi definierade en större historisk
Bear-episod som en nedgång:

- På minst 50 % från tidigare pristopp.
- Med minst 90 dagar mellan topp och botten.

Vi undersökte även hur lång tid det tog
innan priset återhämtade sin tidigare topp.

#### Resultat – Ethereum

Fyra större Bear-episoder identifierades.

De största nedgångarna var:

- 2016: -67,52 %
- 2018: -94,00 %
- 2021–2022: -79,36 %
- 2025–2026: -67,62 %

Den sista episoden hade inte återhämtat
sin tidigare topp vid dataslutet.

#### Resultat – Bitcoin

Fyra större Bear-episoder identifierades.

- 2017–2018: -83,78 %
- 2021 (april–juli): -53,08 %
- 2021–2022: -76,67 %
- 2025–2026: -53,11 %

Den sista episoden var fortfarande
inte återhämtad vid dataslutet.

#### Jämförelse med SMA 200

Under de större historiska
Bear-episoderna visade SMA-metoden
följande fördelning:

Ethereum:
- SMA Bull: 33,49 %
- SMA Bear: 30,11 %
- Övergång: 36,40 %

Bitcoin:
- SMA Bull: 31,46 %
- SMA Bear: 35,44 %
- Övergång: 33,09 %

Det visar att kortare positiva
trendperioder kan förekomma inom
längre historiska nedgångsepisoder.

#### Tolkning

Metoderna beskriver olika egenskaper
hos marknaden.

Historiska nedgångsepisoder visar
större cykliska prisförändringar.

SMA 200 beskriver den löpande
marknadstrenden.

BMSB ger ytterligare information
om prisets position relativt
veckobaserade glidande medelvärden.

#### Begränsningar

- 50 % och 90 dagar är valda analysgränser.
- Cyklerna identifieras i efterhand.
- En historisk Bear-episod kan innehålla
  både nedgång och efterföljande återhämtning.
- Resultatet påverkas av dataperioden.
- Klassificeringarna är inte prognoser.
- Historiska priscykler kan inte ensamma
  användas för att förutsäga framtida priser.

#### Slutsats

Analysen visar att historiska
marknadscykler och SMA-baserade
trendindikatorer kompletterar varandra.

Vi rekommenderar därför att gruppens
Streamlit-app innehåller:

1. Historiska Bear-episoder.
2. SMA-baserad Bull/Bear-klassificering.
3. SMA 200.
4. 20-week SMA och 21-week EMA.
5. Bull Market Support Band.

Dessa indikatorer kan visas tillsammans
med Shared Curvature Quantile Regression
för att ge bättre historisk marknadskontext.
